# Simultaneous HPS search: GP background + profiled-Gaussian extraction — v15.8

This notebook provides an end-to-end workflow for **bump hunting** and **limit setting** in an invariant-mass spectrum, designed for resonance searches such as the Heavy Photon Search (HPS) program. The core idea is to model the smoothly-falling background with a **Gaussian Process (GP)** (including correlated uncertainties), then test/limit a narrow Gaussian-like signal in a sliding mass window using standard particle-physics frequentist tools.

### Studies this notebook can perform (and why they matter in particle physics)

1. **Nonparametric background modeling with correlated uncertainties (Gaussian Processes)**  
   Smooth backgrounds are ubiquitous in resonance searches. A GP provides a flexible, data-driven model while naturally propagating correlated uncertainties between bins/mass points. This is especially useful when you want to avoid committing to a fixed analytic background function.  
   Reference: Rasmussen & Williams, *Gaussian Processes for Machine Learning* (MIT Press, 2006).  

2. **Local signal extraction (profile likelihood / one-sided discovery test)**  
   At each mass hypothesis \(m\), we fit a signal amplitude \(A\) (optionally allowing \(\hat A<0\) but testing \(A\ge 0\)) and compute the local discovery test statistic and **local p-value \(p_0\)** / significance \(Z\). This is the standard “bump hunt” diagnostic used throughout collider and fixed-target analyses.  
   Reference: Cowan, Cranmer, Gross & Vitells, EPJ C **71** (2011) 1554, arXiv:1007.1727.  

3. **Look-Elsewhere Effect (LEE) / global p-values via an effective trials factor**  
   Because we scan many correlated mass hypotheses, the smallest local \(p_0\) must be corrected for multiple testing to obtain a **global significance**. This notebook includes an “effective trials factor” approximation \(N_\text{eff}\) and uses Sidak/Bonferroni style corrections for reviewer-facing diagnostics.  
   Reference: Gross & Vitells, EPJ C **70** (2010) 525, arXiv:1005.1891.  

4. **Frequentist upper limits using \(CL_s\)**  
   When no significant excess is present, we set one-sided upper limits at (typically) 95% CL. The \(CL_s\) construction is widely used in particle physics for searches near the sensitivity boundary.  
   Reference: Read, J. Phys. G **28** (2002) 2693–2704, DOI: 10.1088/0954-3899/28/10/313.  

5. **Expected limit bands (\(\pm1\sigma\), \(\pm2\sigma\)) from background-only pseudoexperiments**  
   Expected bands are a standard way to communicate sensitivity independent of statistical luck in the observed data. This notebook supports a fast conditional-toy mode and an optional “full procedural” mode that refits the GP on each toy for a more conservative end-to-end uncertainty estimate.

6. **Dataset combinations in overlap regions**  
   Where multiple datasets cover the same mass region, the notebook can form a combined likelihood (assuming statistical independence) and produce combined \(\varepsilon^2\) limits and LEE diagnostics.

### Key external references (quick links)

- G. Cowan, K. Cranmer, E. Gross, O. Vitells — *Asymptotic formulae for likelihood-based tests of new physics* (EPJ C 71 (2011) 1554): https://doi.org/10.1140/epjc/s10052-011-1554-0  
- E. Gross, O. Vitells — *Trial factors for the look elsewhere effect in high energy physics* (EPJ C 70 (2010) 525): https://doi.org/10.1140/epjc/s10052-010-1470-8  
- A. L. Read — *Presentation of search results: the \(CL_s\) technique* (J. Phys. G 28 (2002) 2693): https://doi.org/10.1088/0954-3899/28/10/313  
- C. E. Rasmussen, C. K. I. Williams — *Gaussian Processes for Machine Learning* (2006): http://www.gaussianprocess.org/gpml/  

---

**Version notes (v15.8):**  
- Fixes the expected-band plotting crash by standardizing expected-band column names (`A_med`, `eps2_med`, etc.) and making log-scale plotting robust to missing/non-positive values.  
- Tightens the Look-Elsewhere correction by capping the effective trials factor \(N_\text{eff}\) by the number of scanned mass hypotheses.


In [ ]:
# =============================================================================
# Cell 0 — Runtime + imports
# =============================================================================
import os
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
os.environ["VECLIB_MAXIMUM_THREADS"] = "2"
os.environ["NUMEXPR_NUM_THREADS"] = "2"

import math
import json
import datetime
import glob
from dataclasses import dataclass
from typing import Callable, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

from math import lgamma

# sklearn GPR
from sklearn.gaussian_process import GaussianProcessRegressor
import sklearn.gaussian_process as skgp
from sklearn.base import clone as sk_clone

# Your local gp package used in existing notebooks

# --- Make sure the local `gp` package is importable ---
# This notebook expects your custom `gp` package (the one used in your prior notebooks),
# which is not on PyPI. If `import gp` fails, we add likely repo roots to sys.path.
import sys
from pathlib import Path



# --- needed for parallel bands ---
import joblib


def _add_gp_to_syspath():
    # 1) Try common relative roots from the current working directory
    candidates = [Path("."), Path(".."), Path("../.."), Path("../../..")]
    # 2) Also try walking upward from the notebook directory (works when CWD is different)
    try:
        nb_dir = Path.cwd()
        candidates += [nb_dir] + list(nb_dir.parents)[:6]
    except Exception:
        pass

    def has_gp(p: Path) -> bool:
        return (p / "gp.py").exists() or (p / "gp" / "__init__.py").exists() or (p / "gp").is_dir()

    added = []
    for p in candidates:
        p = p.resolve()
        if has_gp(p) and str(p) not in sys.path:
            sys.path.insert(0, str(p))
            added.append(str(p))

    # 3) As a last resort, search a few levels down for a folder that contains gp/
    if not added:
        roots = [Path(".").resolve(), Path("..").resolve()]
        for r in roots:
            try:
                for q in r.glob("**/gp/__init__.py"):
                    repo_root = q.parents[1]
                    if str(repo_root) not in sys.path:
                        sys.path.insert(0, str(repo_root))
                        added.append(str(repo_root))
                        break
            except Exception:
                continue

    if added:
        print("[path] added to sys.path for gp:", added[0])
    else:
        print("[path] did not find gp on disk; you may need to `pip install -e` the repo containing gp.")

_add_gp_to_syspath()

import gp

# numerics / optimization
from scipy.optimize import minimize
from scipy.stats import norm

# plotting (matplotlib only)
import matplotlib.pyplot as plt

print("Imports OK")


In [ ]:
# =============================================================================
# Cell 0b — Plot style (v12: reviewer-friendly defaults)
# =============================================================================
import matplotlib as mpl

# You can tweak these, but these defaults are chosen to be legible in exported PNG/PDF.
PLOT_STYLE = globals().get("PLOT_STYLE", "paper")  # "paper" | "talk" | "default"

def set_plot_style(style: str = "paper"):
    style = str(style).lower().strip()
    if style == "talk":
        base_font = 16
        lw = 2.0
    elif style == "paper":
        base_font = 13
        lw = 1.8
    else:
        base_font = 12
        lw = 1.6

    mpl.rcParams.update({
        "figure.figsize": (9.5, 5.5),
        "figure.dpi": 120,
        "savefig.dpi": int(globals().get("SAVEFIG_DPI", 300)),
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.05,

        "font.size": base_font,
        "axes.labelsize": base_font,
        "axes.titlesize": base_font + 1,
        "legend.fontsize": base_font - 1,
        "xtick.labelsize": base_font - 1,
        "ytick.labelsize": base_font - 1,

        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linestyle": "-",
        "axes.axisbelow": True,

        "lines.linewidth": lw,
        "lines.markersize": 5,

        "axes.formatter.use_mathtext": True,
        "mathtext.default": "regular",
    })

set_plot_style(PLOT_STYLE)
print(f"[plot] matplotlib style set: {PLOT_STYLE}")


## Inputs

This cell is the only place you should need to edit frequently.

- Set ROOT paths and histogram names per dataset.
- Set per-dataset search ranges (in **GeV**).
- Set mass resolution \(\sigma(m)\) polynomials (in **GeV**) per dataset.
- Set radiative fraction \(f_{\rm rad}(m)\) per dataset (polynomial in **GeV**).
- Choose which datasets are enabled.
- Choose asymptotic vs toys.
- Choose scan step (default 1 MeV).


In [ ]:
# =============================================================================
# Cell 1 — Configuration (v15.8)
# =============================================================================
# This cell is intentionally ordered (top → bottom) for reviewer/publication clarity:
#   0) Output / reproducibility
#   1) Inputs (ROOT paths + histogram names)
#   2) Dataset ranges + enable switches + blinding policy
#   3) Physics models: σ(m) and f_rad(m)
#   4) Scan grid + blinding/masking
#   5) GP preprocessing + kernel policy
#   6) Statistical settings (CLs, expected bands)
#   7) Injection/extraction closure tests (incl. refit-on-toy mode)
#   8) Plot/export settings + provenance
#   9) Optional: functional-form toy analyses
#
# v15 highlights (reliability / robustness):
#   - Decouple the *extraction blind window* from the *GP training exclusion window*.
#     * BLIND_NSIGMA: bins used for extraction/limits at each mass hypothesis.
#     * GP_TRAIN_EXCLUDE_NSIGMA: bins EXCLUDED when training the GP on sidebands.
#       (Can be larger to reduce signal-leakage bias when refitting GPs on toys.)
#   - Injection toys (refit mode): choose whether to inject the *full* Gaussian line-shape
#     (realistic, includes tails in sidebands) or a *window-only* injection (diagnostic).
#   - Expected UL bands: optional "full procedural" toy mode that refits the GP on each toy.

# -----------------------------
# 0) Output / reproducibility
# -----------------------------
OUTPUT_DIR = "outputs/hps_gp_v15_pt8_2015_2016_b2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Validation/debug controls
DEBUG_PRINT = False   # verbose dataset validation prints
FAIL_FAST   = False   # stop immediately on first validation error

# Save controls
SAVE_PER_MASS_FOLDERS = True
SAVE_PLOTS = True
SAVE_SCAN_DIAGNOSTIC_PLOTS = True
SCAN_DIAGNOSTIC_PLOT_EVERY_N = 1     # 1=every mass; set 5/10 to thin; set 0/None to disable
SCAN_DIAGNOSTIC_ZOOM_HALF_SIGMA = 0.5  # show ±0.5σ beyond blind bounds in the zoom panel
SCAN_DIAGNOSTIC_EXPLICIT_MASSES = []   # optional explicit mass list (overrides *_EVERY_N if non-empty)
SAVE_FIT_JSON = True

# If you run an optimization scan on pseudo-datasets, the best bounds can be saved and reused.
KERNEL_LS_OPT_FILE = os.path.join(OUTPUT_DIR, "optimized_gpr_lengthscales.json")

print("Config loaded. OUTPUT_DIR =", OUTPUT_DIR)

# -----------------------------
# 1) Inputs: dataset ROOT paths
# -----------------------------
PATH_2015 = "/Users/emryspeets/research_plots/2015_data/invariant_mass_0pt5mm_full.root"
PATH_2016 = "/Users/emryspeets/research_plots/2016_IMD/EventSelection_Data_10Percent.root"  # 10% 2016
PATH_2021 = "../limited_data_8kbins_0_04.root"  # 2021 0.03% (histogram file produced in your workflow)

# Optional: "MC-only mode" (turn off 2015/2016 and swap 2021 to MC file)
ONLY_2021_MC = False
PATH_2021_MC = "/path/to/2021_MC_dataset.root"

# -----------------------------
# 1b) Inputs: histogram names
# -----------------------------
HIST_2015 = "invariant_mass"
HIST_2016 = "h_Minv_General_Final_1"
HIST_2021 = "unc_vtx_mass_hist"

# -----------------------------
# 2) Dataset ranges + enable switches
# -----------------------------
# Scan ranges (GeV): used for the mass hypothesis scan
RANGE_2015 = (0.035,0.045)#(0.018, 0.130)
RANGE_2016 = (0.035, 0.045)
RANGE_2021 = (0.035, 0.060)

# GP training data ranges (GeV):
# Use (None, None) to use the full histogram extent from the ROOT file.
# These should generally be a *superset* of the scan ranges above.
DATA_RANGE_2015 = (None, None)
DATA_RANGE_2016 = (None, None)
DATA_RANGE_2021 = (0.025, 0.30)

# Enable datasets
ENABLE_2015 = True
ENABLE_2016 = True
ENABLE_2021 = False  # True

# Blinding / visibility policy (for reviewer-safe running)
#   - "observed": compute observed UL/p0 and make blind-window plots (uses blind counts)
#   - "expected_only": DO NOT compute observed quantities or make blind-window plots
DATA_VISIBILITY = {
    "2015": "observed",
    "2016": "observed",
    "2021": "observed",
}

# -----------------------------
# 3) Physics models: σ(m) and f_rad(m)
# -----------------------------
# sigma(m) = sum_i coeffs[i] * m**i  (GeV)
SIGMA_COEFFS_2015 = [-0.0000922283032152, 0.0532190838657]          # linear
SIGMA_COEFFS_2016 = [0.00038, 0.041, -0.27, 3.49, -11.11]           # 4th order
SIGMA_COEFFS_2021 = [0.00286957, -0.00851449, 0.25362319]           # quadratic example

# Radiative fraction f_rad(m) polynomial coefficients (dimensionless)
FRAD_COEFFS_2015 = [0.085]  # from the paper
FRAD_COEFFS_2016 = [0.05]
FRAD_COEFFS_2021 = [0.0529]

# -----------------------------
# 4) Scan grid + blinding/masking
# -----------------------------
MASS_STEP_GEV = 0.001   # 1 MeV
DO_COMBINED = True      # if True and multiple datasets overlap, compute combined eps2 limits

# (A) Extraction blind window (used for A_hat/A_up fits + plotted shading)
BLIND_NSIGMA = 1.64     # blind half-width in sigma units (≈ central 90% for a Gaussian)

# (B) GP training exclusion window (used for GP sideband training)
# IMPORTANT (v15): you can set this > BLIND_NSIGMA to reduce signal-leakage bias
# in refit-on-toy studies (and, potentially, in the real analysis if a large signal exists).
GP_TRAIN_EXCLUDE_NSIGMA = BLIND_NSIGMA

# (C) Optional scan edge guardrails:
# If True, only scan mass points where the *entire* exclusion window (±SCAN_EDGE_GUARD_NSIGMA·σ)
# lies within the dataset's available training range (data_low/data_high if set; else m_low/m_high).
SCAN_REQUIRE_TWO_SIDEBANDS = False
SCAN_EDGE_GUARD_NSIGMA = GP_TRAIN_EXCLUDE_NSIGMA

# -----------------------------
# 4b) Signal template normalization (v14 convention)
# -----------------------------
# TEMPLATE_NORM_MODE:
#   - 'pdf'    : w_i are Gaussian bin probability masses (Σw over blind window = fraction in window)
#   - 'window' : renormalize so Σw=1 over the blind-window bins (legacy / for comparisons)
TEMPLATE_NORM_MODE = "pdf"
TEMPLATE_WARN_IF_RANGE_TRUNCATED = True
TEMPLATE_RANGE_TRUNC_EPS = 5e-3  # warn if Σw over full histogram differs from 1 by more than this

# -----------------------------
# 5) GP preprocessing (log-x/log-y + alpha choice)
# -----------------------------
PRE_LOG = True               # log-x and log-y
PRE_ZERO_ALPHA = 1.0         # alpha when y==0
ALPHA_MODEL = "1/y"          # alpha_i = 1/y_i for y_i>0  (approx Poisson in log-space)
PRE_ALPHA_FIRST_N = 0        # 0 disables
PRE_ALPHA_FIRST_FACTOR = 0.1 # if PRE_ALPHA_FIRST_N>0

# -----------------------------
# 5b) Kernel hyperparameters and length-scale bounds policy
# -----------------------------
# NOTE: If PRE_LOG=True, the GP is trained in x = ln(m), so the RBF length_scale is in Δln(m).
KERNEL_CONSTANT_INIT = 1.0
KERNEL_CONSTANT_BOUNDS = (1e-8, 1e18)

# Manual LS settings (used only if KERNEL_LS_POLICY="manual")
KERNEL_LS_INIT = 1.0
KERNEL_LS_BOUNDS = (0.001, 10.0)

# Resolution-scaled LS settings (default)
KERNEL_LS_POLICY = "resolution_scaled_local"    # "manual" | "resolution_scaled_local" | "resolution_scaled_global"
# Backwards-compatible alias:
#   "resolution_scaled" == "resolution_scaled_local"

# Upper/lower bounds relative to local resolution in GP x-units (σ_x ≈ σ/m for PRE_LOG=True)
KERNEL_LS_RES_UPPER_FACTOR = 8.0
KERNEL_LS_RES_LOWER_FACTOR = 0.5
KERNEL_LS_RES_STAT = "median"
KERNEL_LS_RES_NPTS = 200

# v13.5: optional local ℓ_hi floor/cap
KERNEL_LS_LOCAL_HI_FLOOR_MODE = "none"   # "none" | "floor"
KERNEL_LS_LOCAL_HI_FLOOR_FACTOR = 1.0
KERNEL_LS_LOCAL_HI_CAP_XRANGE_FRAC = None

# Per-dataset factor overrides
KERNEL_LS_RES_UPPER_FACTOR_BY_DATASET = {"2021": 9.0, "2015": 8.0, "2016": 9.0}
KERNEL_LS_RES_LOWER_FACTOR_BY_DATASET = {"2021": 0.5, "2015": 0.5, "2016": 0.5}
KERNEL_LS_RES_STAT_BY_DATASET = {}
KERNEL_LS_RES_NPTS_BY_DATASET = {}

# Optional explicit numeric bounds in x-units (x=m if PRE_LOG=False, x=ln(m) if PRE_LOG=True)
KERNEL_LS_BOUNDS_BY_DATASET = {}
KERNEL_LS_INIT_BY_DATASET = {}

# Optimized length-scale loading safety
FORCE_LOAD_OPTIMIZED_LS = False
PRINT_KERNEL_BOUNDS = False
LOAD_OPTIMIZED_LS_IF_AVAILABLE = False
SAVE_OPTIMIZED_LS = True

# Convenience default kernel object (helper functions create fresh kernels per fit)
PAPER_KERNEL = (
    skgp.kernels.ConstantKernel(KERNEL_CONSTANT_INIT, KERNEL_CONSTANT_BOUNDS)
    * skgp.kernels.RBF(length_scale=KERNEL_LS_INIT, length_scale_bounds=KERNEL_LS_BOUNDS)
)

# -----------------------------
# 5c) GP fit controls
# -----------------------------
NEIGHBORHOOD_REBIN = 5
N_RESTARTS = 12

# -----------------------------
# 5d) Parallelism for the main scan
# -----------------------------
SCAN_PARALLEL = True
SCAN_N_WORKERS = 5
SCAN_PARALLEL_BACKEND = "loky"     # "loky" (processes) | "threading" (threads)
SCAN_THREADS_PER_WORKER = 1        # threadpoolctl limit inside each worker

# -----------------------------
# 6) Statistical settings: CLs + p0
# -----------------------------
CLS_ALPHA = 0.05
CLS_MODE = "asymptotic"   # "asymptotic" or "toys"
CLS_NUM_TOYS = 100        # used only if CLS_MODE="toys"
CLS_SEED_BASE = 12345     # used only when CLS_MODE='toys' (per-mass seeds derived deterministically)

EXTRACT_ALLOW_NEGATIVE = True  # allow A_hat < 0 in profiled extraction
EPS2_LRT_SCALE = 1e10          # combined p0 LRT uses A_tilde = eps2 * SCALE

# -----------------------------
# 6b) Expected UL bands (single + combined)
# -----------------------------
MAKE_UL_BANDS = True
UL_BANDS_TOYS = 100

# Expected-band runtime controls
UL_BANDS_N_RESTARTS = N_RESTARTS
UL_BANDS_N_WORKERS = 6
UL_BANDS_PARALLEL_BACKEND = "loky"
UL_BANDS_THREADS_PER_WORKER = 1
UL_BANDS_SEED = 1245

# v15: optional "full procedural" bands:
# When True, each outer toy is a full pseudo-dataset, and the GP is refit on toy sidebands
# before extracting the blind-window prediction used for the UL.
UL_BANDS_REFIT_GP_ON_TOY = False
UL_BANDS_REFIT_GP_RESTARTS = 0          # recommended: 0 (fast)
UL_BANDS_REFIT_GP_OPTIMIZE = True       # if False: do not optimize kernel hyperparameters on each toy
UL_BANDS_TRAIN_EXCLUDE_NSIGMA = GP_TRAIN_EXCLUDE_NSIGMA  # can be larger for robust bands

# Combined expected bands settings
DO_COMBINED_BANDS = True
COMBINED_BANDS_N_TOYS = 100
COMBINED_BANDS_SEED = 24680

# -----------------------------
# 6c) Background MVN non-negativity policy (used in expected bands and injections)
# -----------------------------
MVN_TRUNC_METHOD = "reject_then_clip"   # "clip" | "reject" | "reject_then_clip"
MVN_TRUNC_MAX_TRIES = 80
# -----------------------------
# 6d) Look-Elsewhere Effect (LEE) diagnostics (optional, reviewer-facing)
# -----------------------------
# We approximate the trials factor as the number of independent resolution elements across the
# scanned mass range:
#   N_eff ≈ ∫ dm / (W · σ(m))
# where W is a configurable "independent width" in units of σ(m). A conservative default is
# W = FWHM/σ ≈ 2.355.
#
# Global p-values are computed with the Sidak correction:
#   p_global = 1 - (1 - p_local)^{N_eff}  ≈ N_eff · p_local (for small p_local),
# and converted to Z_global with the standard one-sided Gaussian convention.
LEE_ENABLE = True
LEE_METHOD = "sidak"                  # "sidak" or "bonferroni"
LEE_INDEP_WIDTH_SIGMA = 2.355         # ~FWHM; controls N_eff magnitude
LEE_SIGMA_LINES = [1, 2, 3, 4, 5]     # which global-σ threshold lines to overlay on p0 plots
LEE_COMBO_SIGMA_METHOD = "min"        # for combinations: "min" uses best σ(m) among active datasets


# -----------------------------
# 7) Injection + extraction studies (closure / pull / coverage)
# -----------------------------
INJECT_SIGNAL = True
INJ_DATASET_KEY = "2015"      # "2015"|"2016"|"2021"
INJ_MASSES_GEV = [0.025, 0.06, 0.09]#[0.025, 0.060, 0.090, 0.070, 0.085, 0.100]
INJ_STRENGTHS = [0, 100, 200, 500, 1000, 2000, 5000, 10000]

# Option: injected strengths in units of the expected σ_A(m)
INJ_STRENGTH_MODE = "sigmaA"    # "absolute" | "sigmaA"
INJ_SIGMA_MULTIPLIERS = [0.0, 1.0, 2.0, 3.0, 5.0]

# σ_A reference choice:
#   - "asimov": n_obs := b_mean  (deterministic, typical expected σ_A)
#   - "poisson": n_obs := Poisson(b_mean)  (adds a single background fluctuation draw)
INJ_SIGMA_A_SOURCE = "asimov"

# How to generate injected signal bin counts:
#   - "multinomial": fixed total yield (rounded) with binomial thinning if Σw<1
#   - "poisson": independent Poisson in each bin with mean A_inj*w_i
INJ_MODE = "poisson"
INJ_NUM_TOYS = 200
INJ_SEED = 314159

INJ_BACKGROUND_TRUNCATION = "reject"    # "reject" (preferred) or "clip"
INJ_EXTRACT_ALLOW_NEGATIVE = True

# v15: injection shape control for the *refit-on-toy* mode:
#   - "full": inject full Gaussian line-shape across the full histogram (realistic; tails in sidebands)
#   - "window": inject only into the extraction blind window (diagnostic; no sideband leakage)
INJ_SHAPE_MODE = "full"

# v15: training exclusion used in refit-on-toy injection studies
# (defaults to UL_BANDS_TRAIN_EXCLUDE_NSIGMA if set, else GP_TRAIN_EXCLUDE_NSIGMA)
INJ_TRAIN_EXCLUDE_NSIGMA = 3.0

# v14.5+: optional (slow) injection-toy mode: refit the GP on each toy's sidebands
INJ_REFIT_GP_ON_TOY = False
INJ_REFIT_GP_RESTARTS = 0
INJ_REFIT_GP_OPTIMIZE = True

# -----------------------------
# 7b) Extraction display plots (single pseudo-dataset per point)
# -----------------------------
# These are *not* full toy ensembles. They generate one representative pseudo-dataset
# at each (m0, Ninj) and save reviewer-facing plots.
EXTRACTION_DISPLAY_STRENGTHS = [0, 1000, 2500, 5000, 10000]
EXTRACTION_DISPLAY_STRENGTH_MODE = "sigmaA"   # "absolute" | "sigmaA"
EXTRACTION_DISPLAY_SIGMA_MULTIPLIERS = [0.0, 2.0, 5.0]
EXTRACTION_DISPLAY_SEED = 271828
EXTRACTION_DISPLAY_MODE = "multinomial"       # fixed total injected signal events
EXTRACTION_DISPLAY_MASS_POINTS = [0.025, 0.06, 0.09]#[0.020, 0.035, 0.055, 0.070, 0.085, 0.100]

# Whether to refit the GP hyperparameters on each injected toy before extraction display.
EXTRACTION_DISPLAY_REFIT_GP = [False]
EXTRACTION_DISPLAY_GP_RESTARTS = N_RESTARTS
EXTRACTION_DISPLAY_ZOOM_HALF_SIGMA = 0.5
EXTRACTION_DISPLAY_TRAIN_EXCLUDE_NSIGMA = INJ_TRAIN_EXCLUDE_NSIGMA

# -----------------------------
# 8) Plot export quality / styling
# -----------------------------
SAVEFIG_DPI = 300
TITLE_PAD = 12.0

# Reviewer-facing blind-window plot tweaks
PLOT_BLIND_ZOOM_HALF_SIGMA = 0.5
PLOT_BLIND_SHADE_ALPHA = 0.25
PLOT_BLIND_SHADE_COLOR = "0.90"
PLOT_BLIND_SHOW_BKG_1SIGMA = True
PLOT_FULL_SHOW_BKG_1SIGMA = True
FULL_PLOT_LOGY = False

# -----------------------------
# 8b) Provenance / run metadata
# -----------------------------
WRITE_RUN_METADATA = True
RUN_METADATA_PATH = os.path.join(OUTPUT_DIR, "run_metadata.json")

# -----------------------------
# 8c) Convenience switches for the optional band-running cell
# -----------------------------
RUN_LIMIT_BANDS_ON = "2015"   # "2015" | "2016" | "2021" | "none"
MAKE_EPS2_BANDS = False

RUN_COMBINED_BANDS = True
RUN_COMBINED_BANDS_DATASETS = ["2015", "2016"]
RUN_COMBINED_BANDS_ONLY_OVERLAP = False

# Apply plot style after config (so SAVEFIG_DPI / PLOT_STYLE edits here take effect)
try:
    set_plot_style(str(globals().get("PLOT_STYLE", "paper")))
except Exception:
    pass

# -----------------------------
# 9) Optional: functional-form toy analyses
# -----------------------------
functional_form_toy_analyses = True

FUNCFORM_JOBS = [
    dict(
        tag="fSigPow",
        year="2016",
        root_path="/Users/emryspeets/GP_fitting/src/Gaussian-Process-Resonance-Search/func_form_make_2016_10pct_output_0p03_0p25_full.root",
        container="fSigPow",
        toy_name_fmt="fSigPow_toy_{i}",
        enabled=False,
    ),
    dict(
        tag="fSigPowExpQ",
        year="2015",
        root_path="/Users/emryspeets/GP_fitting/src/Gaussian-Process-Resonance-Search/func_form_make_2015_0p012_0p15_full_v5.root",
        container="fSigPowExpQ",
        toy_name_fmt="fSigPowExpQ_toy_{i}",
        enabled=True,
    ),
    dict(
        tag="fSigPowExpQ",
        year="2021",
        root_path="/Users/emryspeets/GP_fitting/src/Gaussian-Process-Resonance-Search/func_form_make_2021_0_pt_03_0p030_0p240_full_v5.root",
        container="fSigPowExpQ",
        toy_name_fmt="fSigPowExpQ_toy_{i}",
        enabled=True,
    ),
]

FUNCFORM_N_TOYS = 100
FUNCFORM_TOY_INDEX_RANGE = (0, 99)

FUNCFORM_N_WORKERS = 5
FUNCFORM_JOBLIB_BACKEND = "loky"

FUNCFORM_MASS_STEP_GEV = MASS_STEP_GEV
FUNCFORM_NEIGHBORHOOD_REBIN = NEIGHBORHOOD_REBIN
FUNCFORM_N_RESTARTS = N_RESTARTS
FUNCFORM_EXTRACT_ALLOW_NEGATIVE = True

# Functional-form toy GPR length-scale bounds (v8)
FUNCFORM_LS_POLICY = "resolution_scaled"
FUNCFORM_LS_UPPER_FACTOR = 8.0
FUNCFORM_LS_LOWER_FACTOR = KERNEL_LS_RES_LOWER_FACTOR
FUNCFORM_LS_STAT = KERNEL_LS_RES_STAT
FUNCFORM_LS_NPTS = KERNEL_LS_RES_NPTS

FUNCFORM_LS_OPTIMIZE = False
FUNCFORM_LS_UPPER_FACTOR_GRID = [2.0, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0]
FUNCFORM_LS_OPTIMIZE_N_TOYS = 10
FUNCFORM_LS_RUN_FULL_AFTER_OPTIMIZE = True
FUNCFORM_LS_OPTIMIZE_METRIC = "rms_median"
FUNCFORM_LS_SAVE_OPTIMIZED = True

# -----------------------------
# Progress / runtime diagnostics
# -----------------------------
PRINT_TOY_PROGRESS = True
TOY_PROGRESS_EVERY = 100


## Dataset abstraction

We keep per-dataset settings together, so that the simultaneous fit and output logic can remain generic.


In [ ]:
# =============================================================================
# Cell 2 — DatasetConfig + core physics utilities
# =============================================================================

ALPHA_EM = 1.0 / 137.0  # fine structure constant

@dataclass
class DatasetConfig:
    key: str
    label: str
    root_path: str
    hist_name: str

    m_low: float
    m_high: float

    sigma_coeffs: List[float]
    frad_coeffs: List[float]

    data_low: Optional[float] = None
    data_high: Optional[float] = None

    enabled: bool = True

    # --- piecewise config ---
    sigma_tail_m0: Optional[float] = None          # GeV; if None -> pure polynomial
    sigma_tail_slope_floor: float = 0.0            # GeV/GeV; enforce non-decreasing tail
    sigma_tail_slope_override: Optional[float] = None  # if you want to force a slope manually

    # cached values for speed/consistency (computed once)
    _sigma_m0: Optional[float] = None
    _slope_m0: Optional[float] = None

    @staticmethod
    def _poly(coeffs: List[float], m: float) -> float:
        return float(sum(c * (m ** i) for i, c in enumerate(coeffs)))

    @staticmethod
    def _poly_deriv(coeffs: List[float], m: float) -> float:
        # d/dm Σ c_i m^i = Σ i c_i m^(i-1)
        return float(sum(i * c * (m ** (i - 1)) for i, c in enumerate(coeffs[1:], start=1)))

    def __post_init__(self):
        if self.sigma_tail_m0 is not None:
            m0 = float(self.sigma_tail_m0)
            self._sigma_m0 = self._poly(self.sigma_coeffs, m0)

            slope = self._poly_deriv(self.sigma_coeffs, m0)
            if self.sigma_tail_slope_override is not None:
                slope = float(self.sigma_tail_slope_override)

            # enforce non-decreasing tail
            self._slope_m0 = max(float(slope), float(self.sigma_tail_slope_floor))

    def sigma(self, m: float) -> float:
        m = float(m)
        if self.sigma_tail_m0 is None or m <= float(self.sigma_tail_m0):
            return self._poly(self.sigma_coeffs, m)

        # linear extension beyond m0
        return float(self._sigma_m0 + self._slope_m0 * (m - float(self.sigma_tail_m0)))

    def frad(self, m: float) -> float:
        return float(sum(c * (float(m) ** i) for i, c in enumerate(self.frad_coeffs)))


def _poly_str(coeffs: List[float], name: str = "p") -> str:
    return name + "(m)=" + " + ".join([f"{c:.3g}*m^{i}" for i, c in enumerate(coeffs)])

def _rng_str(lo: Optional[float], hi: Optional[float]) -> str:
    if lo is None and hi is None:
        return "auto(full hist)"
    return f"[{('auto' if lo is None else f'{lo:.3f}')},{('auto' if hi is None else f'{hi:.3f}')}]"

def make_datasets() -> Dict[str, DatasetConfig]:
    p2021 = PATH_2021_MC if ONLY_2021_MC else PATH_2021
    ds = {
        "2015": DatasetConfig(
            key="2015", label="HPS 2015", root_path=PATH_2015, hist_name=HIST_2015,
            m_low=RANGE_2015[0], m_high=RANGE_2015[1],
            sigma_coeffs=SIGMA_COEFFS_2015, frad_coeffs=FRAD_COEFFS_2015,
            data_low=DATA_RANGE_2015[0], data_high=DATA_RANGE_2015[1],
            enabled=ENABLE_2015 and (not ONLY_2021_MC),
        ),
        "2016": DatasetConfig(
            key="2016", label="HPS 2016", root_path=PATH_2016, hist_name=HIST_2016,
            m_low=RANGE_2016[0], m_high=RANGE_2016[1],
            sigma_coeffs=SIGMA_COEFFS_2016, frad_coeffs=FRAD_COEFFS_2016,
            data_low=DATA_RANGE_2016[0], data_high=DATA_RANGE_2016[1],
            enabled=ENABLE_2016 and (not ONLY_2021_MC),
            # imposing piecewise 
            sigma_tail_m0=0.18,
            sigma_tail_slope_floor=0.0,
            sigma_tail_slope_override=0.0239,
        ),
        "2021": DatasetConfig(
            key="2021", label="HPS 2021" + (" (MC)" if ONLY_2021_MC else ""),
            root_path=p2021, hist_name=HIST_2021,
            m_low=RANGE_2021[0], m_high=RANGE_2021[1],
            sigma_coeffs=SIGMA_COEFFS_2021, frad_coeffs=FRAD_COEFFS_2021,
            data_low=DATA_RANGE_2021[0], data_high=DATA_RANGE_2021[1],
            enabled=ENABLE_2021,
        ),
    }
    return {k: v for k, v in ds.items() if v.enabled}

DATASETS = make_datasets()
print("Enabled datasets:", list(DATASETS.keys()))
for k, d in DATASETS.items():
    print(
        f"  {k}: scan=[{d.m_low:.3f},{d.m_high:.3f}]  "
        f"gp={_rng_str(d.data_low, d.data_high)}  "
        f"sigma: {_poly_str(d.sigma_coeffs,'σ')}  frad: {_poly_str(d.frad_coeffs,'f')}"
    )


In [ ]:
# =============================================================================
# Cell 2b — Fast validation (files, histograms, ranges, sigma/frad sanity)
# =============================================================================
import os
from pprint import pprint

# Debug toggles (set in Inputs cell if you like)
DEBUG_PRINT = bool(globals().get('DEBUG_PRINT', True))
FAIL_FAST   = bool(globals().get('FAIL_FAST', False))

def _validate_hist_with_uproot(path: str, hist_name: str):
    try:
        import uproot
    except Exception as e:
        raise RuntimeError("uproot is required for validation but not installed in this env") from e
    if not os.path.exists(path):
        raise FileNotFoundError(f"ROOT file not found: {path}")
    f = uproot.open(path)
    if hist_name not in f:
        # show a few keys to help debug
        keys = list(f.keys())[:30]
        raise KeyError(f"Histogram '{hist_name}' not found in {path}. First keys: {keys}")
    h = f[hist_name]
    # TH1-like objects support to_numpy()
    vals, edges = h.to_numpy()
    vals = vals.astype(float)
    edges = edges.astype(float)
    return vals, edges

def _effective_gp_range(ds: DatasetConfig, lo_e: float, hi_e: float) -> Tuple[float, float]:
    lo = lo_e if ds.data_low is None else float(ds.data_low)
    hi = hi_e if ds.data_high is None else float(ds.data_high)
    # clip to actual histogram
    lo = max(lo_e, lo)
    hi = min(hi_e, hi)
    # ensure it covers the scan window (auto-fix if the user accidentally made gp range too small)
    lo = max(lo_e, min(lo, float(ds.m_low)))
    hi = min(hi_e, max(hi, float(ds.m_high)))
    return float(lo), float(hi)

def validate_datasets(datasets: Dict[str, DatasetConfig]):
    report = {}
    for k, ds in datasets.items():
        try:
            vals, edges = _validate_hist_with_uproot(ds.root_path, ds.hist_name)
            lo_e, hi_e = float(edges[0]), float(edges[-1])
            tot = float(np.sum(vals))

            scan_ok = (ds.m_low < ds.m_high) and (ds.m_low < hi_e) and (ds.m_high > lo_e)
            gp_lo, gp_hi = _effective_gp_range(ds, lo_e, hi_e)
            gp_ok = (gp_lo < gp_hi)

            # scan window should lie inside gp range
            subset_ok = (ds.m_low >= gp_lo - 1e-12) and (ds.m_high <= gp_hi + 1e-12)

            # sigma/frad sanity at a representative mass inside the scan window and histogram
            m_test = float(0.5 * (max(ds.m_low, lo_e) + min(ds.m_high, hi_e)))
            sig = ds.sigma(m_test)
            fr = ds.frad(m_test)

            ok = True
            msgs = []
            if not scan_ok:
                ok = False
                msgs.append(f"Scan range [{ds.m_low},{ds.m_high}] does not overlap hist edges [{lo_e},{hi_e}]")
            if not gp_ok:
                ok = False
                msgs.append("GP range is invalid after clipping to histogram edges.")
            if not subset_ok:
                # not fatal (we auto-fix in _effective_gp_range/_build_model), but warn loudly
                msgs.append(f"WARNING: scan range not subset of configured GP range; will auto-expand GP range to include scan.")
            if not np.isfinite(sig) or sig <= 0:
                ok = False
                msgs.append(f"sigma(m={m_test:.3f}) is not positive/finite: {sig}")
            if not np.isfinite(fr) or fr <= 0:
                # allow frad issues to be non-fatal if user is still validating polynomials
                msgs.append(f"WARNING: frad(m={m_test:.3f}) is not positive/finite: {fr}")

            report[k] = dict(
                ok=ok,
                root_path=ds.root_path,
                hist_name=ds.hist_name,
                hist_edges=(lo_e, hi_e),
                total_counts=tot,
                scan_range=(ds.m_low, ds.m_high),
                gp_range_effective=(gp_lo, gp_hi),
                sigma_at_test=sig,
                frad_at_test=fr,
                messages=msgs,
            )
        except Exception as e:
            report[k] = dict(ok=False, error=str(e))
            if FAIL_FAST:
                raise
    return report

_validation = validate_datasets(DATASETS)
if DEBUG_PRINT:
    pprint(_validation)
bad = [k for k, v in _validation.items() if not v.get("ok", False)]
if bad:
    print("\n[validate] problems found in:", bad)
else:
    print("\n[validate] all enabled datasets look OK")


## Preprocessing and GPR fit helpers

This reproduces your log-space fit workflow:

- \(x \to \log x\) and \(y \to \log y\) when enabled.
- Heteroscedastic \(\alpha_i\) on the diagonal, with:
  - \(y_i=0\) bins handled safely (zero padding in log-y and a fixed \(\alpha\)).
  - default \(\alpha_i = 1/y_i\) for positive bins.


In [ ]:
# =============================================================================
# Cell 3 — Preprocessing + GPR  (v11: LS bounds are computed in the actual GP x-units)
# =============================================================================

def _alpha_var_log_from_counts(y: np.ndarray) -> np.ndarray:
    y = np.asarray(y, dtype=float)
    alpha = np.full_like(y, PRE_ZERO_ALPHA, dtype=float)
    pos = y > 0.0
    if ALPHA_MODEL == "1/y":
        alpha[pos] = 1.0 / np.clip(y[pos], 1e-12, None)
    else:
        alpha[pos] = 1.0 / np.clip(y[pos], 1.0, None)

    if PRE_ALPHA_FIRST_N > 0:
        k = min(PRE_ALPHA_FIRST_N, alpha.size)
        alpha[:k] *= PRE_ALPHA_FIRST_FACTOR
    return alpha

def _preprocess_xy_for_gpr(X: np.ndarray, y: np.ndarray):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    X_in = np.log(np.clip(X, 1e-12, None)) if PRE_LOG else X.copy()
    y_in = np.zeros_like(y, dtype=float)

    pos = y > 0.0
    if PRE_LOG:
        y_in[pos] = np.log(y[pos])
        alpha = _alpha_var_log_from_counts(y)
    else:
        y_in = y.copy()
        alpha = np.clip(y, 1.0, None)

    return X_in, y_in, alpha


def _sigma_x_from_sigma(m: np.ndarray, sigma: np.ndarray) -> np.ndarray:
    """Convert detector mass resolution σ(m) (GeV) into an *x-space* scale σ_x(m).

    The GP is trained in:
      - x = ln(m) if PRE_LOG=True
      - x = m     if PRE_LOG=False

    We define a natural x-scale:
        σ_x(m) = x(m + σ(m)) - x(m)

    For PRE_LOG=True and σ<<m:  σ_x ≈ σ/m.
    For PRE_LOG=False:          σ_x = σ (GeV).
    """
    m = np.asarray(m, float)
    sigma = np.asarray(sigma, float)
    m = np.clip(m, 1e-12, None)
    sigma = np.clip(sigma, 0.0, None)
    if PRE_LOG:
        return np.log(m + sigma) - np.log(m)
    return sigma



# -----------------------------
# v13.5: helper cache + unit conversions
# -----------------------------
_SIGMA_X_STAT_CACHE = {}  # (ds_key, PRE_LOG, stat, npts) -> sigma_x_stat

def _sigma_x_stat_cached(ds, *, stat: str, npts: int) -> float:
    """Dataset-wide typical σ_x statistic (cached).

    This is used to implement a *floor* for the local LS upper bound, so that when σ(m)/m decreases
    with m (common), the local x-space bound ℓ_hi(m) doesn't become artificially tight at high mass.
    """
    key = (str(getattr(ds, "key", "")), bool(PRE_LOG), str(stat).lower().strip(), int(npts))
    if key in _SIGMA_X_STAT_CACHE:
        return float(_SIGMA_X_STAT_CACHE[key])
    # Factors do not matter here; we only need the base σ_x statistic returned by the helper.
    _, _, base = _kernel_bounds_from_resolution_global(ds, upper_factor=1.0, lower_factor=1.0, stat=stat, npts=npts)
    base = float(base)
    _SIGMA_X_STAT_CACHE[key] = base
    return base

def length_scale_x_to_mass_delta(ls_x: float, mass: float) -> float:
    """Convert a GP x-space length-scale into an intuitive GeV-equivalent Δm around 'mass'.

    - If PRE_LOG=True:  x = ln(m)  =>  Δm ≈ m·(exp(ℓ)-1)
    - If PRE_LOG=False: x = m      =>  Δm = ℓ
    """
    m = float(mass)
    lx = float(ls_x)
    if (not np.isfinite(m)) or (m <= 0.0) or (not np.isfinite(lx)):
        return float("nan")
    if PRE_LOG:
        return float(m * (np.exp(lx) - 1.0))
    return float(lx)
def _kernel_bounds_from_resolution_global(ds,
                                         *,
                                         upper_factor: float,
                                         lower_factor: float,
                                         stat: str,
                                         npts: int) -> Tuple[float, float, float]:
    """Return (ls_lower, ls_upper, base_sigma_x) in *x-units*, using a dataset-wide statistic.

    This is the legacy v11/v12 behavior: sample σ(m) across the dataset mass range and pick a single
    representative σ_x value (median/mean/min/max). The resulting bounds do **not** vary with the
    mass hypothesis in the scan.

    Notes
    -----
    - σ_x(m) is defined consistently with the GP's x-axis:
        x = ln(m)  if PRE_LOG=True
        x = m      if PRE_LOG=False
      via: σ_x(m) = x(m+σ(m)) - x(m).
    """
    stat = str(stat).lower().strip()
    npts = int(max(10, npts))

    mgrid = np.linspace(float(ds.m_low), float(ds.m_high), npts)
    sgrid = np.array([float(ds.sigma(float(m))) for m in mgrid], dtype=float)
    sx = _sigma_x_from_sigma(mgrid, sgrid)

    # Defensive: remove non-finite entries
    sx = sx[np.isfinite(sx)]
    if sx.size == 0:
        base = float(KERNEL_LS_INIT)
    else:
        if stat == "median":
            base = float(np.median(sx))
        elif stat == "mean":
            base = float(np.mean(sx))
        elif stat == "min":
            base = float(np.min(sx))
        elif stat == "max":
            base = float(np.max(sx))
        else:
            raise ValueError(f"Unknown KERNEL_LS_RES_STAT='{stat}' (expected median/mean/min/max)")

    # Bounds (keep strictly positive)
    base = max(base, 1e-12)
    ls_lo = max(1e-12, float(lower_factor) * base)
    ls_hi = max(ls_lo * 1.001, float(upper_factor) * base)
    return ls_lo, ls_hi, base


def _kernel_bounds_from_resolution_local(ds,
                                        mass: float,
                                        *,
                                        upper_factor: float,
                                        lower_factor: float) -> Tuple[float, float, float]:
    """Return (ls_lower, ls_upper, base_sigma_x) in *x-units*, using σ(mass).

    This is the v13 bug-fix for the scan + expected-band code paths:
    the allowed RBF length-scale range is tied to the **local** detector mass resolution,
    so the upper bound grows/shrinks with σ(m) as the mass hypothesis changes.

    Default target: ls_hi = upper_factor * σ_x(mass)
    """
    m = float(mass)
    s = float(ds.sigma(m))
    base = float(_sigma_x_from_sigma(np.array([m], float), np.array([s], float))[0])

    # Bounds (keep strictly positive)
    base = max(base, 1e-12)
    ls_lo = max(1e-12, float(lower_factor) * base)
    ls_hi = max(ls_lo * 1.001, float(upper_factor) * base)
    return ls_lo, ls_hi, base


def compute_kernel_ls_bounds(ds: DatasetConfig,
                             *,
                             mass: Optional[float] = None,
                             policy: Optional[str] = None,
                             upper_factor: Optional[float] = None,
                             lower_factor: Optional[float] = None,
                             stat: Optional[str] = None,
                             npts: Optional[int] = None) -> Dict[str, float]:
    """Compute the RBF length-scale bounds/init for a dataset (and optionally a mass hypothesis).

    Returns a dict with:
      - ls_lo, ls_hi, ls_init
      - sigma_x (the σ_x used to set the bounds; NaN for manual)
      - policy_used (string)
    """
    # Policy selection
    pol = str(policy if policy is not None else KERNEL_LS_POLICY).lower().strip()
    # Back-compat alias: in v13, 'resolution_scaled' is interpreted as *local* scaling.
    if pol == "resolution_scaled":
        pol = "resolution_scaled_local"

    ds_key = str(getattr(ds, "key", ""))

    # Per-dataset factor overrides (optional)
    uf_by = dict(KERNEL_LS_RES_UPPER_FACTOR_BY_DATASET or {})
    lf_by = dict(KERNEL_LS_RES_LOWER_FACTOR_BY_DATASET or {})
    st_by = dict(KERNEL_LS_RES_STAT_BY_DATASET or {})
    np_by = dict(KERNEL_LS_RES_NPTS_BY_DATASET or {})

    uf = float(upper_factor if upper_factor is not None else uf_by.get(ds_key, KERNEL_LS_RES_UPPER_FACTOR))
    lf = float(lower_factor if lower_factor is not None else lf_by.get(ds_key, KERNEL_LS_RES_LOWER_FACTOR))
    st = str(stat if stat is not None else st_by.get(ds_key, KERNEL_LS_RES_STAT))
    np_ = int(npts if npts is not None else np_by.get(ds_key, KERNEL_LS_RES_NPTS))

    if pol == "manual":
        ls_lo, ls_hi = float(KERNEL_LS_BOUNDS[0]), float(KERNEL_LS_BOUNDS[1])
        if ls_hi <= ls_lo:
            raise ValueError(f"KERNEL_LS_BOUNDS must satisfy hi>lo; got {KERNEL_LS_BOUNDS}")
        ls_init = float(min(max(KERNEL_LS_INIT, ls_lo), ls_hi))
        return dict(ls_lo=ls_lo, ls_hi=ls_hi, ls_init=ls_init, sigma_x=float("nan"), policy_used=pol)

    if pol == "resolution_scaled_local":
        if mass is None:
            # Fallback: if no mass is provided, we cannot do local scaling. Use global as a safe default.
            ls_lo, ls_hi, base = _kernel_bounds_from_resolution_global(ds, upper_factor=uf, lower_factor=lf, stat=st, npts=np_)
            ls_init = float(np.sqrt(ls_lo * ls_hi))
            return dict(ls_lo=float(ls_lo), ls_hi=float(ls_hi), ls_init=float(ls_init), sigma_x=float(base), policy_used="resolution_scaled_global")

        ls_lo, ls_hi, base = _kernel_bounds_from_resolution_local(ds, float(mass), upper_factor=uf, lower_factor=lf)

        # v13.5: optional floor/cap on ℓ_hi to avoid overly tight local bounds when σ(m)/m decreases with m.
        floor_mode = str(KERNEL_LS_LOCAL_HI_FLOOR_MODE).lower().strip()
        if floor_mode == "dataset_stat":
            try:
                base_stat = float(_sigma_x_stat_cached(ds, stat=st, npts=np_))
                hi_floor = float(uf) * float(base_stat) * float(KERNEL_LS_LOCAL_HI_FLOOR_FACTOR)
                if np.isfinite(hi_floor):
                    ls_hi = max(float(ls_hi), float(hi_floor))
            except Exception:
                pass

        cap_frac = KERNEL_LS_LOCAL_HI_CAP_XRANGE_FRAC
        if cap_frac is not None:
            try:
                cap_frac = float(cap_frac)
                mlo = float(ds.data_low if ds.data_low is not None else ds.m_low)
                mhi = float(ds.data_high if ds.data_high is not None else ds.m_high)
                xlo = float(np.log(max(mlo, 1e-12)) if PRE_LOG else mlo)
                xhi = float(np.log(max(mhi, 1e-12)) if PRE_LOG else mhi)
                xr = abs(xhi - xlo)
                cap = float(cap_frac) * float(xr)
                if np.isfinite(cap) and cap > 0:
                    ls_hi = min(float(ls_hi), float(cap))
            except Exception:
                pass

        # Ensure strict ordering and a stable initializer
        ls_hi = max(float(ls_hi), float(ls_lo) * 1.001)
        ls_init = float(np.sqrt(ls_lo * ls_hi))
        return dict(ls_lo=float(ls_lo), ls_hi=float(ls_hi), ls_init=float(ls_init), sigma_x=float(base), policy_used=pol)

    if pol == "resolution_scaled_global":
        ls_lo, ls_hi, base = _kernel_bounds_from_resolution_global(ds, upper_factor=uf, lower_factor=lf, stat=st, npts=np_)
        ls_init = float(np.sqrt(ls_lo * ls_hi))
        return dict(ls_lo=float(ls_lo), ls_hi=float(ls_hi), ls_init=float(ls_init), sigma_x=float(base), policy_used=pol)

    raise ValueError(f"Unknown KERNEL_LS_POLICY='{pol}' (expected 'manual', 'resolution_scaled_local', or 'resolution_scaled_global')")


def make_kernel_for_dataset(ds: DatasetConfig,
                            *,
                            mass: Optional[float] = None,
                            policy: Optional[str] = None,
                            upper_factor: Optional[float] = None,
                            lower_factor: Optional[float] = None,
                            stat: Optional[str] = None,
                            npts: Optional[int] = None,
                            optimized_by_year: Optional[Dict[str, object]] = None):
    """Create a fresh ConstantKernel*RBF kernel for a given DatasetConfig.

    v13 notes / bug fixes:
      - The default 'resolution_scaled' behavior is now **local**: LS bounds are computed from σ(mass)
        when a mass hypothesis is provided (scan + expected-band code paths).
      - A new explicit policy name is supported for clarity:
            KERNEL_LS_POLICY = 'resolution_scaled_local'  (default in v13)
            KERNEL_LS_POLICY = 'resolution_scaled_global' (legacy v11/v12 behavior)
      - Avoid `globals().get(...)` for kernel settings so parallel workers (joblib/loky) inherit the
        configured policy/factors correctly.
    """
    opt = optimized_by_year or {}
    ds_key = str(getattr(ds, "key", ""))

    # -----------------------------
    # (1) Explicit per-dataset numeric override (highest priority)
    # -----------------------------
    bounds_by_ds = dict(KERNEL_LS_BOUNDS_BY_DATASET or {})
    init_by_ds   = dict(KERNEL_LS_INIT_BY_DATASET or {})

    if ds_key in bounds_by_ds:
        b = bounds_by_ds[ds_key]
        if b is not None and len(b) == 2:
            ls_lo, ls_hi = float(b[0]), float(b[1])
            if ls_hi <= ls_lo:
                raise ValueError(f"KERNEL_LS_BOUNDS_BY_DATASET[{ds_key}] must satisfy hi>lo; got {b}")
            ls_init = float(init_by_ds.get(ds_key, math.sqrt(ls_lo * ls_hi)))
            ls_init = float(min(max(ls_init, ls_lo), ls_hi))
            if bool(PRINT_KERNEL_BOUNDS):
                print(f"[kernel] {ds_key}: explicit bounds override: ({ls_lo:.6g}, {ls_hi:.6g}) init={ls_init:.6g}")
            return (
                skgp.kernels.ConstantKernel(float(KERNEL_CONSTANT_INIT), KERNEL_CONSTANT_BOUNDS)
                * skgp.kernels.RBF(length_scale=float(ls_init), length_scale_bounds=(float(ls_lo), float(ls_hi)))
            )

    # -----------------------------
    # (2) Optimized file entry (optional; disabled by default)
    # -----------------------------
    pol_now = str(policy if policy is not None else KERNEL_LS_POLICY).lower().strip()
    if pol_now == "resolution_scaled":
        pol_now = "resolution_scaled_local"

    # For local-scaling policy, absolute LS bounds from a file generally do not make sense because
    # the correct scale changes with mass. We therefore only apply optimized-file bounds for
    # manual/global policies.
    allow_opt_file = bool(LOAD_OPTIMIZED_LS_IF_AVAILABLE) and (pol_now in ["manual", "resolution_scaled_global"])

    if allow_opt_file and ds_key in opt:
        ent = opt[ds_key]
        try:
            b = ent.get("length_scale_bounds", None)
            if b is not None and len(b) == 2:
                ls_lo, ls_hi = float(b[0]), float(b[1])
                ls_init = float(ent.get("length_scale_init", min(max(KERNEL_LS_INIT, ls_lo), ls_hi)))
                ls_init = float(min(max(ls_init, ls_lo), ls_hi))
                if bool(PRINT_KERNEL_BOUNDS):
                    src = ent.get("_source", "optimized_file")
                    print(f"[kernel] {ds_key}: optimized-file bounds ({src}): ({ls_lo:.6g}, {ls_hi:.6g}) init={ls_init:.6g}")
                return (
                    skgp.kernels.ConstantKernel(float(KERNEL_CONSTANT_INIT), KERNEL_CONSTANT_BOUNDS)
                    * skgp.kernels.RBF(length_scale=float(ls_init), length_scale_bounds=(float(ls_lo), float(ls_hi)))
                )
        except Exception:
            pass

    # -----------------------------
    # (3) Policy-based kernel choice
    # -----------------------------
    info = compute_kernel_ls_bounds(ds, mass=mass, policy=policy, upper_factor=upper_factor,
                                   lower_factor=lower_factor, stat=stat, npts=npts)
    ls_lo, ls_hi, ls_init = float(info["ls_lo"]), float(info["ls_hi"]), float(info["ls_init"])
    pol_used = str(info.get("policy_used", pol_now))

    if bool(PRINT_KERNEL_BOUNDS):
        units = "ln(m)" if PRE_LOG else "m"
        sx = info.get("sigma_x", float("nan"))
        if np.isfinite(sx):
            print(f"[kernel] {ds_key}: policy={pol_used}  σ_x={sx:.6g} [{units}]  bounds=({ls_lo:.6g},{ls_hi:.6g})  init={ls_init:.6g}")
        else:
            print(f"[kernel] {ds_key}: policy={pol_used}  bounds=({ls_lo:.6g},{ls_hi:.6g})  init={ls_init:.6g}")

    return (
        skgp.kernels.ConstantKernel(float(KERNEL_CONSTANT_INIT), KERNEL_CONSTANT_BOUNDS)
        * skgp.kernels.RBF(length_scale=float(ls_init), length_scale_bounds=(float(ls_lo), float(ls_hi)))
    )

def _fit_gpr(X_train: np.ndarray,
             y_train: np.ndarray,
             restarts: int,
             kernel=None,
             *,
             optimize: bool = True):
    """Fit a GaussianProcessRegressor with the notebook's preprocessing convention.

    Parameters
    ----------
    optimize:
        - True  (default): optimize kernel hyperparameters (with `restarts` restarts).
        - False: freeze kernel hyperparameters (no optimizer; restarts forced to 0).

    Notes
    -----
    `optimize=False` is useful for "procedural toys" studies where you want to include
    Poisson fluctuations but *not* allow the GP hyperparameters to chase small bumps/tails.
    """
    X_in, y_in, alpha = _preprocess_xy_for_gpr(X_train, y_train)
    ker = PAPER_KERNEL if kernel is None else kernel

    # Avoid sharing a mutable kernel instance across fits (especially in parallel mode)
    try:
        ker = sk_clone(ker)
    except Exception:
        pass

    n_restarts = int(restarts) if bool(optimize) else 0
    opt = "fmin_l_bfgs_b" if bool(optimize) else None

    try:
        gpr = GaussianProcessRegressor(
            kernel=ker,
            alpha=alpha,
            n_restarts_optimizer=n_restarts,
            optimizer=opt,
            normalize_y=False,
        )
    except TypeError:
        # Very old sklearn fallback: no `optimizer` kwarg
        gpr = GaussianProcessRegressor(
            kernel=ker,
            alpha=alpha,
            n_restarts_optimizer=n_restarts,
            normalize_y=False,
        )

    return gpr.fit(X_in.reshape(-1, 1), y_in)

def _predict_counts_from_log_gpr(gpr: GaussianProcessRegressor, X_query: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    Xq = np.asarray(X_query, dtype=float)
    Xq_in = np.log(np.clip(Xq, 1e-12, None)) if PRE_LOG else Xq.copy()

    mu, cov = gpr.predict(Xq_in.reshape(-1, 1), return_cov=True)
    mu = np.asarray(mu, dtype=float).reshape(-1)
    cov = np.asarray(cov, dtype=float)

    if not PRE_LOG:
        return mu, cov

    diag = np.clip(np.diag(cov), 0.0, None)
    E = np.exp(mu + 0.5 * diag)

    # lognormal moment transform
    C = np.clip(cov, -40.0, 40.0)
    EyEj = np.outer(E, E)
    cov_y = EyEj * (np.exp(C) - 1.0)
    return E, cov_y



def _predict_counts_mean_var_from_log_gpr(gpr: GaussianProcessRegressor, X_query: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    '''Fast diagonal-only prediction: returns (mean, variance) in count-space.

    Uses gpr.predict(return_std=True) which avoids constructing an NxN covariance matrix.
    If PRE_LOG is True, assumes the GPR is over log-counts and applies the lognormal moment transform.
    '''
    Xq = np.asarray(X_query, dtype=float).reshape(-1)
    Xq_in = np.log(np.clip(Xq, 1e-12, None)) if PRE_LOG else Xq.copy()

    mu, std = gpr.predict(Xq_in.reshape(-1, 1), return_std=True)
    mu = np.asarray(mu, dtype=float).reshape(-1)
    std = np.asarray(std, dtype=float).reshape(-1)

    if not PRE_LOG:
        return mu, np.clip(std**2, 0.0, None)

    var_z = np.clip(std**2, 0.0, 40.0)
    mean_y = np.exp(mu + 0.5 * var_z)
    var_y = (np.exp(var_z) - 1.0) * np.exp(2.0 * mu + var_z)  # = mean_y^2 * (exp(var_z)-1)
    return np.asarray(mean_y, float), np.asarray(var_y, float)

print("Preprocess + GPR helpers ready (v11).")


## Signal template and CLs

We build a **binned Gaussian template** in the blind window using the dataset \(\sigma(m)\).

CLs can be run in:
- **asymptotic** mode (fast),
- **toys** mode (more robust but slower).


In [ ]:
# =============================================================================
# Cell 4 — Template + CLs (single dataset or concatenated bins)  (v15)
# =============================================================================
from math import erf, sqrt

def gaussian_bin_integrals(edges: np.ndarray, m0: float, sigma: float) -> np.ndarray:
    """Per-bin probability masses under a unit-normalized Gaussian.

    This returns p_i = ∫_{bin i} N(m0, sigma) dm, with no additional renormalization.
    For a finite histogram range, sum(p_i) = CDF(max_edge) - CDF(min_edge) ≤ 1.
    """
    e = np.asarray(edges, dtype=float)
    sigma = float(sigma)
    if not np.isfinite(sigma) or sigma <= 0:
        # Defensive fallback: uniform mass across bins
        out = np.full(max(1, e.size - 1), 1.0 / max(1, e.size - 1), dtype=float)
        return out

    z = (e - float(m0)) / (sqrt(2.0) * sigma)
    cdf = 0.5 * (1.0 + np.vectorize(erf)(z))
    integ = np.diff(cdf)
    return np.clip(integ, 0.0, None)

def normalize_template(w: np.ndarray) -> np.ndarray:
    """Normalize a non-negative template to sum to unity on the provided bins."""
    w = np.asarray(w, dtype=float)
    s = float(np.sum(w))
    if not np.isfinite(s) or s <= 0:
        return np.full_like(w, 1.0 / max(1, w.size))
    return w / s

def template_fraction_in_edges(edges: np.ndarray, mass: float, sigma_val: float) -> float:
    """Return Σ_i p_i for the given edges (i.e. the Gaussian probability mass in that range)."""
    return float(np.sum(gaussian_bin_integrals(edges, mass, sigma_val)))

def build_template(edges: np.ndarray, mass: float, sigma_val: float, *, norm: Optional[str] = None) -> np.ndarray:
    """Build a Gaussian signal template on the given bin edges.

    norm:
      - None: uses global TEMPLATE_NORM_MODE (default set in Inputs)
      - "window"/"range": renormalize to Σ w_i = 1 over the provided bins
      - "pdf"/"none": do not renormalize (w_i are Gaussian bin probabilities)

    In "pdf" mode, Σ w_i over the blind-window bins equals the signal fraction in that window.
    """
    if norm is None:
        norm = str(globals().get("TEMPLATE_NORM_MODE", "window")).lower().strip()
    else:
        norm = str(norm).lower().strip()

    w = gaussian_bin_integrals(edges, mass, sigma_val)

    if norm in ("pdf", "none", "raw"):
        return w

    if norm in ("window", "range"):
        return normalize_template(w)

    raise ValueError(f"Unknown template norm mode: {norm!r}")


def draw_bkg_mvn_nonneg(mean: np.ndarray,
                        cov: Optional[np.ndarray],
                        size: int,
                        rng: np.random.Generator,
                        *,
                        method: Optional[str] = None,
                        max_tries: Optional[int] = None) -> np.ndarray:
    """Draw non-negative MVN background toys.

    method:
      - "clip": draw MVN then clip negatives to 0 (fast, distorts correlations near 0)
      - "reject": rejection sample until all components are non-negative (best fidelity, may be slower)
      - "reject_then_clip": rejection sample with a fallback clip fill if acceptance is low

    This function is used for expected-band toys and toy-CLs modes.
    """
    m = np.asarray(mean, dtype=float).reshape(-1)
    size = int(max(1, size))
    if cov is None:
        return np.tile(m, (size, 1))

    C = np.asarray(cov, dtype=float)
    method = str(method if method is not None else globals().get("MVN_TRUNC_METHOD", "clip")).lower().strip()
    max_tries = int(max_tries if max_tries is not None else globals().get("MVN_TRUNC_MAX_TRIES", 80))

    # Helper to propose MVN draws robustly
    def _propose(n: int) -> np.ndarray:
        try:
            return rng.multivariate_normal(m, C, size=n, check_valid="ignore", tol=1e-8)
        except Exception:
            diag = np.clip(np.diag(C), 0.0, None)
            return rng.normal(loc=m, scale=np.sqrt(diag), size=(n, m.size))

    if method == "clip":
        return np.clip(_propose(size), 0.0, None)

    # rejection sampling
    out = np.empty((size, m.size), dtype=float)
    filled = 0
    tries = 0
    while filled < size and tries < max_tries:
        # oversample a bit to reduce loop count when acceptance is decent
        n_prop = max(1, int(1.3 * (size - filled)))
        prop = _propose(n_prop)
        ok = np.all(prop >= 0.0, axis=1)
        acc = prop[ok]
        if acc.size > 0:
            n_take = min(acc.shape[0], size - filled)
            out[filled:filled+n_take] = acc[:n_take]
            filled += n_take
        tries += 1

    if filled < size:
        if method == "reject_then_clip":
            # fill remainder with clipped proposals
            rem = size - filled
            out[filled:] = np.clip(_propose(rem), 0.0, None)
            filled = size
        else:
            # strict reject mode: fall back to mean if we failed to fill
            out[filled:] = np.tile(m, (size - filled, 1))
            filled = size

    return out

def _safe_mvn_draw(mean: np.ndarray, cov: Optional[np.ndarray], size: int, rng: np.random.Generator) -> np.ndarray:
    """Backward-compatible alias used throughout earlier notebook versions."""
    return draw_bkg_mvn_nonneg(mean, cov, size=size, rng=rng)

def _log_lr(n: np.ndarray, b: np.ndarray, s: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    n = np.asarray(n, dtype=float)
    b = np.asarray(b, dtype=float)
    s = np.asarray(s, dtype=float)
    if b.ndim == 1 and n.ndim > 1:
        b = np.broadcast_to(b, n.shape)
    b_eff = np.clip(b, eps, None)
    sb_eff = np.clip(b + s, eps, None)
    term = -s + n * (np.log(sb_eff) - np.log(b_eff))
    return np.sum(term, axis=-1)

def cls_amplitude_asymptotic(A: float,
                             n_obs: np.ndarray,
                             b_mean: np.ndarray,
                             cov: Optional[np.ndarray],
                             template: np.ndarray,
                             eps: float = 1e-12) -> Tuple[float, float, float]:
    A = float(max(A, 0.0))
    b = np.asarray(b_mean, dtype=float)
    s = A * np.asarray(template, dtype=float)
    n = np.asarray(n_obs, dtype=float)

    b_eff = np.clip(b, eps, None)
    c = np.log1p(s / b_eff)
    S = float(np.sum(s))
    lnQ_obs = float(-S + np.dot(c, n))

    cov_term = 0.0
    if cov is not None:
        try:
            cov_term = float(c @ np.asarray(cov, dtype=float) @ c)
        except Exception:
            cov_term = 0.0

    mu_b = float(-S + np.dot(c, b))
    var_b = float(np.dot(c * c, b) + cov_term)
    sd_b = math.sqrt(max(var_b, eps))

    sb = b + s
    mu_sb = float(-S + np.dot(c, sb))
    var_sb = float(np.dot(c * c, sb) + cov_term)
    sd_sb = math.sqrt(max(var_sb, eps))

    def Phi(x: float) -> float:
        return 0.5 * (1.0 + erf(x / sqrt(2.0)))

    z_b = (lnQ_obs - mu_b) / sd_b
    z_sb = (lnQ_obs - mu_sb) / sd_sb

    CL_b = float(Phi(z_b))
    CL_sb = float(Phi(z_sb))
    CL_b = max(CL_b, 1e-9)
    return CL_sb / CL_b, CL_sb, CL_b

def cls_amplitude_toys(A: float,
                       n_obs: np.ndarray,
                       b_mean: np.ndarray,
                       cov: Optional[np.ndarray],
                       template: np.ndarray,
                       rng: np.random.Generator,
                       num_toys: int,
                       floor: float = 1e-12) -> Tuple[float, float, float]:
    A = float(max(A, 0.0))
    s = A * template
    lnQ_obs = float(_log_lr(n_obs, b_mean, s, eps=floor))

    b_toys = _safe_mvn_draw(b_mean, cov, size=num_toys, rng=rng)

    n_b = rng.poisson(b_toys)
    lnQ_b = _log_lr(n_b, b_toys, s, eps=floor)

    sb_means = b_toys + s
    n_sb = rng.poisson(sb_means)
    lnQ_sb = _log_lr(n_sb, b_toys, s, eps=floor)

    CL_b = float(np.mean(lnQ_b <= lnQ_obs))
    CL_sb = float(np.mean(lnQ_sb <= lnQ_obs))
    CL_b = max(CL_b, 1e-9)
    return CL_sb / CL_b, CL_sb, CL_b

def cls_limit_for_amplitude(n_obs: np.ndarray,
                            b_mean: np.ndarray,
                            b_cov: Optional[np.ndarray],
                            edges: np.ndarray,
                            mass: float,
                            sigma_val: float,
                            alpha: float = CLS_ALPHA,
                            mode: str = CLS_MODE,
                            num_toys: int = CLS_NUM_TOYS,
                            seed: int = 1) -> Tuple[float, Dict[str, np.ndarray]]:
    template = build_template(edges, mass, sigma_val)
    rng = np.random.default_rng(seed)

    def cls_at(A):
        if mode == "asymptotic":
            return cls_amplitude_asymptotic(A, n_obs, b_mean, b_cov, template)[0]
        return cls_amplitude_toys(A, n_obs, b_mean, b_cov, template, rng, max(1, int(num_toys)))[0]

    b_sum = float(np.sum(b_mean))
    A_lo = 0.0
    f_tmpl = float(np.sum(template))
    if (not np.isfinite(f_tmpl)) or f_tmpl <= 0:
        f_tmpl = 1.0
    A_hi = max(1.0, 3.0 * math.sqrt(max(b_sum, 1.0)) / f_tmpl)

    cls_hi = cls_at(A_hi)
    it = 0
    while cls_hi > alpha and A_hi < 1e7 and it < 40:
        A_hi *= 2.0
        cls_hi = cls_at(A_hi)
        it += 1

    gridA = [A_lo, A_hi]
    gridC = [cls_at(A_lo), cls_hi]
    for _ in range(40):
        Amid = 0.5 * (A_lo + A_hi)
        cls_mid = cls_at(Amid)
        gridA.append(Amid); gridC.append(cls_mid)
        if abs(cls_mid - alpha) < 1e-2:
            A_lo = A_hi = Amid
            break
        if cls_mid > alpha:
            A_lo = Amid
        else:
            A_hi = Amid
        if abs(A_hi - A_lo) <= max(1e-3, 1e-3 * (1.0 + A_hi)):
            break

    return 0.5 * (A_lo + A_hi), {"A_grid": np.array(gridA), "CLs_grid": np.array(gridC)}

print("Template + CLs helpers ready.")


## Analytic p0 utilities (exact profiled LRT with Gaussian template)

For discovery-style p0 at each mass hypothesis \(m_0\), we test for the existence of a **Gaussian signal bump**
(with per-bin template \(w_i\) constructed by integrating the Gaussian resolution over the bin edges and normalizing
over the blind window), using the **profile-likelihood ratio** in the *Poisson + Gaussian-rate-uncertainty* model:

\[
\lambda_i(A,\theta)=\bar b_i + (L\theta)_i + A\,w_i,\qquad \theta\sim\mathcal N(0, I),\quad LL^T=\mathrm{Cov}_b.
\]

We compute the test statistic

\[
q_0 = -2\ln\frac{\max_{\theta}\,L(A=0,\theta)}{\max_{A\ge 0,\theta}\,L(A,\theta)}
\]

by **explicitly** maximizing the penalized log-likelihood under both the null and the constrained alternative (no Wald shortcut).
We then report the one-sided asymptotic discovery mapping \(Z\approx\sqrt{q_0}\) and \(p_0 = 1-\Phi(Z)\).


In [ ]:
# =============================================================================
# Cell 5 — Analytic p0 via exact profiled LRT (Gaussian template + Gaussian prior)
# =============================================================================

def _chol_with_jitter_fallback(C: np.ndarray, jitter0: float = 1e-10) -> np.ndarray:
    """Fallback square-root factor with numerical regularization.

    Kept separate so Cell 5 utilities can be evaluated before the main fitter cell.
    """
    C = np.asarray(C, float)
    if C.ndim != 2 or C.shape[0] != C.shape[1]:
        raise ValueError(f"Covariance must be square; got shape={C.shape}")
    if not np.all(np.isfinite(C)):
        raise ValueError("Covariance contains non-finite entries (NaN/inf).")

    C = 0.5 * (C + C.T)
    B = C.shape[0]
    if B == 0:
        return np.zeros((0, 0), float)

    diag = np.diag(C)
    scale = float(np.max(np.abs(diag))) if diag.size else 1.0
    scale = max(scale, 1.0)
    I = np.eye(B)

    jitter = float(jitter0) * scale
    for _ in range(8):
        try:
            return np.linalg.cholesky(C + jitter * I)
        except np.linalg.LinAlgError:
            jitter *= 10.0

    # Eigenvalue-clipped symmetric square root
    w, V = np.linalg.eigh(C)
    floor = max(1e-12 * scale, float(jitter0) * scale)
    w = np.clip(w, floor, None)
    return V @ np.diag(np.sqrt(w)) @ V.T


def _get_chol(C: np.ndarray) -> np.ndarray:
    # Use the fitter's chol helper if available, otherwise fallback.
    f = globals().get("_chol_with_jitter", None)
    if callable(f):
        return f(C)
    return _chol_with_jitter_fallback(C)

def _profile_theta_given_A(
    n_obs: np.ndarray,
    b_mean: np.ndarray,
    b_cov: np.ndarray,
    template: np.ndarray,
    A_fixed: float,
    th0: Optional[np.ndarray] = None,
) -> Dict[str, object]:
    '''Profile over θ with A fixed (Gaussian-prior additive background model).

    Model:
      n_i ~ Poisson(λ_i),  λ = b_mean + (L θ) + A_fixed * w
      θ ~ N(0, I),  with LL^T = b_cov

    Returns:
      dict(theta_hat, nll, success, message)
    '''
    n = np.asarray(n_obs, float)
    n = np.clip(n, 0.0, None)
    b = np.asarray(b_mean, float)
    b = np.clip(b, 1e-12, None)
    C = np.asarray(b_cov, float)
    w = np.asarray(template, float)

    B = b.size
    if C.shape != (B, B):
        raise ValueError(f"Cov shape mismatch: {C.shape} vs {(B,B)}")
    if w.shape != (B,):
        raise ValueError(f"Template shape mismatch: {w.shape} vs {(B,)}")

    L = _get_chol(C)

    # Numerical floor to keep log finite
    eps = 1e-9 * max(1.0, float(np.median(b)))

    if th0 is None:
        th0 = np.zeros(B, float)
    else:
        th0 = np.asarray(th0, float).reshape(B)

    A_fixed = float(A_fixed)

    def nll_and_grad(th):
        th = np.asarray(th, float)
        lam = b + L @ th + A_fixed * w
        lam_eff = np.maximum(lam, eps)

        ll = np.sum(n * np.log(lam_eff) - lam_eff) - 0.5 * float(np.dot(th, th))
        nll = -float(ll)

        r = (n / lam_eff) - 1.0
        g_th = -(L.T @ r) + th
        return nll, g_th

    res = minimize(
        fun=lambda th: nll_and_grad(th)[0],
        x0=th0,
        jac=lambda th: nll_and_grad(th)[1],
        method="L-BFGS-B",
        options=dict(maxiter=400, ftol=1e-10),
    )

    return dict(
        theta_hat=np.asarray(res.x, float),
        nll=float(res.fun) if np.isfinite(getattr(res, "fun", np.nan)) else float("nan"),
        success=bool(res.success),
        status=int(getattr(res, "status", -1)),
        message=str(getattr(res, "message", "")),
    )

def p0_profiled_gaussian_LRT(
    n_obs: np.ndarray,
    b_mean: np.ndarray,
    b_cov: np.ndarray,
    template: np.ndarray,
) -> Tuple[float, float, float, Dict[str, object]]:
    '''Exact (numerical) profiled LRT for A>=0 vs A=0.

    Returns:
      (p0, Z, q0, info)

    where q0 = -2 ln Lambda, Z ~ sqrt(q0) (one-sided), p0 = 1 - Phi(Z).

    Notes:
      * Uses the same Poisson + Gaussian-prior additive model as fit_A_profiled_gaussian.
      * The "exact" part here is that we explicitly maximize the penalized likelihood under
        both hypotheses (A=0 and A>=0), rather than using the Wald shortcut Z≈Ahat/sigmaA.
    '''
    # Alternative: maximize over (A>=0, θ)
    alt = fit_A_profiled_gaussian(
        n_obs=np.asarray(n_obs, int),
        b_mean=b_mean,
        b_cov=b_cov,
        template=template,
        allow_negative=False,   # discovery test is one-sided
    )

    nll_alt = float(alt.get("nll", float("nan")))
    A_hat = float(alt.get("A_hat", float("nan")))
    sigma_A = float(alt.get("sigma_A", float("nan")))
    ok_alt = bool(alt.get("success", False))

    # Null: maximize over θ with A fixed at 0
    null = _profile_theta_given_A(
        n_obs=n_obs,
        b_mean=b_mean,
        b_cov=b_cov,
        template=template,
        A_fixed=0.0,
        th0=None,
    )
    nll0 = float(null.get("nll", float("nan")))
    ok_null = bool(null.get("success", False))

    q0 = 0.0
    if np.isfinite(nll0) and np.isfinite(nll_alt):
        q0 = max(0.0, 2.0 * (nll0 - nll_alt))

    Z = float(np.sqrt(q0)) if q0 > 0 else 0.0
    p0 = float(norm.sf(Z))
    p0 = min(max(p0, 0.0), 1.0)

    info = dict(
        q0=float(q0),
        Z=float(Z),
        p0=float(p0),
        A_hat=float(A_hat),
        sigma_A=float(sigma_A),
        nll_alt=float(nll_alt),
        nll0=float(nll0),
        ok_alt=bool(ok_alt),
        ok_null=bool(ok_null),
        ok=bool(ok_alt and ok_null),
    )
    return float(p0), float(Z), float(q0), info

print("Analytic p0 (exact profiled LRT) helpers ready.")


## Profiled Gaussian prior signal extraction (A can go negative)

Extraction fit:

- Observations: \(n_i\) in blind window bins
- Background mean: \(b_i\) and covariance \(C\) from GP prediction
- Background nuisance: \(b_i \to b_i + \delta_i\) with \(\delta \sim \mathcal{N}(0,C)\)
- Signal: \(A\,w_i\) with \(w\) a binned Gaussian template

We reparameterize \(\delta=L\theta\) with \(LL^T=C\), so the Gaussian prior is \(-\frac{1}{2}\theta^T\theta\).


In [ ]:
# =============================================================================
# Cell 7 — Profiled Gaussian-prior signal extraction (v15)
# =============================================================================

def _chol_with_jitter(C: np.ndarray, jitter0: float = 1e-10, max_tries: int = 8) -> np.ndarray:
    """Return a numerically safe square-root factor of a (nearly) covariance matrix.

    We primarily attempt a Cholesky factorization with progressively larger diagonal jitter.
    If the matrix is not positive definite due to numerical round-off (common for GP
    posterior covariances), we fall back to an eigenvalue-clipped symmetric square root.

    Notes
    -----
    The returned matrix L is used only as a *covariance square root* via (L @ theta),
    so it need not be strictly lower-triangular in the fallback path—only satisfy
    L L^T ≈ C (up to the applied regularization).
    """
    C = np.asarray(C, float)
    if C.ndim != 2 or C.shape[0] != C.shape[1]:
        raise ValueError(f"Covariance must be square; got shape={C.shape}")
    if not np.all(np.isfinite(C)):
        raise ValueError("Covariance contains non-finite entries (NaN/inf).")

    # Symmetrize to suppress tiny antisymmetric numerical noise
    C = 0.5 * (C + C.T)

    B = C.shape[0]
    if B == 0:
        return np.zeros((0, 0), float)

    # Scale jitter to the problem size (avoid absolute jitter that's too small)
    diag = np.diag(C)
    scale = float(np.max(np.abs(diag))) if diag.size else 1.0
    scale = max(scale, 1.0)

    I = np.eye(B)
    jitter = float(jitter0) * scale

    for _ in range(int(max_tries)):
        try:
            return np.linalg.cholesky(C + jitter * I)
        except np.linalg.LinAlgError:
            jitter *= 10.0

    # --- Fallback: eigenvalue-clipped symmetric square root ---
    # This handles indefinite matrices by projecting to the nearest PSD approximation.
    try:
        w, V = np.linalg.eigh(C)
        floor = max(1e-12 * scale, float(jitter0) * scale)
        w = np.clip(w, floor, None)
        return V @ np.diag(np.sqrt(w)) @ V.T
    except np.linalg.LinAlgError:
        # Last resort: ignore correlations
        d = np.clip(np.diag(C), 1e-12 * scale, None)
        return np.diag(np.sqrt(d))


def profile_theta_given_A(
    n_obs: np.ndarray,
    b_mean: np.ndarray,
    b_cov: np.ndarray,
    template: np.ndarray,
    *,
    A_fixed: float,
    lam_floor: float = 1e-12,
) -> Dict[str, object]:
    """Profile the Gaussian nuisance parameters θ for a *fixed* signal amplitude A.

    Model:
      λ = b_mean + L θ + A · template
      θ ~ N(0, I)

    Returns a dict with:
      theta_hat, delta_b_hat, lambda_hat, b_fit, success, nll
    """
    n = np.asarray(n_obs, float)
    n = np.clip(n, 0.0, None)
    b = np.asarray(b_mean, float)
    C = np.asarray(b_cov, float)
    w = np.asarray(template, float)

    B = b.size
    if B == 0:
        return dict(theta_hat=np.array([]), delta_b_hat=np.array([]), lambda_hat=np.array([]), b_fit=np.array([]),
                    success=False, nll=float("nan"))

    b = np.clip(b, 1e-12, None)
    L = _chol_with_jitter(C)

    # Relative floor helps in very low-count bins
    eps = max(float(lam_floor), 1e-9 * max(1.0, float(np.median(b))))

    A = float(A_fixed)

    def nll_and_grad(theta: np.ndarray):
        th = np.asarray(theta, float)
        delta_b = (L @ th).astype(float)
        lam = (b + delta_b + A * w).astype(float)

        lam_eff = np.maximum(lam, eps)
        r = (n / lam_eff) - 1.0

        nll = -float(np.sum(n * np.log(lam_eff) - lam_eff)) + 0.5 * float(th @ th)

        gth = -(L.T @ r - th)

        bad = lam < eps
        if np.any(bad):
            delta = (eps - lam[bad])
            k = 1e6
            nll += float(k * np.dot(delta, delta))
            dpen = (-2.0 * k) * delta
            gth += (L[bad].T @ dpen)

        return nll, np.asarray(gth, float)

    x0 = np.zeros(B, float)
    res = minimize(
        fun=lambda th: nll_and_grad(th)[0],
        x0=x0,
        jac=lambda th: nll_and_grad(th)[1],
        method="L-BFGS-B",
        bounds=[(None, None)] * B,
        options=dict(maxiter=500, ftol=1e-10),
    )

    thhat = np.asarray(res.x, float)
    delta_b = (L @ thhat).astype(float)
    lamhat = (b + delta_b + A * w).astype(float)

    return dict(
        theta_hat=thhat,
        delta_b_hat=delta_b,
        lambda_hat=lamhat,
        b_fit=(b + delta_b).astype(float),
        success=bool(getattr(res, "success", False)),
        nll=float(getattr(res, "fun", float("nan"))),
    )


def fit_A_profiled_gaussian_details(
    n_obs: np.ndarray,
    b_mean: np.ndarray,
    b_cov: np.ndarray,
    template: np.ndarray,
    *,
    allow_negative: bool = True,
    lam_floor: float = 1e-12,
) -> Dict[str, object]:
    """Profile-likelihood fit for A with Gaussian nuisance parameters for the background.

    Returns a dict with:
      A_hat, sigma_A, theta_hat, delta_b_hat, b_fit, lambda_hat, success, nll
    """
    n = np.asarray(n_obs, float)
    n = np.clip(n, 0.0, None)
    b = np.asarray(b_mean, float)
    C = np.asarray(b_cov, float)
    w = np.asarray(template, float)

    B = b.size
    if B == 0:
        return dict(A_hat=float("nan"), sigma_A=float("nan"), theta_hat=np.array([]), delta_b_hat=np.array([]),
                    b_fit=np.array([]), lambda_hat=np.array([]), success=False, nll=float("nan"))

    b = np.clip(b, 1e-12, None)
    L = _chol_with_jitter(C)

    # Relative floor helps in very low-count bins
    eps = max(float(lam_floor), 1e-9 * max(1.0, float(np.median(b))))

    # --- GLS start for A and σA (robust initialisation) ---
    def _gls_start_and_sigma():
        V = C + np.diag(np.clip(b, 1.0, None))
        V = V + (eps * 100.0) * np.eye(B)
        try:
            Vinv_w = np.linalg.solve(V, w)
            denom = float(np.dot(w, Vinv_w))
            if (not np.isfinite(denom)) or denom <= 0:
                raise ValueError("non-positive denom")
            Vinv_r = np.linalg.solve(V, (n - b))
            num = float(np.dot(w, Vinv_r))
            A0 = num / denom
            sig0 = float(np.sqrt(1.0 / denom))
            return float(A0), float(sig0)
        except Exception:
            A0 = float(np.sum(n - b))
            sig0 = float(np.sqrt(np.sum(np.clip(b, 1.0, None))))
            return float(A0), float(sig0)

    A0, sig0 = _gls_start_and_sigma()
    if not allow_negative:
        A0 = max(0.0, A0)

    def nll_and_grad(x: np.ndarray):
        A = float(x[0])
        th = np.asarray(x[1:], float)

        lam = b + L @ th + A * w
        lam_eff = np.maximum(lam, eps)

        ll = np.sum(n * np.log(lam_eff) - lam_eff) - 0.5 * float(np.dot(th, th))
        nll = -float(ll)

        r = (n / lam_eff) - 1.0
        gA = -float(np.dot(w, r))
        gth = -(L.T @ r - th)

        bad = lam < eps
        if np.any(bad):
            delta = (eps - lam[bad])
            k = 1e6
            nll += float(k * np.dot(delta, delta))
            dpen = (-2.0 * k) * delta
            gA += float(np.dot(w[bad], dpen))
            gth += (L[bad].T @ dpen)

        g = np.concatenate(([gA], np.asarray(gth, float)))
        return nll, g

    bounds = None
    if not allow_negative:
        bounds = [(0.0, None)] + [(None, None)] * B

    x0 = np.concatenate(([A0], np.zeros(B)))
    res = minimize(
        fun=lambda x: nll_and_grad(x)[0],
        x0=x0,
        jac=lambda x: nll_and_grad(x)[1],
        method="L-BFGS-B",
        bounds=bounds,
        options=dict(maxiter=500, ftol=1e-10),
    )

    Ahat = float(res.x[0])
    thhat = np.asarray(res.x[1:], float)
    delta_b = (L @ thhat).astype(float)
    lamhat = (b + delta_b + Ahat * w).astype(float)
    lamhat_eff = np.maximum(lamhat, eps)

    # Profile information for σ_A (Schur complement)
    W = n / (lamhat_eff ** 2)
    I_AA = float(np.sum(W * (w ** 2)))
    I_Ath = (w * W) @ L
    I_thth = (L.T * W) @ L + np.eye(B)

    sigA = float("nan")
    try:
        sol = np.linalg.solve(I_thth, I_Ath.reshape(-1, 1)).reshape(-1)
        I_prof = I_AA - float(I_Ath @ sol)
        varA = 1.0 / max(I_prof, 1e-18)
        sigA = float(np.sqrt(varA))
    except Exception:
        sigA = float(sig0)

    if not allow_negative and Ahat < 0:
        Ahat = 0.0

    return dict(
        A_hat=float(Ahat),
        sigma_A=float(sigA),
        theta_hat=thhat,
        delta_b_hat=delta_b,
        b_fit=(b + delta_b).astype(float),
        lambda_hat=lamhat,
        success=bool(getattr(res, "success", False)),
        nll=float(getattr(res, "fun", float("nan"))),
    )


def fit_A_profiled_gaussian(
    n_obs: np.ndarray,
    b_mean: np.ndarray,
    b_cov: np.ndarray,
    template: np.ndarray,
    *,
    allow_negative: bool = True,
) -> Dict[str, object]:
    """Thin wrapper returning only the essentials (and nll for QA)."""
    d = fit_A_profiled_gaussian_details(n_obs, b_mean, b_cov, template, allow_negative=allow_negative)
    return dict(A_hat=float(d["A_hat"]), sigma_A=float(d["sigma_A"]), success=bool(d["success"]), nll=float(d.get("nll", np.nan)))


print("Profiled A extraction utilities ready (v15).")


## Histogram loading and per-dataset background estimation

For each dataset and mass hypothesis:

1. Load and rebin/limit the histogram to the dataset range.
2. Exclude the blind window from training.
3. Fit GPR on sidebands using the **single global kernel**.
4. Predict \(\mu\) and \(C\) in the blind window in **count space**.


In [ ]:
# =============================================================================
# Cell 7 — Histogram access + per-dataset background estimation  (v11)
# =============================================================================

@dataclass
class BlindPrediction:
    mu: np.ndarray
    cov: np.ndarray
    obs: np.ndarray
    edges: np.ndarray
    sigma_val: float
    blind: Tuple[float, float]
    blind_train: Tuple[float, float]

    x_full: np.ndarray
    y_full: np.ndarray
    mu_full: np.ndarray
    edges_full: np.ndarray

    integral_density: float

    # v15: record the nsigma choices used for this prediction (useful for plots + provenance)
    blind_nsigma: float = float("nan")
    train_exclude_nsigma: float = float("nan")

    # v13: local resolution scale in the GP x-units (sigma_x(mass))
    sigma_x: float = float("nan")

    # v13: extra GP diagnostics (useful for hyper-parameter scans / QA)
    const_init: float = float("nan")
    const_opt: float = float("nan")
    lml: float = float("nan")
    n_train: int = 0
    n_blind: int = 0


    # v11: debugging / provenance
    kernel_str: str = ""
    ls_lo: float = float("nan")
    ls_hi: float = float("nan")
    ls_init: float = float("nan")
    ls_opt: float = float("nan")

    # v12: optional diagonal variance for the full-range prediction (for plot bands)
    var_full: Optional[np.ndarray] = None


# -----------------------------
# v11: load optimized length-scale bounds safely (optional)
# -----------------------------
def _load_optimized_ls_by_year(path: Optional[str]) -> Dict[str, object]:
    if not path:
        return {}
    p = str(path)
    if not os.path.exists(p):
        return {}

    try:
        with open(p, "r") as f:
            cfg = json.load(f)
    except Exception as e:
        print(f"[kernel][WARN] Failed to read optimized LS config at {p}: {e}")
        return {}

    meta = dict(cfg.get("meta", {}) or {})
    by_year = dict(cfg.get("by_year", cfg.get("results", {})) or {})

    # Legacy file (no meta) is ambiguous when PRE_LOG was changed.
    if len(meta) == 0:
        if bool(globals().get("FORCE_LOAD_OPTIMIZED_LS", False)):
            print(f"[kernel][WARN] Optimized LS file has no meta; FORCE_LOAD_OPTIMIZED_LS=True so loading anyway: {p}")
        else:
            print(f"[kernel][WARN] Optimized LS file has no meta; ignoring for safety (set FORCE_LOAD_OPTIMIZED_LS=True to force): {p}")
            return {}

    # If meta exists, validate transform compatibility
    if len(meta) > 0:
        pre_log_file = bool(meta.get("pre_log", meta.get("PRE_LOG", None)))
        xform_file = str(meta.get("x_transform", "ln" if pre_log_file else "identity")).lower().strip()

        pre_log_now = bool(globals().get("PRE_LOG", True))
        xform_now = "ln" if pre_log_now else "identity"

        ok = (pre_log_file == pre_log_now) and (xform_file == xform_now)
        if not ok and not bool(globals().get("FORCE_LOAD_OPTIMIZED_LS", False)):
            print(f"[kernel][WARN] Optimized LS config transform mismatch; ignoring {p}\n"
                  f"             file: pre_log={pre_log_file} x_transform={xform_file}\n"
                  f"             now : pre_log={pre_log_now} x_transform={xform_now}\n"
                  f"             (set FORCE_LOAD_OPTIMIZED_LS=True to force)")
            return {}
        if not ok and bool(globals().get("FORCE_LOAD_OPTIMIZED_LS", False)):
            print(f"[kernel][WARN] Optimized LS config transform mismatch, but FORCE_LOAD_OPTIMIZED_LS=True so loading {p}")

    # tag entries for debugging
    out = {}
    for k, ent in by_year.items():
        try:
            ent2 = dict(ent)
            ent2["_source"] = p
            out[str(k)] = ent2
        except Exception:
            pass

    if bool(globals().get("DEBUG_PRINT", False)):
        print(f"[kernel][INFO] loaded optimized LS config from {p}: keys={list(out.keys())}")

    return out

_OPTIMIZED_LS_BY_YEAR: Dict[str, object] = {}
try:
    if bool(globals().get("LOAD_OPTIMIZED_LS_IF_AVAILABLE", False)):
        _OPTIMIZED_LS_BY_YEAR = _load_optimized_ls_by_year(globals().get("KERNEL_LS_OPT_FILE", None))
except Exception:
    _OPTIMIZED_LS_BY_YEAR = {}


# -----------------------------
# gp package compatibility wrapper
# -----------------------------
def _gp_model(h, *, kernel=None, **kwargs):
    """Compatibility wrapper for gp.GaussianProcessModel.

    Some versions of your local `gp` package require `kernel` as a mandatory argument.
    We pass the per-dataset kernel used by sklearn so any internal diagnostics reflect
    the actual LS bounds (and not the legacy PAPER_KERNEL defaults).
    """
    ker = PAPER_KERNEL if kernel is None else kernel

    # 1) keyword kernel + keyword h
    try:
        return gp.GaussianProcessModel(h=h, kernel=ker, **kwargs)
    except TypeError as e1:
        # 2) positional kernel, keyword h
        try:
            return gp.GaussianProcessModel(ker, h=h, **kwargs)
        except TypeError:
            # 3) positional kernel, positional h
            try:
                return gp.GaussianProcessModel(ker, h, **kwargs)
            except TypeError:
                raise e1


def _build_model(ds: DatasetConfig, blind: Tuple[float, float], rebin: int, *, kernel=None):
    # Probe histogram edges WITHOUT assuming gp signature defaults
    probe = _gp_model((ds.root_path, ds.hist_name), kernel=kernel)
    edges_all = np.asarray(probe.histogram.axes[0].edges, float)
    first_edge = float(edges_all[0]); last_edge = float(edges_all[-1])

    # Choose histogram range presented to the GP.
    # - If data_low/high are None -> use full histogram extent
    # - Always auto-expand so that [m_low,m_high] is a subset (otherwise your scan would
    #   be asking the GP to predict outside its own domain).
    req_lo = first_edge if ds.data_low is None else float(ds.data_low)
    req_hi = last_edge  if ds.data_high is None else float(ds.data_high)

    lower = max(first_edge, req_lo)
    upper = min(last_edge,  req_hi)

    # Auto-expand to include scan window if user configured a too-narrow gp range
    lower = max(first_edge, min(lower, float(ds.m_low)))
    upper = min(last_edge,  max(upper, float(ds.m_high)))

    manip = gp._hist.manipulation.rebin_and_limit(int(rebin), float(lower), float(upper))

    model = _gp_model(
        (ds.root_path, ds.hist_name),
        kernel=kernel,
        n_restarts_optimizer=0,
        blind_range=blind,
        modify_histogram=manip,
    )
    return model



def _blind_pred_detail(model, gpr: GaussianProcessRegressor, blind: Tuple[float, float]):
    Xc = model.histogram.axes[0].centers
    vals = model.histogram.values().astype(int)
    edges = np.asarray(model.histogram.axes[0].edges, dtype=float)

    msk = (Xc >= blind[0]) & (Xc <= blind[1])
    idx = np.where(msk)[0]
    if idx.size == 0:
        raise RuntimeError("Blind window has no bins")

    Xb = Xc[msk]
    e_slice = edges[idx[0] : idx[-1] + 2]

    mu, cov = _predict_counts_from_log_gpr(gpr, Xb)
    obs = vals[msk]
    return np.asarray(mu, float), np.asarray(cov, float), np.asarray(obs, int), np.asarray(e_slice, float)


def _compute_integral_density(model, mass: float, sigma_val: float) -> float:
    # Counts in +/- 2*sigma divided by total bin-width in that region
    ax = model.histogram.axes[0]
    vals = model.histogram.values().astype(float)
    nb = ax.size

    lo, hi = mass - 2.0 * sigma_val, mass + 2.0 * sigma_val
    i0 = max(0, int(ax.index(lo)))
    i1 = min(int(nb), int(ax.index(hi)) + 1)
    if i1 <= i0:
        i0 = max(0, min(i0, nb - 1))
        i1 = i0 + 1

    integral_counts = float(np.sum(vals[i0:i1]))
    widths = np.asarray(ax.widths, dtype=float)
    if widths.ndim == 0:
        integral_size = (i1 - i0) * float(widths)
    else:
        integral_size = float(np.sum(widths[i0:i1]))
    if not np.isfinite(integral_size) or integral_size <= 0:
        raise ValueError("Non-positive integral_size")

    return float(integral_counts / integral_size)


def _extract_rbf_bounds_and_scale(kernel) -> Tuple[float, float, float]:
    """Best-effort helper for debugging: return (ls_lo, ls_hi, ls_init) from a Constant*RBF kernel."""
    try:
        rbf = kernel.k2 if hasattr(kernel, "k2") else kernel
        b = getattr(rbf, "length_scale_bounds", None)
        ls = getattr(rbf, "length_scale", None)
        if b is None or ls is None:
            return float("nan"), float("nan"), float("nan")
        ls0 = float(np.atleast_1d(ls)[0])
        lo = float(b[0]); hi = float(b[1])
        return lo, hi, ls0
    except Exception:
        return float("nan"), float("nan"), float("nan")


def estimate_background_for_dataset(ds: DatasetConfig, mass: float,
                                   rebin: int = NEIGHBORHOOD_REBIN,
                                   restarts: int = N_RESTARTS,
                                   train_exclude_nsigma: Optional[float] = None) -> BlindPrediction:
    sigma_val = ds.sigma(mass)
    blind = (mass - BLIND_NSIGMA * sigma_val, mass + BLIND_NSIGMA * sigma_val)
    train_nsig = float(train_exclude_nsigma) if (train_exclude_nsigma is not None) else float(globals().get("GP_TRAIN_EXCLUDE_NSIGMA", BLIND_NSIGMA))
    blind_train = (mass - train_nsig * sigma_val, mass + train_nsig * sigma_val)


    # v11: build the per-dataset kernel once and pass it to both gp-model (for I/O) and sklearn fit
    kernel = make_kernel_for_dataset(ds, mass=mass, optimized_by_year=_OPTIMIZED_LS_BY_YEAR)
    ls_lo, ls_hi, ls_init = _extract_rbf_bounds_and_scale(kernel)
    # v13: local resolution in the GP x-units (used for diagnostics and LS/sigma_x ratios)
    try:
        sigma_x = float(_sigma_x_from_sigma(np.array([mass], float), np.array([sigma_val], float))[0])
    except Exception:
        sigma_x = float("nan")

    # v13: kernel ConstantKernel amplitude (init)
    const_init = float("nan")
    try:
        ck = kernel.k1 if hasattr(kernel, "k1") else kernel
        const_init = float(np.atleast_1d(getattr(ck, "constant_value", np.nan))[0])
    except Exception:
        pass


    model = _build_model(ds, blind, rebin=rebin, kernel=kernel)

    X = model.histogram.axes[0].centers
    y = model.histogram.values().astype(float)

    mask_train = (X < blind_train[0]) | (X > blind_train[1])
    X_train = X[mask_train]
    y_train = y[mask_train]
    n_train = int(np.asarray(X_train).size)


    gpr = _fit_gpr(X_train, y_train, restarts=int(restarts), kernel=kernel)
    # v13: GP fit quality and optimized hyperparameters
    lml = float(getattr(gpr, "log_marginal_likelihood_value_", np.nan))
    const_opt = float("nan")
    try:
        ck_opt = gpr.kernel_.k1 if hasattr(gpr.kernel_, "k1") else gpr.kernel_
        const_opt = float(np.atleast_1d(getattr(ck_opt, "constant_value", np.nan))[0])
    except Exception:
        pass


    mu_blind, cov_blind, obs_blind, edges_blind = _blind_pred_detail(model, gpr, blind)
    n_blind = int(np.asarray(obs_blind).size)


    mu_full, var_full = _predict_counts_mean_var_from_log_gpr(gpr, X)

    integral_density = _compute_integral_density(model, mass, sigma_val)

    # fitted length-scale (after optimization)
    ls_opt = float("nan")
    try:
        rbf_opt = gpr.kernel_.k2 if hasattr(gpr.kernel_, "k2") else gpr.kernel_
        ls_opt = float(np.atleast_1d(getattr(rbf_opt, "length_scale", np.nan))[0])
    except Exception:
        pass

    return BlindPrediction(
        mu=mu_blind, cov=cov_blind, obs=obs_blind, edges=edges_blind,
        sigma_val=sigma_val, blind=blind, blind_train=blind_train,
        x_full=np.asarray(X, float),
        y_full=np.asarray(y, float),
        mu_full=np.asarray(mu_full, float),
        edges_full=np.asarray(model.histogram.axes[0].edges, float),
        integral_density=integral_density,
        blind_nsigma=float(BLIND_NSIGMA), train_exclude_nsigma=float(train_nsig),
        sigma_x=float(sigma_x),
        const_init=float(const_init), const_opt=float(const_opt),
        lml=float(lml),
        n_train=int(n_train), n_blind=int(n_blind),
        var_full=np.asarray(var_full, float),
        kernel_str=str(getattr(gpr, "kernel_", "")),
        ls_lo=float(ls_lo), ls_hi=float(ls_hi), ls_init=float(ls_init), ls_opt=float(ls_opt),
    )

print("Per-dataset background estimation ready (v13).")


## Conversions between \(A\) and \(\epsilon^2\)

We use:

\[
\epsilon^2 = \frac{2\alpha_{\rm EM} A}{3\pi\, m\, f_{\rm rad}(m)\, I(m)}
\]

and the inverse:

\[
A(\epsilon^2) = \epsilon^2 \cdot \frac{3\pi\, m\, f_{\rm rad}(m)\, I(m)}{2\alpha_{\rm EM}}.
\]


In [ ]:
# =============================================================================
# Cell 8 — A <-> epsilon^2 conversions (per dataset)
# =============================================================================
def epsilon2_from_A(ds: DatasetConfig, mass: float, A: float, integral_density: float) -> float:
    frad = ds.frad(mass)
    if (not np.isfinite(frad)) or frad <= 0 or (not np.isfinite(integral_density)) or integral_density <= 0:
        return float("nan")
    return float((2.0 * ALPHA_EM * float(A)) / (3.0 * math.pi * float(mass) * frad * integral_density))

def A_from_epsilon2(ds: DatasetConfig, mass: float, eps2: float, integral_density: float) -> float:
    frad = ds.frad(mass)
    if (not np.isfinite(frad)) or frad <= 0 or (not np.isfinite(integral_density)) or integral_density <= 0:
        return float("nan")
    return float(float(eps2) * (3.0 * math.pi * float(mass) * frad * integral_density) / (2.0 * ALPHA_EM))

print("Conversion helpers ready.")


## Plot helpers and folder layout

Per mass point we save:

- data vs background (full range)
- blind window overlay (data and background prediction)
- signal template overlay scaled to \(A_{\rm up}\) and to \(A_{\hat{}}\)
- s/b template plot


In [ ]:
# =============================================================================
# Cell 9 — Plot helpers + folder layout (v14: template-leakage aware overlays)
# =============================================================================

def _mass_tag(mass_gev: float) -> str:
    return f"m{int(round(mass_gev*1000)):03d}MeV"

def make_mass_folder(base_dir: str, mass: float) -> str:
    d = os.path.join(base_dir, _mass_tag(mass))
    os.makedirs(d, exist_ok=True)
    return d

def ensure_dir(p: str):
    os.makedirs(p, exist_ok=True)

def _grid(ax, *, which="both"):
    ax.grid(True, which=which, alpha=0.25)
    try:
        ax.minorticks_on()
        ax.grid(True, which="minor", alpha=0.12)
    except Exception:
        pass


def _set_title_above(ax, title: str, *, pad: float = 12.0):
    """Set a title with extra padding so it doesn't collide with the top axis spine in saved figures."""
    try:
        ax.set_title(str(title), pad=float(pad))
    except Exception:
        ax.set_title(str(title))

def _shade_blind_window(
    ax,
    blind: Tuple[float, float],
    *,
    blind_train: Optional[Tuple[float, float]] = None,
    alpha: Optional[float] = None,
    color: Optional[str] = None,
    alpha_train: Optional[float] = None,
    color_train: Optional[str] = None,
    label: Optional[str] = "blind window",
    label_train: Optional[str] = "GP training excluded",
    zorder: int = 0,
):
    """Consistent shading for the extraction blind window (and optional GP training exclusion window).

    If `blind_train` is provided and differs from `blind`, the train-exclusion region is shaded first
    with a lighter alpha so reviewers can see both regions.
    """
    a = float(PLOT_BLIND_SHADE_ALPHA if alpha is None else alpha)
    c = str(PLOT_BLIND_SHADE_COLOR if color is None else color)

    # Optional broader exclusion window (used for GP training)
    if blind_train is not None:
        try:
            bt0, bt1 = float(blind_train[0]), float(blind_train[1])
            b0, b1 = float(blind[0]), float(blind[1])
            # Only draw if meaningfully different
            if (bt0 < b0 - 1e-15) or (bt1 > b1 + 1e-15):
                a_tr = float(min(0.18, 0.60 * a) if alpha_train is None else alpha_train)
                c_tr = str(c if color_train is None else color_train)
                ax.axvspan(bt0, bt1, color=str(c_tr), alpha=float(a_tr), lw=0, zorder=int(zorder), label=label_train)
        except Exception:
            pass

    # Extraction blind window
    try:
        ax.axvspan(float(blind[0]), float(blind[1]), color=str(c), alpha=float(a), lw=0, zorder=int(zorder), label=label)
        ax.axvline(float(blind[0]), color="0.25", lw=0.8, alpha=0.6, zorder=int(zorder) + 1)
        ax.axvline(float(blind[1]), color="0.25", lw=0.8, alpha=0.6, zorder=int(zorder) + 1)
    except Exception:
        pass

def _add_info_box(
    ax,
    text: str,
    *,
    loc: str = "upper right",
    fontsize: int = 9,
    alpha: float = 0.90,
):
    """Anchored, framed text box (useful for fit summaries next to legends)."""
    try:
        from matplotlib.offsetbox import AnchoredText
        at = AnchoredText(str(text), loc=str(loc), prop=dict(size=int(fontsize)), frameon=True, borderpad=0.45)
        at.patch.set_alpha(float(alpha))
        try:
            at.patch.set_boxstyle("round,pad=0.35")
        except Exception:
            pass
        ax.add_artist(at)
        return at
    except Exception:
        # Fallback to a normal axis-text bbox
        ax.text(
            0.98, 0.98, str(text),
            transform=ax.transAxes,
            va="top", ha="right",
            fontsize=int(fontsize),
            bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="0.5", alpha=float(alpha)),
        )
        return None

def _annotate_template_fractions(ax, pred: BlindPrediction, mass: float):
    """Annotate fraction of a unit-area Gaussian that lies in the blind window and in the plotted histogram range."""
    try:
        f_win  = float(template_fraction_in_edges(pred.edges, mass, pred.sigma_val))
        f_full = float(template_fraction_in_edges(pred.edges_full, mass, pred.sigma_val))
        txt = rf"$\sum w_\mathrm{{win}}={f_win:.3f}$,  $\sum w_\mathrm{{full}}={f_full:.3f}$"
        ax.text(0.02, 0.98, txt, transform=ax.transAxes, va="top", ha="left", fontsize=9)
    except Exception:
        pass

def plot_full_range(ds: DatasetConfig, mass: float, pred: BlindPrediction, outpath: str,
                    title_extra: str = "", A_show: Optional[float] = None):
    """Full-range sanity plot: data vs GP mean, with blind-window shading.

    Optionally shows:
      - a diagonal-only 1-sigma band if pred.var_full exists and PLOT_FULL_SHOW_BKG_1SIGMA=True
      - an overlay of GP + A·template on the *full* histogram if A_show is provided
        (template is drawn in PDF mode: non-renormalized Gaussian bin probabilities).
    """
    x = np.asarray(pred.x_full, float)
    y = np.asarray(pred.y_full, float)
    mu = np.asarray(pred.mu_full, float)

    fig, ax = plt.subplots(figsize=(10.5, 5.8))
    ax.step(x, y, where="mid", label="data")
    ax.plot(x, mu, label="GP mean")

    if bool(globals().get("PLOT_FULL_SHOW_BKG_1SIGMA", False)):
        v = getattr(pred, "var_full", None)
        if v is not None:
            v = np.asarray(v, float).reshape(-1)
            if v.size == mu.size and np.isfinite(v).any():
                sig = np.sqrt(np.clip(v, 0.0, None))
                lo = np.clip(mu - sig, 0.0, None)
                hi = mu + sig
                ax.fill_between(x, lo, hi, alpha=0.20, label=r"GP $\pm1\sigma$ (diag)")

    if A_show is not None and np.isfinite(A_show):
        try:
            w_full = build_template(np.asarray(pred.edges_full, float), mass, pred.sigma_val, norm="pdf")
            if w_full.size == mu.size:
                ax.plot(x, mu + float(A_show) * w_full,
                        label=f"GP + A·template (A={float(A_show):.2g})")
        except Exception:
            pass

    _shade_blind_window(ax, pred.blind, blind_train=getattr(pred, "blind_train", None), label="blind window")

    ax.set_xlabel("m (GeV)")
    ax.set_ylabel("Counts / bin")
    _set_title_above(ax, f"{ds.label} — full range @ {mass:.3f} GeV {title_extra}".strip())

    if bool(globals().get("FULL_PLOT_LOGY", False)):
        ax.set_yscale("log")

    _annotate_template_fractions(ax, pred, mass)

    _grid(ax)
    ax.legend(loc="best", frameon=True)
    fig.tight_layout()
    fig.savefig(outpath, dpi=int(globals().get("SAVEFIG_DPI", 300)))
    plt.close(fig)


def plot_blind_window(
    ds: DatasetConfig,
    mass: float,
    pred: BlindPrediction,
    outpath: str,
    *,
    A_up: Optional[float] = None,
    A_hat: Optional[float] = None,
    b_fit: Optional[np.ndarray] = None,
    lam_fit: Optional[np.ndarray] = None,
    title_extra: str = "",
    zoom_half_sigma: Optional[float] = None,
):
    """Reviewer-facing zoom plot around the blind window.

    v14.5 updates:
      - Always shows data *both inside and outside* the blind window (±zoom_half_sigma·σ beyond bounds)
      - Consistent blind-window shading + boundary lines
      - Title padding so it renders above the top axis spine in saved PNGs
    """
    # --- Blind-window arrays (for profiled overlays) ---
    edges_win = np.asarray(pred.edges, float)
    centers_win = 0.5 * (edges_win[:-1] + edges_win[1:])
    w_win = build_template(edges_win, mass, pred.sigma_val, norm=None)
    mu_win = np.asarray(pred.mu, float)

    # --- Full-range arrays (for context: show bins outside blind window) ---
    x_full = np.asarray(pred.x_full, float).reshape(-1)
    y_full = np.asarray(pred.y_full, float).reshape(-1)
    mu_full = np.asarray(pred.mu_full, float).reshape(-1)

    # --- Zoom range: include data outside blind window ---
    zhs = float(zoom_half_sigma if zoom_half_sigma is not None else globals().get("PLOT_BLIND_ZOOM_HALF_SIGMA", 0.5))
    zlo = float(pred.blind[0] - zhs * pred.sigma_val)
    zhi = float(pred.blind[1] + zhs * pred.sigma_val)

    m_zoom = (x_full >= zlo) & (x_full <= zhi)
    xz = x_full[m_zoom]
    yz = y_full[m_zoom]
    muz = mu_full[m_zoom]

    yerr = np.sqrt(np.clip(yz, 1.0, None))

    fig, ax = plt.subplots(figsize=(8.9, 5.8))

    # Blind window shading (always)
    _shade_blind_window(
        ax, pred.blind,
        alpha=float(globals().get("PLOT_BLIND_SHADE_ALPHA", 0.25)),
        color=str(globals().get("PLOT_BLIND_SHADE_COLOR", "0.90")),
        label="blind window",
        zorder=0,
    )

    # Data + GP mean (zoom region, includes outside-blind bins)
    ax.errorbar(xz, yz, yerr=yerr, fmt="o", label="data")
    ax.plot(xz, muz, label="GP mean")

    # Optional 1σ band (diag) using the full-range diagonal variance
    if bool(globals().get("PLOT_BLIND_SHOW_BKG_1SIGMA", True)):
        vfull = getattr(pred, "var_full", None)
        if vfull is not None:
            v = np.asarray(vfull, float).reshape(-1)
            if v.size == mu_full.size:
                vz = v[m_zoom]
                if vz.size == muz.size and np.isfinite(vz).any():
                    sig = np.sqrt(np.clip(vz, 0.0, None))
                    lo = np.clip(muz - sig, 0.0, None)
                    hi = muz + sig
                    ax.fill_between(xz, lo, hi, alpha=0.22, label=r"GP $\pm1\sigma$ (diag)")

    # Profiled / extracted overlays (defined on blind-window bins)
    if lam_fit is not None:
        lam_fit = np.asarray(lam_fit, float).reshape(-1)
        if lam_fit.size == mu_win.size:
            ax.plot(centers_win, lam_fit, lw=2.0, label=r"profiled $\hat\lambda$")

    if b_fit is not None:
        b_fit = np.asarray(b_fit, float).reshape(-1)
        if b_fit.size == mu_win.size:
            ax.plot(centers_win, b_fit, lw=1.8, ls=":", label=r"shifted $\hat b$")

    if A_hat is not None and np.isfinite(A_hat):
        ax.plot(centers_win, mu_win + float(A_hat) * w_win, lw=2.0, label=r"$\mu+\hat A\,w$")
    if A_up is not None and np.isfinite(A_up):
        ax.plot(centers_win, mu_win + float(A_up) * w_win, lw=2.0, ls="--", label=r"$\mu+A_{\rm up}\,w$")

    _annotate_template_fractions(ax, pred, mass)

    ax.set_xlabel("mass [GeV]")
    ax.set_ylabel("counts / bin")

    title = f"{ds.label} blind-window fit @ m={mass*1000:.1f} MeV"
    if title_extra:
        title += " " + str(title_extra)
    _set_title_above(ax, title, pad=float(globals().get("TITLE_PAD", 12.0)))

    ax.set_xlim(zlo, zhi)

    _grid(ax)
    ax.legend(loc=str(globals().get("PLOT_BLIND_LEGEND_LOC", "best")), fontsize=9, frameon=True)
    fig.tight_layout()
    fig.savefig(outpath, dpi=int(globals().get("SAVEFIG_DPI", 180)))
    plt.close(fig)

def plot_s_over_b(ds: DatasetConfig, mass: float, pred: BlindPrediction, A_show: float, outpath: str):
    """Reviewer-facing diagnostic: per-bin s/b in the blind window for the shown amplitude."""
    edges = np.asarray(pred.edges, float)
    centers = 0.5*(edges[:-1] + edges[1:])

    w = build_template(edges, mass, pred.sigma_val, norm=None)
    s = float(A_show) * w
    b = np.clip(np.asarray(pred.mu, float), 1e-12, None)

    fig, ax = plt.subplots(figsize=(8.6, 4.8))
    ax.step(centers, s/b, where="mid")
    ax.set_xlabel("m (GeV)")
    ax.set_ylabel("s/b per bin")
    _set_title_above(ax, f"{ds.label} — s/b in blind window @ {mass:.3f} GeV (A={float(A_show):.2g})")
    _grid(ax)
    fig.tight_layout()
    fig.savefig(outpath, dpi=int(globals().get("SAVEFIG_DPI", 300)))
    plt.close(fig)

def plot_template_leakage(ds: DatasetConfig, mass: float, pred: BlindPrediction, outpath: str, title_extra: str = ""):
    """Standalone plot: Gaussian template probability mass across the full histogram, with blind window shading."""
    edges = np.asarray(pred.edges_full, float)
    centers = 0.5*(edges[:-1] + edges[1:])
    w = build_template(edges, mass, pred.sigma_val, norm="pdf")

    fig, ax = plt.subplots(figsize=(10.5, 4.2))
    ax.step(centers, w, where="mid", label="template (PDF mode)")
    _shade_blind_window(ax, pred.blind, blind_train=getattr(pred, "blind_train", None), label="blind window")

    ax.set_xlabel("m (GeV)")
    ax.set_ylabel("Per-bin probability")
    _set_title_above(ax, f"{ds.label} — template leakage @ {mass:.3f} GeV {title_extra}".strip())

    _annotate_template_fractions(ax, pred, mass)
    _grid(ax)
    ax.legend(loc="best", frameon=True)
    fig.tight_layout()
    fig.savefig(outpath, dpi=int(globals().get("SAVEFIG_DPI", 300)))
    plt.close(fig)

print("Plot helpers ready (v15).")


# =============================================================================
# v14 — Per-mass scan diagnostic plot (full range + blind-edge zoom + extracted component)
# =============================================================================

def plot_scan_diagnostic_panels(
    ds: DatasetConfig,
    mass: float,
    pred: BlindPrediction,
    *,
    A_up: Optional[float] = None,
    A_hat: Optional[float] = None,
    sigma_A: Optional[float] = None,
    outpath: str,
    zoom_half_sigma: Optional[float] = None,
):
    """Multi-panel goodness-of-fit / interpretability plot for a single mass hypothesis.

    Panels:
      1) full histogram range: data, GP mean, (optional) diag ±1σ, and overlays for A_hat / A_up.
      2) zoom region around the blind window: includes ±(zoom_half_sigma·σ) beyond the blind bounds.
         Shows data, GP mean, profiled background and profiled fits for A=0, A_hat, A_up (inside the blind bins).
      3) extracted signal component in the blind window: (data − profiled background) with overlays A_hat·w and A_up·w.

    Notes:
      - Template weights are drawn with the notebook's template convention (TEMPLATE_NORM_MODE).
      - The profiled background is only defined for bins in the blind window (where the GP covariance is used).
    """
    zoom_half_sigma = float(zoom_half_sigma if zoom_half_sigma is not None else globals().get("SCAN_DIAGNOSTIC_ZOOM_HALF_SIGMA", 0.5))

    # --- Blind-window components ---
    edges_win = np.asarray(pred.edges, float)
    centers_win = 0.5 * (edges_win[:-1] + edges_win[1:])
    w_win = build_template(edges_win, mass, pred.sigma_val, norm=None)

    n_obs = np.asarray(pred.obs, float)
    b_mu  = np.asarray(pred.mu, float)
    b_cov = np.asarray(pred.cov, float)

    # Fit details for best-fit A (for plotting δb and λ̂)
    fitd = fit_A_profiled_gaussian_details(
        n_obs, b_mu, b_cov, w_win,
        allow_negative=bool(globals().get("EXTRACT_ALLOW_NEGATIVE", True)),
    )
    Ahat = float(fitd.get("A_hat", np.nan))
    sigA = float(fitd.get("sigma_A", np.nan))
    delta_b_hat = np.asarray(fitd.get("delta_b_hat", np.zeros_like(b_mu)), float)
    b_fit_hat = b_mu + delta_b_hat
    lam_hat = np.asarray(fitd.get("lambda_hat", b_mu), float)

    if A_hat is None or not np.isfinite(A_hat):
        A_hat = Ahat
    if sigma_A is None or not np.isfinite(sigma_A):
        sigma_A = sigA

    # Profile for A=0 and (optionally) A_up
    prof0 = profile_theta_given_A(n_obs, b_mu, b_cov, w_win, A_fixed=0.0)
    b_fit0 = b_mu + np.asarray(prof0.get("delta_b_hat", 0.0), float)
    lam0 = np.asarray(prof0.get("lambda_hat", b_mu), float)

    profU = None
    b_fitU = None
    lamU = None
    if A_up is not None and np.isfinite(A_up):
        profU = profile_theta_given_A(n_obs, b_mu, b_cov, w_win, A_fixed=float(A_up))
        b_fitU = b_mu + np.asarray(profU.get("delta_b_hat", 0.0), float)
        lamU = np.asarray(profU.get("lambda_hat", b_mu), float)

    # Template fractions (useful annotation in PDF-mode)
    f_win = float(np.sum(w_win))
    try:
        f_full = float(template_fraction_in_edges(np.asarray(pred.edges_full, float), mass, pred.sigma_val))
    except Exception:
        f_full = float("nan")

    # --- Full-range components (for context) ---
    x_full = np.asarray(pred.x_full, float)
    y_full = np.asarray(pred.y_full, float)
    mu_full = np.asarray(pred.mu_full, float)
    var_full = getattr(pred, "var_full", None)
    edges_full = np.asarray(pred.edges_full, float)

    # PDF-mode on full histogram range (overlay)
    w_full = None
    try:
        w_full = build_template(edges_full, mass, pred.sigma_val, norm="pdf")
        if w_full.size != mu_full.size:
            w_full = None
    except Exception:
        w_full = None

    # --- Zoom mask in the full histogram ---
    zlo = float(pred.blind[0] - zoom_half_sigma * pred.sigma_val)
    zhi = float(pred.blind[1] + zoom_half_sigma * pred.sigma_val)
    m_zoom = (x_full >= zlo) & (x_full <= zhi)

    # --- Figure ---
    fig = plt.figure(figsize=(11.2, 10.2))
    gs = fig.add_gridspec(3, 1, height_ratios=[1.15, 1.15, 0.85], hspace=0.16)

    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[1, 0])
    ax3 = fig.add_subplot(gs[2, 0])

    # (1) Full range
    ax1.step(x_full, y_full, where="mid", label="data")
    ax1.plot(x_full, mu_full, label="GP mean")

    if bool(globals().get("PLOT_FULL_SHOW_BKG_1SIGMA", False)) and (var_full is not None):
        v = np.asarray(var_full, float).reshape(-1)
        if v.size == mu_full.size and np.isfinite(v).any():
            sig = np.sqrt(np.clip(v, 0.0, None))
            lo = np.clip(mu_full - sig, 0.0, None)
            hi = mu_full + sig
            ax1.fill_between(x_full, lo, hi, alpha=0.20, label=r"GP $\pm1\sigma$ (diag)")

    if w_full is not None and A_hat is not None and np.isfinite(A_hat):
        ax1.plot(x_full, mu_full + float(A_hat) * w_full, label=rf"GP + $\hat{{A}}$ template ($\hat{{A}}$={float(A_hat):.2g})")
    if w_full is not None and A_up is not None and np.isfinite(A_up):
        ax1.plot(x_full, mu_full + float(A_up) * w_full, linestyle="--", label=rf"GP + $A_{{up}}$ template ($A_{{up}}$={float(A_up):.2g})")

    _shade_blind_window(ax1, pred.blind, blind_train=getattr(pred, "blind_train", None), label="blind window")
    ax1.set_ylabel("Counts / bin")
    _set_title_above(ax1, f"{ds.label} — scan diagnostic @ {mass:.3f} GeV")
    if bool(globals().get("FULL_PLOT_LOGY", False)):
        ax1.set_yscale("log")
    _grid(ax1)
    ax1.legend(loc="upper left", frameon=True)

    # Annotate template fractions / fit numbers
    txt = rf"$\sum w_{{win}}$={f_win:.3f}"
    if np.isfinite(f_full):
        txt += rf",  $\sum w_{{full}}$={f_full:.3f}"
    if A_hat is not None and np.isfinite(A_hat) and sigma_A is not None and np.isfinite(sigma_A):
        txt += rf"\n$\hat{{A}}$={float(A_hat):.2g}$\pm${float(sigma_A):.2g}"
    if A_up is not None and np.isfinite(A_up):
        txt += rf"\n$A_{{up}}$={float(A_up):.2g}"
    _add_info_box(ax1, txt, loc="upper right", fontsize=9)

    # (2) Zoom around blind window (include outside bins from the full histogram)
    xz = x_full[m_zoom]
    yz = y_full[m_zoom]
    muz = mu_full[m_zoom]

    yerr = np.sqrt(np.clip(yz, 1.0, None))
    ax2.errorbar(xz, yz, yerr=yerr, fmt="o", label="data")

    ax2.plot(xz, muz, label="GP mean")
    if bool(globals().get("PLOT_FULL_SHOW_BKG_1SIGMA", False)) and (var_full is not None):
        v = np.asarray(var_full, float).reshape(-1)[m_zoom]
        if v.size == muz.size and np.isfinite(v).any():
            sig = np.sqrt(np.clip(v, 0.0, None))
            lo = np.clip(muz - sig, 0.0, None)
            hi = muz + sig
            ax2.fill_between(xz, lo, hi, alpha=0.20, label=r"GP $\pm1\sigma$ (diag)")

    # Blind-window profiled fits (defined only on blind bins)
    ax2.plot(centers_win, b_fit0, linestyle=":", label=r"profiled bkg (A=0)")
    ax2.plot(centers_win, lam_hat, label=rf"profiled fit ($\hat{{A}}$={float(A_hat):.2g})")
    if lamU is not None:
        ax2.plot(centers_win, lamU, linestyle="--", label=rf"profiled fit ($A_{{up}}$={float(A_up):.2g})")

    _shade_blind_window(ax2, pred.blind, blind_train=getattr(pred, "blind_train", None), label="blind window")
    ax2.set_xlim(zlo, zhi)
    ax2.set_ylabel("Counts / bin")
    _set_title_above(ax2, rf"Zoom: [{zlo:.3f}, {zhi:.3f}] GeV (±{zoom_half_sigma:.1f}σ beyond blind)")
    _grid(ax2)
    ax2.legend(loc="best", frameon=True, fontsize=9)

    # (3) Extracted signal component inside blind window
    ywin = np.asarray(pred.obs, float)
    yerr_win = np.sqrt(np.clip(ywin, 1.0, None))
    # Use profiled bkg for Ahat as the baseline (most relevant for extraction)
    sig_data = ywin - b_fit_hat
    ax3.errorbar(centers_win, sig_data, yerr=yerr_win, fmt="o", label=r"data − profiled bkg")

    if A_hat is not None and np.isfinite(A_hat):
        ax3.plot(centers_win, float(A_hat) * w_win, label=rf"$\hat{{A}}\,w$")

    if A_up is not None and np.isfinite(A_up):
        ax3.plot(centers_win, float(A_up) * w_win, linestyle="--", label=rf"$A_{{up}}\,w$")

    ax3.axhline(0.0, color="k", linewidth=0.8, alpha=0.5)
    ax3.set_xlabel("m (GeV)")
    ax3.set_ylabel("Signal counts / bin")
    _set_title_above(ax3, "Extracted signal component (blind bins)")
    _grid(ax3)
    ax3.legend(loc="best", frameon=True, fontsize=9)

    fig.tight_layout()
    fig.savefig(outpath, dpi=int(globals().get("SAVEFIG_DPI", 300)))
    plt.close(fig)


## Single-dataset evaluation at one mass

For each dataset active at a mass hypothesis we compute:

- \(A_{\rm up}\) from CLs (with \(A\ge 0\))
- \(\epsilon^2_{\rm up}\)
- analytic \(p_0\) and \(Z\) on the blind-window sum
- optional extraction \(A_{\hat{}}\) (allow negative)


In [ ]:
# =============================================================================
# Cell 10 — Evaluate a single dataset at one mass hypothesis (v15)
# =============================================================================

@dataclass
class SingleResult:
    dataset: str
    mass: float
    A_up: float
    eps2_up: float
    p0_analytic: float
    Z_analytic: float
    A_hat: float
    sigma_A: float
    extract_success: bool


def _dataset_visibility(ds: DatasetConfig) -> str:
    vis = str((globals().get("DATA_VISIBILITY", {}) or {}).get(ds.key, "observed")).lower().strip()
    if vis not in ("observed", "expected_only"):
        vis = "observed"
    return vis


def _stable_seed_from_tag(tag: str, *, base: int = 0) -> int:
    """Deterministic 32-bit seed from a string tag (stable across sessions and processes)."""
    import hashlib
    h = hashlib.md5(tag.encode("utf-8")).hexdigest()[:8]
    return int((base + int(h, 16)) % (2**31 - 1))


def evaluate_single_dataset(
    ds: DatasetConfig,
    mass: float,
    *,
    do_extraction: bool = True,
    compute_observed: Optional[bool] = None,
    return_fit_details: bool = False,
) -> Tuple[SingleResult, BlindPrediction, Optional[Dict[str, object]]]:
    """Evaluate one dataset at a single mass point.

    If DATA_VISIBILITY[ds.key] == "expected_only", this function will (by default) suppress observed
    quantities that require looking at counts in the blind window (UL/p0/extraction). Sideband-only
    GP fitting still occurs, and the GP prediction objects are returned for QA.

    Returns:
      (SingleResult, BlindPrediction, fit_details_or_None)

    Notes:
      - In v14, the signal template defaults to TEMPLATE_NORM_MODE='pdf' (non-renormalized in the blind window),
        so A is interpreted as a *total* yield parameter (with Σw_win giving the in-window fraction).
    """
    pred = estimate_background_for_dataset(ds, mass)

    # Template range coverage sanity check (important in TEMPLATE_NORM_MODE='pdf')
    if bool(globals().get("TEMPLATE_WARN_IF_RANGE_TRUNCATED", True)):
        try:
            f_full = float(template_fraction_in_edges(np.asarray(pred.edges_full, float), mass, pred.sigma_val))
            eps = float(globals().get("TEMPLATE_RANGE_TRUNC_EPS", 5e-3))
            if np.isfinite(f_full) and abs(1.0 - f_full) > eps:
                print(
                    f"[template][WARN] {ds.key} m={mass:.6g}: "
                    f"Σw over full histogram range = {f_full:.6f} (|1-Σw|>{eps}); "
                    "consider widening the histogram range or interpreting A as yield-in-range."
                )
        except Exception:
            pass

    vis = _dataset_visibility(ds)
    if compute_observed is None:
        compute_observed = (vis == "observed")

    if not compute_observed:
        return (
            SingleResult(
                ds.key,
                float(mass),
                float("nan"),
                float("nan"),
                float("nan"),
                float("nan"),
                float("nan"),
                float("nan"),
                False,
            ),
            pred,
            None,
        )

    # --- CLs upper limit on A ---
    seed = None
    if str(globals().get("CLS_MODE", "asymptotic")).lower().strip() == "toys":
        base = int(globals().get("CLS_SEED_BASE", 12345))
        seed = _stable_seed_from_tag(f"CLS:{ds.key}:{float(mass):.6f}", base=base)

    A_up, _ = cls_limit_for_amplitude(
        n_obs=pred.obs,
        b_mean=pred.mu,
        b_cov=pred.cov,
        edges=pred.edges,
        mass=mass,
        sigma_val=pred.sigma_val,
        alpha=CLS_ALPHA,
        mode=CLS_MODE,
        num_toys=CLS_NUM_TOYS,
        seed=seed,
    )
    eps2_up = epsilon2_from_A(ds, mass, A_up, pred.integral_density)

    # --- Analytic p0/Z using the exact profiled LRT ---
    tmpl = build_template(pred.edges, mass, pred.sigma_val, norm=None)
    p0, Z, q0, info = p0_profiled_gaussian_LRT(pred.obs, pred.mu, pred.cov, tmpl)

    # --- Optional extraction (A_hat, sigma_A) ---
    fit_details = None
    if do_extraction:
        if return_fit_details or bool(globals().get("SAVE_SCAN_DIAGNOSTIC_PLOTS", False)):
            fit_details = fit_A_profiled_gaussian_details(
                pred.obs, pred.mu, pred.cov, tmpl,
                allow_negative=bool(globals().get("EXTRACT_ALLOW_NEGATIVE", True)),
            )
            A_hat = float(fit_details.get("A_hat", float("nan")))
            sigma_A = float(fit_details.get("sigma_A", float("nan")))
            ok = bool(fit_details.get("success", False))
        else:
            fit = fit_A_profiled_gaussian(pred.obs, pred.mu, pred.cov, tmpl,
                                          allow_negative=bool(globals().get("EXTRACT_ALLOW_NEGATIVE", True)))
            A_hat = float(fit["A_hat"])
            sigma_A = float(fit["sigma_A"])
            ok = bool(fit["success"])
    else:
        A_hat, sigma_A, ok = float("nan"), float("nan"), False

    return (
        SingleResult(
            ds.key,
            float(mass),
            float(A_up),
            float(eps2_up),
            float(p0),
            float(Z),
            float(A_hat),
            float(sigma_A),
            bool(ok),
        ),
        pred,
        fit_details if return_fit_details else None,
    )


print("Single-dataset evaluation ready (v15).")


## Combined (simultaneous) fit in overlap regions

We combine datasets at a mass \(m\) by treating \(\epsilon^2\) as the shared parameter:

- For dataset \(d\): \(A_d(\epsilon^2) = \epsilon^2 \cdot K_d(m)\) where \(K_d\) is determined by \(f_{\rm rad}\) and \(I(m)\).
- Total likelihood is the product over datasets and bins.

Implementation:
- concatenate blind-window bins across datasets
- use block-diagonal covariance (datasets independent)


In [ ]:
# =============================================================================
# Cell 11 — Combined CLs on epsilon^2 (v12: refactor for fast toy loops)
# =============================================================================

@dataclass
class CombinedResult:
    mass: float
    eps2_up: float
    p0_analytic: float
    Z_analytic: float

def _concat_block_diag(covs: List[np.ndarray]) -> np.ndarray:
    sizes = [c.shape[0] for c in covs]
    out = np.zeros((sum(sizes), sum(sizes)), float)
    i = 0
    for C in covs:
        n = C.shape[0]
        out[i:i+n, i:i+n] = C
        i += n
    return out

def build_combined_components(mass: float,
                              ds_list: List[DatasetConfig],
                              preds: List[BlindPrediction]) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Build concatenated (obs, b, cov, s_unit) vectors for a shared mass hypothesis.

    s_unit is the expected *counts per unit epsilon^2* in each concatenated bin.
    """
    obs = np.concatenate([p.obs for p in preds]).astype(int)
    b = np.concatenate([p.mu for p in preds]).astype(float)
    cov = _concat_block_diag([p.cov for p in preds]).astype(float)

    templates = [build_template(p.edges, mass, p.sigma_val) for p in preds]
    Ks = [A_from_epsilon2(ds, mass, 1.0, p.integral_density) for ds, p in zip(ds_list, preds)]
    s_unit = np.concatenate([K * t for K, t in zip(Ks, templates)]).astype(float)

    return obs, b, cov, s_unit

def combined_cls_limit_epsilon2_from_vectors(
    obs: np.ndarray,
    b: np.ndarray,
    cov: np.ndarray,
    s_unit: np.ndarray,
    *,
    alpha: float = CLS_ALPHA,
    mode: str = CLS_MODE,
    num_toys: int = CLS_NUM_TOYS,
    seed: int = 1,
    eps_hi0: float = 1e-10,
    eps_hi_max: float = 1.0,
) -> float:
    """Compute a CLs upper limit on epsilon^2 for concatenated bins."""
    obs = np.asarray(obs, int).reshape(-1)
    b = np.asarray(b, float).reshape(-1)
    cov = np.asarray(cov, float)
    s_unit = np.asarray(s_unit, float).reshape(-1)

    rng = np.random.default_rng(int(seed))

    def cls_at_eps2(eps2: float) -> float:
        eps2 = float(max(eps2, 0.0))
        tmpl = s_unit  # counts per unit eps2
        if mode == "asymptotic":
            return float(cls_amplitude_asymptotic(eps2, obs, b, cov, tmpl)[0])

        # toy mode
        s = eps2 * s_unit
        lnQ_obs = float(_log_lr(obs, b, s))

        b_toys = draw_bkg_mvn_nonneg(b, cov, size=int(num_toys), rng=rng)
        n_b = rng.poisson(b_toys)
        lnQ_b = _log_lr(n_b, b_toys, s)

        sb_means = b_toys + s
        n_sb = rng.poisson(sb_means)
        lnQ_sb = _log_lr(n_sb, b_toys, s)

        CL_b = float(np.mean(lnQ_b <= lnQ_obs))
        CL_sb = float(np.mean(lnQ_sb <= lnQ_obs))
        CL_b = max(CL_b, 1e-9)
        return float(CL_sb / CL_b)

    # bracket + bisect in eps2
    eps_lo = 0.0
    eps_hi = float(eps_hi0)
    cls_hi = cls_at_eps2(eps_hi)
    it = 0
    while cls_hi > alpha and eps_hi < float(eps_hi_max) and it < 80:
        eps_hi *= 2.0
        cls_hi = cls_at_eps2(eps_hi)
        it += 1

    for _ in range(80):
        mid = 0.5 * (eps_lo + eps_hi)
        c = cls_at_eps2(mid)
        if abs(c - alpha) < 1e-2:
            eps_lo = eps_hi = mid
            break
        if c > alpha:
            eps_lo = mid
        else:
            eps_hi = mid
        if abs(eps_hi - eps_lo) <= max(1e-16, 1e-3 * max(1e-16, eps_hi)):
            break

    return float(0.5 * (eps_lo + eps_hi))

def combined_cls_limit_epsilon2(
    mass: float,
    ds_list: List[DatasetConfig],
    preds: List[BlindPrediction],
    *,
    alpha: float = CLS_ALPHA,
    mode: str = CLS_MODE,
    num_toys: int = CLS_NUM_TOYS,
    seed: int = 1,
    obs_override: Optional[np.ndarray] = None,
    precomputed: Optional[Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]] = None,
) -> float:
    """Convenience wrapper: build components from ds_list/preds and compute the eps2 limit.

    v12 features:
      - obs_override: use a provided observed vector (for toys) without rebuilding templates
      - precomputed: pass (obs, b, cov, s_unit) directly for speed
    """
    if precomputed is None:
        obs, b, cov, s_unit = build_combined_components(float(mass), ds_list, preds)
    else:
        obs, b, cov, s_unit = precomputed

    if obs_override is not None:
        obs = np.asarray(obs_override, int).reshape(-1)

    return combined_cls_limit_epsilon2_from_vectors(
        obs, b, cov, s_unit,
        alpha=float(alpha), mode=str(mode),
        num_toys=int(num_toys), seed=int(seed)
    )

def evaluate_combined(mass: float,
                      ds_list: List[DatasetConfig],
                      preds: List[BlindPrediction]) -> CombinedResult:
    # --- combined CLs UL ---
    obs, b, cov, s_unit = build_combined_components(float(mass), ds_list, preds)
    eps2_up = combined_cls_limit_epsilon2_from_vectors(obs, b, cov, s_unit)

    # --- combined analytic p0/Z using exact profiled LRT on epsilon^2 ---
    SCALE = float(EPS2_LRT_SCALE)
    tmpl = s_unit / SCALE
    p0, Z, q0, info = p0_profiled_gaussian_LRT(obs, b, cov, tmpl)

    return CombinedResult(float(mass), float(eps2_up), float(p0), float(Z))

print("Combined evaluation ready (v12).")


## Scan driver

We scan over the union of active dataset ranges in 1 MeV steps.

At each mass point:
- run each active dataset individually
- if >=2 datasets overlap, run the combined fit
- optionally write per-mass JSON + plots
- append rows to:
  - per-dataset CSV
  - combined CSV (overlap points)


In [ ]:
# =============================================================================
# Cell 12 — Scan driver + CSV writers (v15)
# =============================================================================

from joblib import Parallel, delayed
from threadpoolctl import threadpool_limits


def active_datasets_for_mass(mass: float, datasets: Dict[str, DatasetConfig]) -> List[DatasetConfig]:
    """Return datasets active at this mass point.

    Guardrails:
      - If SCAN_REQUIRE_TWO_SIDEBANDS=True, require the full blind window (±SCAN_EDGE_GUARD_NSIGMA·σ(m))
        to lie within the dataset's GP training range (data_low/data_high if set; else m_low/m_high).

    This suppresses unstable fits near boundaries where only one sideband is available.
    """
    out: List[DatasetConfig] = []
    m = float(mass)

    require_two = bool(globals().get("SCAN_REQUIRE_TWO_SIDEBANDS", False))
    guard_nsig = float(globals().get("SCAN_EDGE_GUARD_NSIGMA", 2.0))

    for ds in datasets.values():
        if not bool(getattr(ds, "enabled", True)):
            continue
        if (m < float(ds.m_low)) or (m > float(ds.m_high)):
            continue

        if require_two:
            try:
                s = float(ds.sigma(m))
                lo = float(ds.data_low if ds.data_low is not None else ds.m_low)
                hi = float(ds.data_high if ds.data_high is not None else ds.m_high)
                if (m - guard_nsig * s) <= lo:
                    continue
                if (m + guard_nsig * s) >= hi:
                    continue
            except Exception:
                # fail-open on guardrail errors
                pass

        out.append(ds)

    return out


def union_scan_grid(datasets: Dict[str, DatasetConfig], step: float) -> np.ndarray:
    lo = min([d.m_low for d in datasets.values()])
    hi = max([d.m_high for d in datasets.values()])
    masses = np.arange(lo, hi + 0.5 * step, step)
    return np.round(masses, 3)


def _write_json(path: str, payload: dict):
    ensure_dir(os.path.dirname(path) or ".")
    with open(path, "w") as f:
        json.dump(payload, f, indent=2, sort_keys=True)


def _ds_visibility(ds: DatasetConfig) -> str:
    return str((globals().get("DATA_VISIBILITY", {}) or {}).get(ds.key, "observed")).lower().strip()


def _all_observed(ds_list: List[DatasetConfig]) -> bool:
    return all(_ds_visibility(ds) == "observed" for ds in ds_list)


def write_run_metadata(path: str, datasets: Dict[str, DatasetConfig], extra: Optional[dict] = None):
    """Write a lightweight provenance blob for reviewers / reproducibility."""
    keys = [
        # core scan / GP / CLs knobs
        "MASS_STEP_GEV", "BLIND_NSIGMA", "NEIGHBORHOOD_REBIN",
        "PRE_LOG", "LOG_X", "KERNEL_POLICY", "KERNEL_LS_BOUNDS", "KERNEL_LS_INIT",
        "KERNEL_LS_RES_LOWER_FACTOR", "KERNEL_LS_RES_UPPER_FACTOR",
        "KERNEL_LS_LOCAL_HI_FLOOR_MODE", "KERNEL_LS_LOCAL_HI_CAP_XRANGE_FRAC",
        "N_RESTARTS", "CLS_MODE", "CLS_ALPHA", "CLS_NUM_TOYS", "CLS_SEED_BASE",
        "TEMPLATE_NORM_MODE", "TEMPLATE_WARN_IF_RANGE_TRUNCATED", "TEMPLATE_RANGE_TRUNC_EPS",
        # parallelism
        "SCAN_PARALLEL", "SCAN_N_WORKERS", "SCAN_PARALLEL_BACKEND", "SCAN_THREADS_PER_WORKER",
        # expected-band knobs
        "MAKE_UL_BANDS", "RUN_LIMIT_BANDS_ON", "UL_BANDS_N_WORKERS", "UL_BANDS_THREADS_PER_WORKER", "UL_BANDS_SEED",
        "DO_COMBINED", "DO_COMBINED_BANDS", "COMBINED_BANDS_N_TOYS", "COMBINED_BANDS_SEED",
        "MVN_TRUNC_METHOD", "MVN_TRUNC_MAX_TRIES",
        # plotting / output
        "OUTPUT_DIR", "SAVE_PLOTS", "SAVEFIG_DPI",
        "SAVE_SCAN_DIAGNOSTIC_PLOTS", "SCAN_DIAGNOSTIC_PLOT_EVERY_N", "SCAN_DIAGNOSTIC_ZOOM_HALF_SIGMA",
    ]
    cfg = {k: globals().get(k, None) for k in keys}

    ds_cfg = {}
    for k, ds in datasets.items():
        ds_cfg[k] = {
            "label": ds.label,
            "root_path": ds.root_path,
            "hist_name": ds.hist_name,
            "m_low": float(ds.m_low), "m_high": float(ds.m_high),
            "data_low": None if ds.data_low is None else float(ds.data_low),
            "data_high": None if ds.data_high is None else float(ds.data_high),
            "enabled": bool(ds.enabled),
            "visibility": _ds_visibility(ds),
            "sigma_tail_m0": ds.sigma_tail_m0,
            "sigma_tail_slope_floor": float(ds.sigma_tail_slope_floor),
            "sigma_tail_slope_override": ds.sigma_tail_slope_override,
        }

    payload = {
        "version": "v15",
        "created_utc": datetime.datetime.utcnow().isoformat() + "Z",
        "config": cfg,
        "datasets": ds_cfg,
    }
    if extra:
        payload["extra"] = dict(extra)

    _write_json(path, payload)
    print("[meta] wrote run metadata to", path)


def _should_make_scan_diag(m: float) -> bool:
    if not bool(globals().get("SAVE_SCAN_DIAGNOSTIC_PLOTS", False)):
        return False
    explicit = globals().get("SCAN_DIAGNOSTIC_EXPLICIT_MASSES", []) or []
    if len(explicit) > 0:
        try:
            return any(abs(float(m) - float(x)) < 5e-4 for x in explicit)  # ~0.5 MeV tolerance
        except Exception:
            return False
    every = globals().get("SCAN_DIAGNOSTIC_PLOT_EVERY_N", 1)
    if every is None:
        return False
    try:
        every = int(every)
    except Exception:
        return False
    if every <= 0:
        return False
    # Decide based on the MeV tag to avoid drift with float indexing
    return (int(round(float(m) * 1000)) % int(round(every * MASS_STEP_GEV * 1000)) == 0) if every > 1 else True


def run_scan(datasets: Dict[str, DatasetConfig]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    masses = union_scan_grid(datasets, MASS_STEP_GEV)

    # Write metadata once at the start
    if bool(globals().get("WRITE_RUN_METADATA", True)):
        try:
            write_run_metadata(
                str(globals().get("RUN_METADATA_PATH", os.path.join(OUTPUT_DIR, "run_metadata.json"))),
                datasets,
            )
        except Exception as e:
            print("[meta][WARN] failed to write run metadata:", e)

    # Per-mass worker
    def _process_one_mass(m: float):
        m = float(m)
        rows_single_local = []
        rows_comb_local = []

        ds_here = active_datasets_for_mass(m, datasets)
        if not ds_here:
            return rows_single_local, rows_comb_local

        mass_dir = make_mass_folder(OUTPUT_DIR, m) if SAVE_PER_MASS_FOLDERS else OUTPUT_DIR

        preds_here: List[BlindPrediction] = []
        ds_list_here: List[DatasetConfig] = []

        for ds in ds_here:
            ds_dir = os.path.join(mass_dir, ds.key)
            ensure_dir(ds_dir)

            compute_obs = (_ds_visibility(ds) == "observed")

            try:
                res, pred, fitd = evaluate_single_dataset(
                    ds, m,
                    do_extraction=True,
                    compute_observed=compute_obs,
                    return_fit_details=bool(globals().get("SAVE_SCAN_DIAGNOSTIC_PLOTS", False)),
                )
                preds_here.append(pred)
                ds_list_here.append(ds)

                # Only write plots that reveal blind-window counts if compute_obs=True
                if SAVE_PLOTS and compute_obs:
                    # Full-range context (optionally overlay A_hat)
                    A_show = float(res.A_hat) if np.isfinite(res.A_hat) else float(res.A_up)
                    plot_full_range(ds, m, pred, os.path.join(ds_dir, "fit_full.png"), A_show=A_show)

                    # Blind-window plot with both Ahat/Aup overlays when available
                    b_fit = None
                    lam_fit = None
                    if isinstance(fitd, dict):
                        try:
                            b_fit = np.asarray(fitd.get("b_fit", None), float) if fitd.get("b_fit", None) is not None else None
                            lam_fit = np.asarray(fitd.get("lambda_hat", None), float) if fitd.get("lambda_hat", None) is not None else None
                        except Exception:
                            b_fit, lam_fit = None, None

                    plot_blind_window(
                        ds, m, pred, os.path.join(ds_dir, "blind_fit.png"),
                        A_up=float(res.A_up), A_hat=float(res.A_hat),
                        b_fit=b_fit, lam_fit=lam_fit,
                        title_extra="(Ahat + UL overlays)",
                    )
                    plot_s_over_b(ds, m, pred, float(res.A_up), os.path.join(ds_dir, "s_over_b_ul.png"))

                    # v14 diagnostic panels (zoom around blind edges etc.)
                    if _should_make_scan_diag(m):
                        plot_scan_diagnostic_panels(
                            ds, m, pred,
                            A_up=float(res.A_up),
                            A_hat=float(res.A_hat),
                            sigma_A=float(res.sigma_A),
                            outpath=os.path.join(ds_dir, "scan_diagnostic.png"),
                            zoom_half_sigma=float(globals().get("SCAN_DIAGNOSTIC_ZOOM_HALF_SIGMA", 0.5)),
                        )

                if SAVE_FIT_JSON:
                    # If compute_obs=False we still write non-sensitive fit metadata (no observed UL/p0)
                    payload = {
                        "dataset": ds.key,
                        "mass_GeV": float(m),
                        "sigma_val": float(getattr(pred, "sigma_val", np.nan)),
                        "sigma_x": float(getattr(pred, "sigma_x", np.nan)),
                        "blind": list(getattr(pred, "blind", (np.nan, np.nan))),
                        "integral_density": float(getattr(pred, "integral_density", np.nan)),
                        "kernel_str": str(getattr(pred, "kernel_str", "")),
                        "lml": float(getattr(pred, "lml", np.nan)),
                        "const_init": float(getattr(pred, "const_init", np.nan)),
                        "const_opt": float(getattr(pred, "const_opt", np.nan)),
                        "n_train": int(getattr(pred, "n_train", 0)),
                        "n_blind": int(getattr(pred, "n_blind", 0)),
                        "ls_lo": float(getattr(pred, "ls_lo", np.nan)),
                        "ls_hi": float(getattr(pred, "ls_hi", np.nan)),
                        "ls_init": float(getattr(pred, "ls_init", np.nan)),
                        "ls_opt": float(getattr(pred, "ls_opt", np.nan)),
                        "visibility": _ds_visibility(ds),
                    }
                    if compute_obs:
                        payload.update({
                            "A_up": float(res.A_up),
                            "eps2_up": float(res.eps2_up),
                            "p0_analytic": float(res.p0_analytic),
                            "Z_analytic": float(res.Z_analytic),
                            "A_hat": float(res.A_hat),
                            "sigma_A": float(res.sigma_A),
                            "extract_success": bool(res.extract_success),
                        })
                    _write_json(os.path.join(ds_dir, "numbers.json"), payload)

                # Row for scan CSV (NaNs if expected-only)
                sigma_x = float(getattr(pred, "sigma_x", np.nan))
                sigma_val = float(getattr(pred, "sigma_val", np.nan))

                row = {
                    "dataset": ds.key,
                    "mass_GeV": float(res.mass),

                    # local resolution + blind window
                    "sigma_val": sigma_val,
                    "sigma_x": sigma_x,
                    "blind_lo": float(getattr(pred, "blind", (np.nan, np.nan))[0]),
                    "blind_hi": float(getattr(pred, "blind", (np.nan, np.nan))[1]),

                    # GP hyper-parameters / diagnostics
                    "lml": float(getattr(pred, "lml", np.nan)),
                    "const_init": float(getattr(pred, "const_init", np.nan)),
                    "const_opt": float(getattr(pred, "const_opt", np.nan)),
                    "ls_lo": float(getattr(pred, "ls_lo", np.nan)),
                    "ls_hi": float(getattr(pred, "ls_hi", np.nan)),
                    "ls_init": float(getattr(pred, "ls_init", np.nan)),
                    "ls_opt": float(getattr(pred, "ls_opt", np.nan)),

                    "ls_lo_over_sigma_x": (float(getattr(pred, "ls_lo", np.nan)) / sigma_x) if np.isfinite(sigma_x) and sigma_x > 0 else np.nan,
                    "ls_hi_over_sigma_x": (float(getattr(pred, "ls_hi", np.nan)) / sigma_x) if np.isfinite(sigma_x) and sigma_x > 0 else np.nan,
                    "ls_opt_over_sigma_x": (float(getattr(pred, "ls_opt", np.nan)) / sigma_x) if np.isfinite(sigma_x) and sigma_x > 0 else np.nan,

                    # x->GeV-equivalent length scales (useful when PRE_LOG=True)
                    "ls_lo_dmass": float(length_scale_x_to_mass_delta(float(getattr(pred, "ls_lo", np.nan)), m)),
                    "ls_hi_dmass": float(length_scale_x_to_mass_delta(float(getattr(pred, "ls_hi", np.nan)), m)),
                    "ls_opt_dmass": float(length_scale_x_to_mass_delta(float(getattr(pred, "ls_opt", np.nan)), m)),
                    "ls_hi_over_sigma": (float(length_scale_x_to_mass_delta(float(getattr(pred, "ls_hi", np.nan)), m)) / sigma_val) if np.isfinite(sigma_val) and sigma_val > 0 else np.nan,
                    "ls_opt_over_sigma": (float(length_scale_x_to_mass_delta(float(getattr(pred, "ls_opt", np.nan)), m)) / sigma_val) if np.isfinite(sigma_val) and sigma_val > 0 else np.nan,

                    "ls_opt_hit_hi": bool(
                        np.isfinite(float(getattr(pred, "ls_opt", np.nan)))
                        and np.isfinite(float(getattr(pred, "ls_hi", np.nan)))
                        and (float(getattr(pred, "ls_opt", np.nan)) >= 0.98 * float(getattr(pred, "ls_hi", np.nan)))
                    ),

                    "n_train": int(getattr(pred, "n_train", 0)),
                    "n_blind": int(getattr(pred, "n_blind", 0)),
                    "kernel_str": str(getattr(pred, "kernel_str", "")),

                    # statistics (may be NaN if expected-only visibility)
                    "A_up": float(res.A_up),
                    "eps2_up": float(res.eps2_up),
                    "p0_analytic": float(res.p0_analytic),
                    "Z_analytic": float(res.Z_analytic),
                    "A_hat": float(res.A_hat),
                    "sigma_A": float(res.sigma_A),
                    "extract_success": bool(res.extract_success),
                    "visibility": _ds_visibility(ds),
                }

                rows_single_local.append(row)

            except Exception as e:
                # Record a failure row (keeps scan outputs aligned)
                try:
                    with open(os.path.join(ds_dir, "error.txt"), "w") as ef:
                        ef.write(str(e))
                except Exception:
                    pass

                try:
                    _sigma_val_fail = float(ds.sigma(m))
                    _sigma_x_fail = float(_sigma_x_from_sigma(np.array([m], float), np.array([_sigma_val_fail], float))[0])
                except Exception:
                    _sigma_val_fail = np.nan
                    _sigma_x_fail = np.nan

                rows_single_local.append({
                    "dataset": ds.key,
                    "mass_GeV": float(m),
                    "sigma_val": float(_sigma_val_fail),
                    "sigma_x": float(_sigma_x_fail),
                    "blind_lo": np.nan,
                    "blind_hi": np.nan,
                    "lml": np.nan,
                    "const_init": np.nan,
                    "const_opt": np.nan,
                    "ls_lo": np.nan,
                    "ls_hi": np.nan,
                    "ls_init": np.nan,
                    "ls_opt": np.nan,
                    "ls_lo_over_sigma_x": np.nan,
                    "ls_hi_over_sigma_x": np.nan,
                    "ls_opt_over_sigma_x": np.nan,
                    "ls_lo_dmass": np.nan,
                    "ls_hi_dmass": np.nan,
                    "ls_opt_dmass": np.nan,
                    "ls_hi_over_sigma": np.nan,
                    "ls_opt_over_sigma": np.nan,
                    "ls_opt_hit_hi": False,
                    "n_train": 0,
                    "n_blind": 0,
                    "kernel_str": "",
                    "A_up": np.nan,
                    "eps2_up": np.nan,
                    "p0_analytic": np.nan,
                    "Z_analytic": np.nan,
                    "A_hat": np.nan,
                    "sigma_A": np.nan,
                    "extract_success": False,
                    "visibility": _ds_visibility(ds),
                    "error": str(e),
                })

        # Combined observed fit only if:
        #   - DO_COMBINED enabled
        #   - >=2 datasets cover this mass
        #   - ALL datasets participating are in visibility="observed"
        if bool(globals().get("DO_COMBINED", False)) and (len(ds_list_here) >= 2) and _all_observed(ds_list_here):
            try:
                comb = evaluate_combined(m, ds_list_here, preds_here)

                if SAVE_PLOTS:
                    cdir = os.path.join(mass_dir, "combined")
                    ensure_dir(cdir)
                    fig, ax = plt.subplots(figsize=(7.8, 4.6))
                    ax.axis("off")
                    ax.text(0.02, 0.78, f"Combined @ {m:.3f} GeV", fontsize=13)
                    ax.text(0.02, 0.55, f"eps2_up = {comb.eps2_up:.3e}", fontsize=13)
                    ax.text(0.02, 0.32, f"p0 = {comb.p0_analytic:.3e}   Z = {comb.Z_analytic:.2f}", fontsize=13)
                    fig.tight_layout()
                    fig.savefig(os.path.join(cdir, "combined_summary.png"), dpi=int(globals().get("SAVEFIG_DPI", 300)))
                    plt.close(fig)

                if SAVE_FIT_JSON:
                    cdir = os.path.join(mass_dir, "combined")
                    ensure_dir(cdir)
                    _write_json(os.path.join(cdir, "numbers.json"), {
                        "mass_GeV": float(m),
                        "datasets": [d.key for d in ds_list_here],
                        "eps2_up": float(comb.eps2_up),
                        "p0_analytic": float(comb.p0_analytic),
                        "Z_analytic": float(comb.Z_analytic),
                    })

                rows_comb_local.append({
                    "mass_GeV": float(comb.mass),
                    "datasets": "+".join([d.key for d in ds_list_here]),
                    "n_datasets": int(len(ds_list_here)),
                    "eps2_up": float(comb.eps2_up),
                    "p0_analytic": float(comb.p0_analytic),
                    "Z_analytic": float(comb.Z_analytic),
                })

            except Exception as e:
                rows_comb_local.append({
                    "mass_GeV": float(m),
                    "datasets": "+".join([d.key for d in ds_list_here]),
                    "n_datasets": int(len(ds_list_here)),
                    "eps2_up": np.nan,
                    "p0_analytic": np.nan,
                    "Z_analytic": np.nan,
                    "error": str(e),
                })

        return rows_single_local, rows_comb_local

    # Main loop (optionally parallel over masses)
    scan_parallel = bool(globals().get("SCAN_PARALLEL", False))
    n_workers = int(globals().get("SCAN_N_WORKERS", 1) or 1)
    backend = str(globals().get("SCAN_PARALLEL_BACKEND", "loky"))
    threads_per_worker = int(globals().get("SCAN_THREADS_PER_WORKER", 1) or 1)

    results = []
    if scan_parallel and n_workers > 1:
        print(f"[scan] running in parallel: n_workers={n_workers}, backend='{backend}', threads/worker={threads_per_worker}")
        with threadpool_limits(limits=threads_per_worker):
            results = Parallel(n_jobs=n_workers, backend=backend)(
                delayed(_process_one_mass)(float(m)) for m in masses
            )
    else:
        print("[scan] running sequentially")
        with threadpool_limits(limits=threads_per_worker):
            for m in masses:
                results.append(_process_one_mass(float(m)))
                # progress print every 25 MeV
                if int(round(float(m) * 1000)) % 25 == 0:
                    print(f"[scan] reached {float(m):.3f} GeV")

    # Flatten results
    rows_single = []
    rows_comb = []
    for rs, rc in results:
        rows_single.extend(rs)
        rows_comb.extend(rc)

    # result tables
    df_single = pd.DataFrame(rows_single)
    if len(df_single) > 0:
        df_single = df_single.sort_values(["dataset", "mass_GeV"]).reset_index(drop=True)

    df_comb = pd.DataFrame(rows_comb)
    if len(df_comb) > 0 and ("mass_GeV" in df_comb.columns):
        df_comb = df_comb.sort_values(["mass_GeV"]).reset_index(drop=True)

    single_path = os.path.join(OUTPUT_DIR, "results_single.csv")
    comb_path = os.path.join(OUTPUT_DIR, "results_combined.csv")
    ensure_dir(OUTPUT_DIR)
    df_single.to_csv(single_path, index=False)
    df_comb.to_csv(comb_path, index=False)
    # Backward-compatible alias (some external scripts expect 'combined.csv')
    comb_path_alias = os.path.join(OUTPUT_DIR, "combined.csv")
    df_comb.to_csv(comb_path_alias, index=False)

    print("Wrote:", single_path)
    print("Wrote:", comb_path)
    print("Wrote:", comb_path_alias)

    return df_single, df_comb


print("Scan driver ready (v15).")


## Summary plots

- Per dataset: \(\epsilon^2_{\rm up}(m)\)
- Combined: \(\epsilon^2_{\rm up}(m)\) in overlap regions


In [ ]:
# =============================================================================
# Cell 14 — Summary plotting utilities (v15.8)
# =============================================================================
#
# v14 update:
#   - Provide publication-style summary curves for BOTH:
#       (1) A_up (signal-yield UL)
#       (2) ε²_up (physics parameter UL)
#   - If expected-band CSVs exist (ul_bands_<ds>.csv), overlay median and ±1σ/2σ bands.
#   - For combined, only ε² plots are produced (A_up is not meaningful).
# =============================================================================



# --- LEE helper (local to summary plotting cell; duplicated to keep cell order independent) ---
def _lee_trials_from_grid_summary(masses: np.ndarray, ds: DatasetConfig, *, indep_width_sigma: float = 2.355) -> float:
    """Effective trials factor for a discrete mass scan (summary plotting helper).

    N_eff ≈ ∫ dm / (W·σ(m)) with W≈FWHM/σ by default, capped by the number of tested mass points.
    """
    m = np.asarray(masses, float)
    m = m[np.isfinite(m)]
    if m.size < 2:
        return 1.0
    m = np.sort(m)

    dm = np.diff(m)
    mids = 0.5 * (m[:-1] + m[1:])
    W = max(float(indep_width_sigma), 1e-6)

    Neff = 0.0
    for dmi, mi in zip(dm, mids):
        try:
            sig = float(ds.sigma(float(mi)))
        except Exception:
            continue
        if not np.isfinite(sig) or sig <= 0:
            continue
        Neff += float(dmi) / (W * float(sig))

    Neff = max(float(Neff), 1.0)
    Neff = min(float(Neff), float(m.size))
    return float(Neff)

def _p_global_from_local_summary(p_local: np.ndarray, *, Neff: float, method: str = "sidak") -> np.ndarray:
    p = np.asarray(p_local, float)
    p = np.clip(p, 0.0, 1.0)
    N = max(float(Neff), 1.0)
    meth = str(method).lower().strip()
    if meth == "bonferroni":
        return np.clip(N*p, 0.0, 1.0)
    return np.clip(-np.expm1(N*np.log1p(-p)), 0.0, 1.0)


def _p_local_from_global_summary(p_global: float, *, Neff: float, method: str = "sidak") -> float:
    """Invert the trials-factor approximation.

    Given a desired global p-value, return the corresponding *local* p-value threshold.
    """
    pg = float(np.clip(p_global, 0.0, 1.0))
    N = max(float(Neff), 1.0)
    meth = str(method).lower().strip()
    if meth == "bonferroni":
        return float(np.clip(pg / N, 0.0, 1.0))
    # Sidak: p_global = 1 - (1 - p_local)^N  =>  p_local = 1 - (1 - p_global)^(1/N)
    return float(np.clip(1.0 - (1.0 - pg)**(1.0 / N), 0.0, 1.0))

def _z_from_p_one_sided_summary(p: np.ndarray) -> np.ndarray:
    from scipy.stats import norm
    p = np.asarray(p, float)
    p = np.clip(p, 1e-300, 1.0)
    return norm.isf(p)


def _maybe_read_csv(path: str) -> Optional[pd.DataFrame]:
    try:
        if os.path.exists(path):
            return pd.read_csv(path)
    except Exception as e:
        print(f"[summary][WARN] failed to read {path}: {e}")
    return None



def _safe_tight_layout(fig, **kwargs):
    """Best-effort tight layout; never crash a production run due to mathtext/layout edge cases."""
    try:
        fig.tight_layout(**kwargs)
    except Exception:
        # Fall back silently; caller may still save the figure.
        pass

def _grid_summary(ax):
    ax.grid(True, which="major", alpha=0.25)
    try:
        ax.minorticks_on()
        ax.grid(True, which="minor", alpha=0.12)
    except Exception:
        pass


def _overlay_expected_bands(ax, df_b: pd.DataFrame, *, x: str, med: str, lo1: str, hi1: str, lo2: str, hi2: str):
    """HEP-style expected bands: 2σ (yellow), 1σ (green), median (dashed)."""
    m = df_b[x].to_numpy(float)
    ax.fill_between(m, df_b[lo2].to_numpy(float), df_b[hi2].to_numpy(float),
                    color="gold", alpha=0.35, label=r"Expected ±2σ")
    ax.fill_between(m, df_b[lo1].to_numpy(float), df_b[hi1].to_numpy(float),
                    color="limegreen", alpha=0.50, label=r"Expected ±1σ")
    ax.plot(m, df_b[med].to_numpy(float), linestyle="--", color="black", linewidth=1.4, label="Expected median")


def plot_dataset_limit_curves(
    df_single: pd.DataFrame,
    ds: DatasetConfig,
    *,
    outdir: str,
    bands_csv: Optional[str] = None,
):
    """Produce dataset-level observed limit curves (A and ε²), with optional expected bands overlay."""
    ensure_dir(outdir)
    sub = df_single[df_single["dataset"] == ds.key].copy()
    if sub.empty:
        print(f"[summary] no rows for dataset {ds.key}")
        return

    # Observed-only points might include NaNs for expected-only visibility
    sub_obs = sub[np.isfinite(sub["eps2_up"])].copy()
    if sub_obs.empty:
        print(f"[summary] dataset {ds.key}: no observed limits (maybe visibility='expected_only')")
        return

    # Load expected bands (if available)
    df_b = None
    if bands_csv is None:
        bands_csv = os.path.join(OUTPUT_DIR, f"ul_bands_{ds.key}.csv")
    df_b = _maybe_read_csv(bands_csv)

    # --- A_up plot ---
    fig, ax = plt.subplots(figsize=(9.2, 5.4))
    ax.plot(sub_obs["mass_GeV"], sub_obs["A_up"], linewidth=1.6, label="Observed")

    if df_b is not None and all(c in df_b.columns for c in ["A_med", "A_lo1", "A_hi1", "A_lo2", "A_hi2"]):
        _overlay_expected_bands(ax, df_b, x="mass_GeV", med="A_med", lo1="A_lo1", hi1="A_hi1", lo2="A_lo2", hi2="A_hi2")

    ax.set_yscale("log")
    ax.set_xlabel("m (GeV)")
    ax.set_ylabel(r"$A_{\rm up}$ (events)")
    ax.set_title(f"{ds.label}: signal-yield upper limit")
    _grid_summary(ax)
    ax.legend(loc="best", frameon=True)
    _safe_tight_layout(fig)
    outA = os.path.join(outdir, f"summary_A_up_{ds.key}.png")
    fig.savefig(outA, dpi=int(globals().get("SAVEFIG_DPI", 300)))
    plt.close(fig)

    # --- eps2_up plot ---
    fig, ax = plt.subplots(figsize=(9.2, 5.4))
    ax.plot(sub_obs["mass_GeV"], sub_obs["eps2_up"], linewidth=1.6, label="Observed")

    if df_b is not None and all(c in df_b.columns for c in ["eps2_med", "eps2_lo1", "eps2_hi1", "eps2_lo2", "eps2_hi2"]):
        _overlay_expected_bands(ax, df_b, x="mass_GeV", med="eps2_med", lo1="eps2_lo1", hi1="eps2_hi1", lo2="eps2_lo2", hi2="eps2_hi2")

    ax.set_yscale("log")
    ax.set_xlabel("m (GeV)")
    ax.set_ylabel(r"$\varepsilon^2_{\rm up}$")
    ax.set_title(f"{ds.label}: ε² upper limit")
    _grid_summary(ax)
    ax.legend(loc="best", frameon=True)
    _safe_tight_layout(fig)
    outE = os.path.join(outdir, f"summary_eps2_up_{ds.key}.png")
    fig.savefig(outE, dpi=int(globals().get("SAVEFIG_DPI", 300)))
    plt.close(fig)

    # --- Z / p0 diagnostic 
    # --- p0 diagnostic plot (observed only) ---
    if "p0_analytic" in sub_obs.columns:
        fig, ax = plt.subplots(figsize=(9.2, 4.8))

        xP = sub_obs["mass_GeV"].to_numpy(float)
        p0 = np.clip(sub_obs["p0_analytic"].to_numpy(float), 1e-300, 1.0)
        ax.semilogy(xP, p0, linewidth=1.4, label=r"local analytic $p_0$")

        # Optional LEE overlay: horizontal lines showing the *local* p0 threshold
        # needed to reach a given global significance.
        if bool(globals().get("LEE_ENABLE", False)):
            try:
                method = str(globals().get("LEE_METHOD", "sidak"))
                W = float(globals().get("LEE_INDEP_WIDTH_SIGMA", 2.355))
                Neff = _lee_trials_from_grid_summary(xP, ds, indep_width_sigma=W)
                for z in list(globals().get("LEE_SIGMA_LINES", [3, 5])):
                    z = int(z)
                    pglob = float(norm.sf(z))
                    ploc_thr = _p_local_from_global_summary(pglob, Neff=Neff, method=method)
                    ax.axhline(ploc_thr, linestyle="--", linewidth=1.0, alpha=0.8)
                    ax.text(xP[0], ploc_thr*1.05, fr"global {z}$\sigma$", fontsize=9)
                ax.text(
                    0.02, 0.05,
                    f"N_eff ≈ {Neff:.1f} ({method})",
                    transform=ax.transAxes, fontsize=9.5,
                    bbox=dict(boxstyle="round,pad=0.25", facecolor="white", alpha=0.85, edgecolor="0.4"),
                )
            except Exception:
                pass

        ax.axhline(0.05, linestyle=":", linewidth=1.0, alpha=0.8, label="0.05")
        ax.set_xlabel("m (GeV)")
        ax.set_ylabel(r"$p_0$")
        ax.set_title(f"{ds.label}: analytic $p_0$ diagnostic")
        _grid_summary(ax)
        ax.legend(loc="best", frameon=True)
        _safe_tight_layout(fig)
        outP = os.path.join(outdir, f"summary_p0_analytic_{ds.key}.png")
        fig.savefig(outP, dpi=int(globals().get("SAVEFIG_DPI", 300)))
        plt.close(fig)

# plot (observed only) ---
    if "Z_analytic" in sub_obs.columns:
        fig, ax = plt.subplots(figsize=(9.2, 4.8))

        xZ = sub_obs["mass_GeV"].to_numpy(float)
        Zloc = sub_obs["Z_analytic"].to_numpy(float)
        ax.plot(xZ, Zloc, linewidth=1.4, label=r"local $Z$ (analytic)")

        # Optional LEE overlay from p0_analytic if available
        if bool(globals().get("LEE_ENABLE", False)) and ("p0_analytic" in sub_obs.columns):
            try:
                method = str(globals().get("LEE_METHOD", "sidak"))
                W = float(globals().get("LEE_INDEP_WIDTH_SIGMA", 2.355))
                Neff = _lee_trials_from_grid_summary(xZ, ds, indep_width_sigma=W)
                p0 = np.clip(sub_obs["p0_analytic"].to_numpy(float), 1e-300, 1.0)
                pglob = _p_global_from_local_summary(p0, Neff=Neff, method=method)
                Zglob = _z_from_p_one_sided_summary(pglob)
                ax.plot(xZ, Zglob, linestyle="--", linewidth=1.4, label=r"global $Z$ (LEE approx)")
                ax.text(
                    0.02, 0.05,
                    f"N_eff ≈ {Neff:.1f} ({method})",
                    transform=ax.transAxes, fontsize=9.5,
                    bbox=dict(boxstyle="round,pad=0.25", facecolor="white", alpha=0.85, edgecolor="0.4"),
                )
            except Exception:
                pass

        ax.axhline(0.0, color="k", linewidth=0.8, alpha=0.5)
        ax.set_xlabel("m (GeV)")
        ax.set_ylabel(r"$Z$")
        ax.set_title(f"{ds.label}: analytic significance (local vs global)")
        _grid_summary(ax)
        ax.legend(loc="best", frameon=True)
        _safe_tight_layout(fig)
        outZ = os.path.join(outdir, f"summary_Z_local_global_{ds.key}.png")
        fig.savefig(outZ, dpi=int(globals().get("SAVEFIG_DPI", 300)))
        plt.close(fig)

    print(f"[summary] wrote dataset summary plots to {outdir} for {ds.key}")


def plot_combined_eps2_curve(
    df_comb: pd.DataFrame,
    *,
    outdir: str,
    bands_csv: Optional[str] = None,
    label: str = "combined",
):
    """Combined ε² curve (observed), with optional expected bands overlay."""
    if df_comb is None or df_comb.empty:
        return
    ensure_dir(outdir)

    sub_obs = df_comb[np.isfinite(df_comb["eps2_up"])].copy()
    if sub_obs.empty:
        return

    df_b = None
    if bands_csv is not None:
        df_b = _maybe_read_csv(bands_csv)

    fig, ax = plt.subplots(figsize=(9.2, 5.4))
    ax.plot(sub_obs["mass_GeV"], sub_obs["eps2_up"], linewidth=1.8, label="Observed")

    if df_b is not None and all(c in df_b.columns for c in ["eps2_med", "eps2_lo1", "eps2_hi1", "eps2_lo2", "eps2_hi2"]):
        _overlay_expected_bands(ax, df_b, x="mass_GeV", med="eps2_med", lo1="eps2_lo1", hi1="eps2_hi1", lo2="eps2_lo2", hi2="eps2_hi2")

    ax.set_yscale("log")
    ax.set_xlabel("m (GeV)")
    ax.set_ylabel(r"$\varepsilon^2_{\rm up}$")
    ax.set_title(f"{label}: ε² upper limit")
    _grid_summary(ax)
    ax.legend(loc="best", frameon=True)
    _safe_tight_layout(fig)
    out = os.path.join(outdir, f"summary_eps2_up_{label}.png")
    fig.savefig(out, dpi=int(globals().get("SAVEFIG_DPI", 300)))
    plt.close(fig)


    # --- Combined p0 / Z diagnostics (if available) ---
    if "p0_analytic" in sub_obs.columns:
        fig, ax = plt.subplots(figsize=(9.2, 4.8))
        xP = sub_obs["mass_GeV"].to_numpy(float)
        p0 = np.clip(sub_obs["p0_analytic"].to_numpy(float), 1e-300, 1.0)
        ax.semilogy(xP, p0, linewidth=1.4, label=r"local analytic $p_0$")
        ax.axhline(0.05, linestyle=":", linewidth=1.0, alpha=0.8, label="0.05")

        # Optional LEE overlay: use an effective sigma(m) from the participating datasets.
        if bool(globals().get("LEE_ENABLE", False)) and ("datasets" in sub_obs.columns):
            try:
                method = str(globals().get("LEE_METHOD", "sidak"))
                W = float(globals().get("LEE_INDEP_WIDTH_SIGMA", 2.355))

                # Infer participating dataset keys from the first non-empty entry.
                ds_str = str(sub_obs["datasets"].dropna().astype(str).iloc[0])
                keys = [k for k in ds_str.split("+") if k in DATASETS]
                if len(keys) > 0:
                    # Effective sigma choice for combo: default to min (best resolution) across datasets.
                    sig_eff = []
                    for m in xP:
                        sigs = []
                        for k in keys:
                            try:
                                sigs.append(float(DATASETS[k].sigma(float(m))))
                            except Exception:
                                pass
                        sigs = [s for s in sigs if np.isfinite(s) and s > 0]
                        sig_eff.append(min(sigs) if len(sigs) else np.nan)
                    sig_eff = np.asarray(sig_eff, float)
                    # Neff ≈ Σ Δm / (W·σ_eff(m))
                    Neff = 0.0
                    xs = np.asarray(xP, float)
                    order = np.argsort(xs)
                    xs = xs[order]
                    se = sig_eff[order]
                    dm = np.diff(xs)
                    mids = 0.5 * (xs[:-1] + xs[1:])
                    se_mid = 0.5 * (se[:-1] + se[1:])
                    for dmi, si in zip(dm, se_mid):
                        if np.isfinite(si) and si > 0:
                            Neff += float(dmi) / (float(W) * float(si))
                    Neff = float(max(1.0, min(Neff, float(xs.size))))

                    for z in list(globals().get("LEE_SIGMA_LINES", [3, 5])):
                        z = int(z)
                        pglob = float(norm.sf(z))
                        ploc_thr = _p_local_from_global_summary(pglob, Neff=Neff, method=method)
                        ax.axhline(ploc_thr, linestyle="--", linewidth=1.0, alpha=0.8)
                        ax.text(xs[0], ploc_thr*1.05, fr"global {z}$\sigma$", fontsize=9)

                    ax.text(
                        0.02, 0.05,
                        f"N_eff ≈ {Neff:.1f} ({method})",
                        transform=ax.transAxes, fontsize=9.5,
                        bbox=dict(boxstyle="round,pad=0.25", facecolor="white", alpha=0.85, edgecolor="0.4"),
                    )
            except Exception:
                pass

        ax.set_xlabel("m (GeV)")
        ax.set_ylabel(r"$p_0$")
        ax.set_title(f"{label}: analytic $p_0$ diagnostic")
        _grid_summary(ax)
        ax.legend(loc="best", frameon=True)
        _safe_tight_layout(fig)
        outP = os.path.join(outdir, f"summary_p0_analytic_{label}.png")
        fig.savefig(outP, dpi=int(globals().get("SAVEFIG_DPI", 300)))
        plt.close(fig)

    if "Z_analytic" in sub_obs.columns:
        fig, ax = plt.subplots(figsize=(9.2, 4.8))
        xZ = sub_obs["mass_GeV"].to_numpy(float)
        Zloc = sub_obs["Z_analytic"].to_numpy(float)
        ax.plot(xZ, Zloc, linewidth=1.4, label=r"local $Z$ (analytic)")
        ax.axhline(0.0, color="k", linewidth=0.8, alpha=0.5)
        ax.set_xlabel("m (GeV)")
        ax.set_ylabel(r"$Z$")
        ax.set_title(f"{label}: analytic significance diagnostic")
        _grid_summary(ax)
        ax.legend(loc="best", frameon=True)
        _safe_tight_layout(fig)
        outZ = os.path.join(outdir, f"summary_Z_analytic_{label}.png")
        fig.savefig(outZ, dpi=int(globals().get("SAVEFIG_DPI", 300)))
        plt.close(fig)


    print(f"[summary] wrote combined eps2 plot to {out}")


def plot_observed_overlay(
    df_single: pd.DataFrame,
    *,
    outdir: str,
    ycol: str,
    ylabel: str,
    title: str,
    outname: str,
):
    """Overlay observed curves from all datasets (no expected bands)."""
    ensure_dir(outdir)
    fig, ax = plt.subplots(figsize=(9.6, 5.6))
    for ds_key, sub in df_single.groupby("dataset"):
        sub = sub[np.isfinite(sub[ycol])]
        if sub.empty:
            continue
        ax.plot(sub["mass_GeV"], sub[ycol], linewidth=1.4, label=str(ds_key))
    ax.set_yscale("log")
    ax.set_xlabel("m (GeV)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    _grid_summary(ax)
    ax.legend(loc="best", frameon=True, fontsize=9)
    _safe_tight_layout(fig)
    out = os.path.join(outdir, outname)
    fig.savefig(out, dpi=int(globals().get("SAVEFIG_DPI", 300)))
    plt.close(fig)
    print("[summary] wrote", out)


print("Summary plotting utilities ready (v15).")


# --- GP hyper-parameter QA (carried forward from v13/v13.5) ---
def plot_gp_hyperparameters(df_single: pd.DataFrame,
                           outdir: str = "summary_plots/gp_hyperparameters"):
    """Write GP hyper-parameter QA plots from a scan results dataframe.

    Expected columns (v15):
      - mass_GeV
      - ls_lo, ls_hi, ls_opt
      - sigma_x, ls_*_over_sigma_x
      - const_opt (optional), lml (optional)

    The most important sanity check for the v13 length-scale fix is that
    `ls_hi_over_sigma_x` is (approximately) constant across mass when using
    `KERNEL_LS_POLICY='resolution_scaled_local'`.
    """
    if df_single is None or df_single.empty:
        print("[gp] empty dataframe; nothing to plot")
        return

    ensure_dir(outdir)

    # Per-dataset plots
    for ds in sorted(df_single["dataset"].unique()):
        sub = df_single[df_single["dataset"] == ds].copy()
        sub = sub.sort_values("mass_GeV").reset_index(drop=True)
        if len(sub) == 0:
            continue

        x = sub["mass_GeV"].to_numpy(float)

        # ---- (1) Length-scale ratios (dimensionless) ----
        if "ls_opt_over_sigma_x" in sub.columns and np.isfinite(sub["ls_opt_over_sigma_x"].to_numpy(float)).any():
            fig, ax = plt.subplots(figsize=(9.2, 5.2))
            ax.plot(x, sub["ls_opt_over_sigma_x"], label=r"$\ell_{opt}/\sigma_x$")
            if "ls_hi_over_sigma_x" in sub.columns and np.isfinite(sub["ls_hi_over_sigma_x"].to_numpy(float)).any():
                ax.plot(x, sub["ls_hi_over_sigma_x"], linestyle="--", label=r"$\ell_{hi}/\sigma_x$")
            if "ls_lo_over_sigma_x" in sub.columns and np.isfinite(sub["ls_lo_over_sigma_x"].to_numpy(float)).any():
                ax.plot(x, sub["ls_lo_over_sigma_x"], linestyle="--", label=r"$\ell_{lo}/\sigma_x$")
            ax.set_xlabel("m (GeV)")
            ax.set_ylabel(r"Length-scale ratio (dimensionless)")
            ax.set_title(f"{ds}: GP length-scale vs local resolution")
            ax.grid(True, alpha=0.3)
            ax.legend(loc="best", frameon=True)
            _safe_tight_layout(fig)
            fig.savefig(os.path.join(outdir, f"gp_ls_ratio_{ds}.png"), dpi=int(globals().get("SAVEFIG_DPI", 300)))
            plt.close(fig)

        # ---- (1b) GeV-equivalent length-scale ratios (helps interpret PRE_LOG=True runs) ----
        if "ls_opt_over_sigma" in sub.columns and np.isfinite(sub["ls_opt_over_sigma"].to_numpy(float)).any():
            fig, ax = plt.subplots(figsize=(9.2, 5.2))
            ax.plot(x, sub["ls_opt_over_sigma"], label=r"$\Delta m(\ell_{opt})/\sigma(m)$")
            if "ls_hi_over_sigma" in sub.columns and np.isfinite(sub["ls_hi_over_sigma"].to_numpy(float)).any():
                ax.plot(x, sub["ls_hi_over_sigma"], linestyle="--", label=r"$\Delta m(\ell_{hi})/\sigma(m)$")
            ax.set_xlabel("m (GeV)")
            ax.set_ylabel(r"GeV-equivalent scale / $\sigma(m)$")
            ax.set_title(f"{ds}: GP length-scale in GeV units")
            ax.grid(True, alpha=0.3)
            ax.legend(loc="best", frameon=True)
            _safe_tight_layout(fig)
            fig.savefig(os.path.join(outdir, f"gp_ls_ratio_mass_{ds}.png"), dpi=int(globals().get("SAVEFIG_DPI", 300)))
            plt.close(fig)

        # ---- (2) Absolute length-scales in x-units ----
        if "ls_opt" in sub.columns and np.isfinite(sub["ls_opt"].to_numpy(float)).any():
            fig, ax = plt.subplots(figsize=(9.2, 5.2))
            ax.plot(x, sub["ls_opt"], label=r"$\ell_{opt}$")
            if "ls_hi" in sub.columns:
                ax.plot(x, sub["ls_hi"], linestyle="--", label=r"$\ell_{hi}$")
            if "ls_lo" in sub.columns:
                ax.plot(x, sub["ls_lo"], linestyle="--", label=r"$\ell_{lo}$")
            if "sigma_x" in sub.columns and np.isfinite(sub["sigma_x"].to_numpy(float)).any():
                ax.plot(x, sub["sigma_x"], linestyle=":", label=r"$\sigma_x$")
            ax.set_xlabel("m (GeV)")
            units = "ln(m)" if bool(PRE_LOG) else "m"
            ax.set_ylabel(f"Length-scale in x-units ({units})")
            ax.set_title(f"{ds}: GP length-scales (absolute)")
            ax.grid(True, alpha=0.3)
            # Use log-y when values span decades
            try:
                yvals = np.concatenate([sub[k].to_numpy(float) for k in ["ls_lo","ls_opt","ls_hi","sigma_x"] if k in sub.columns])
                yvals = yvals[np.isfinite(yvals) & (yvals > 0)]
                if yvals.size and (np.max(yvals) / np.min(yvals) > 50):
                    ax.set_yscale("log")
            except Exception:
                pass
            ax.legend(loc="best", frameon=True)
            _safe_tight_layout(fig)
            fig.savefig(os.path.join(outdir, f"gp_ls_abs_{ds}.png"), dpi=int(globals().get("SAVEFIG_DPI", 300)))
            plt.close(fig)

        # ---- (3) Kernel amplitude + LML (optional) ----
        if "const_opt" in sub.columns and np.isfinite(sub["const_opt"].to_numpy(float)).any():
            fig, ax = plt.subplots(figsize=(9.2, 5.0))
            ax.plot(x, sub["const_opt"], label="const_opt")
            ax.set_xlabel("m (GeV)")
            ax.set_ylabel("ConstantKernel amplitude")
            ax.set_title(f"{ds}: GP ConstantKernel amplitude (opt)")
            ax.grid(True, alpha=0.3)
            # amplitude can span orders
            try:
                y = sub["const_opt"].to_numpy(float)
                y = y[np.isfinite(y) & (y > 0)]
                if y.size and (np.max(y) / np.min(y) > 50):
                    ax.set_yscale("log")
            except Exception:
                pass
            ax.legend(loc="best", frameon=True)
            _safe_tight_layout(fig)
            fig.savefig(os.path.join(outdir, f"gp_const_{ds}.png"), dpi=int(globals().get("SAVEFIG_DPI", 300)))
            plt.close(fig)

        if "lml" in sub.columns and np.isfinite(sub["lml"].to_numpy(float)).any():
            fig, ax = plt.subplots(figsize=(9.2, 5.0))
            ax.plot(x, sub["lml"], label="log marginal likelihood")
            ax.set_xlabel("m (GeV)")
            ax.set_ylabel("log marginal likelihood")
            ax.set_title(f"{ds}: GP log marginal likelihood")
            ax.grid(True, alpha=0.3)
            ax.legend(loc="best", frameon=True)
            _safe_tight_layout(fig)
            fig.savefig(os.path.join(outdir, f"gp_lml_{ds}.png"), dpi=int(globals().get("SAVEFIG_DPI", 300)))
            plt.close(fig)

    print("Wrote GP hyper-parameter plots to", outdir)


print("Summary plotting ready (v13).")


## Optional: expected UL bands and strong/weak/two-sided p-values

Applied to **one chosen dataset** at a time (`RUN_LIMIT_BANDS_ON`), not the combined fit.

Definitions used:
- `UL_toy` from background-only pseudo-data in the blind window.
- **p-strong** = P(UL_toy ≤ UL_obs)
- **p-weak**   = P(UL_toy ≥ UL_obs)
- **p-two**    = 2·min(p-strong, p-weak)

Results are written to a separate CSV.


In [ ]:
# =============================================================================
# Cell 14 — Expected UL bands (single + combined) (v12)
# =============================================================================

from threadpoolctl import threadpool_limits

# Optional joblib parallelization (safe in notebooks)
try:
    from joblib import Parallel, delayed
    _HAVE_JOBLIB = True
except Exception:
    Parallel = None
    delayed = None
    _HAVE_JOBLIB = False

def cls_limit_for_template(
    n_obs: np.ndarray,
    b_mean: np.ndarray,
    b_cov: Optional[np.ndarray],
    template: np.ndarray,
    *,
    ds: Optional["DatasetConfig"] = None,
    mass: Optional[float] = None,
    integral_density: Optional[float] = None,
    alpha: float = CLS_ALPHA,
    mode: str = "asymptotic",
    use_eps2: bool = True,
    num_toys: int = CLS_NUM_TOYS,
    seed: int = 1,
    A_hi0: Optional[float] = None,
) -> Tuple[float, float]:
    """Compute a CLs upper limit for a fixed template in either A or ε² units.

    Parameters
    ----------
    n_obs : array
        Observed counts in the blind window.
    b_mean : array
        Background mean prediction in the blind window.
    b_cov : array or None
        Background covariance in the blind window (may be None for diagonal/ignored).
    template : array
        Per-bin Gaussian template integrals (NOT renormalized inside the blind region).
    ds, mass, integral_density : optional
        Required only when `use_eps2=True`, to convert the amplitude limit A_up into ε².
    alpha : float
        CLs threshold (e.g. 0.05).
    mode : {'asymptotic','toys'}
        How CLs is evaluated at each A.
    use_eps2 : bool
        If True, return (ε²_up, A_up). If False, return (A_up, A_up).
    """
    template = np.asarray(template, float)
    rng = np.random.default_rng(int(seed))

    def cls_at(A):
        if mode == "asymptotic":
            return float(cls_amplitude_asymptotic(float(A), n_obs, b_mean, b_cov, template)[0])
        return float(cls_amplitude_toys(float(A), n_obs, b_mean, b_cov, template, rng, max(1, int(num_toys)))[0])

    b_sum = float(np.sum(b_mean))
    A_lo = 0.0
    if A_hi0 is None:
        A_hi = max(1.0, 3.0 * math.sqrt(max(b_sum, 1.0)))
    else:
        A_hi = float(A_hi0)

    cls_hi = cls_at(A_hi)
    it = 0
    while cls_hi > alpha and A_hi < 1e7 and it < 40:
        A_hi *= 2.0
        cls_hi = cls_at(A_hi)
        it += 1

    # Bisection
    for _ in range(60):
        Amid = 0.5 * (A_lo + A_hi)
        cls_mid = cls_at(Amid)
        if abs(cls_mid - alpha) < 1e-3:
            A_lo = A_hi = Amid
            break
        if cls_mid > alpha:
            A_lo = Amid
        else:
            A_hi = Amid
        if abs(A_hi - A_lo) <= max(1e-3, 1e-3 * (1.0 + A_hi)):
            break

    A_up = float(0.5 * (A_lo + A_hi))

    if not bool(use_eps2):
        return A_up, A_up

    if ds is None or mass is None or integral_density is None:
        raise ValueError("cls_limit_for_template(use_eps2=True) requires ds, mass, and integral_density.")

    eps2_up = float(epsilon2_from_A(ds, float(mass), float(A_up), float(integral_density)))
    return eps2_up, A_up

def expected_ul_bands_for_dataset(
    ds: DatasetConfig,
    masses: List[float],
    *,
    cls_mode: Optional[str] = None,
    alpha: float = CLS_ALPHA,
    n_toys: int = UL_BANDS_TOYS,
    seed: int = UL_BANDS_SEED,
    restarts: int = UL_BANDS_N_RESTARTS,
    use_eps2: bool = True,
    n_workers: int = UL_BANDS_N_WORKERS,
    backend: str = UL_BANDS_PARALLEL_BACKEND,
    threads_per_worker: int = UL_BANDS_THREADS_PER_WORKER,
    refit_gp_on_toy: Optional[bool] = None,
    refit_restarts: Optional[int] = None,
    refit_optimize: Optional[bool] = None,
    train_exclude_nsigma: Optional[float] = None,
) -> pd.DataFrame:
    """Expected upper-limit (UL) bands vs mass for a single dataset.

    Output columns (v15.8)
    ----------------------
    For each mass hypothesis we write *both* amplitude and physics-parameter bands:
      - A bands:   A_lo2, A_lo1, A_med, A_hi1, A_hi2, A_mean
      - ε² bands:  eps2_lo2, eps2_lo1, eps2_med, eps2_hi1, eps2_hi2, eps2_mean
      - observed (if unblinded): A_obs, eps2_obs
      - auxiliary: p_strong/p_weak/p_two comparing observed UL to toy UL distribution

    Toy modes
    ---------
    (A) Conditional-GP toys (fast, default):
        - Draw background truth in the blind window from the GP posterior MVN (μ, Σ)
          computed from the *real-data* sideband fit.
        - Poisson-sample observed blind-window counts.
        - Compute ULs using the fixed (μ, Σ).

    (B) Full procedural toys with GP refit (slow; optional):
        - Generate a full-range pseudo-dataset with Poisson fluctuations.
        - Refit the GP on the toy sidebands (excluding ±train_exclude_nsigma·σ around the mass).
        - Predict (μ, Σ) in the blind window from the toy-fit GP.
        - Compute ULs using those toy-specific predictions.

    Notes
    -----
    - ULs are computed by solving for the signal amplitude A such that CLs(A)=alpha
      (one-sided, A>=0). The ε² limit is then obtained via the dataset's conversion
      ε²(A; m).
    - This function intentionally returns publication-friendly column names (`A_med`,
      `eps2_med`, etc.) so downstream plotting utilities do not depend on internal naming.
    """
    masses = [float(m) for m in masses]
    child_ss = np.random.SeedSequence(int(seed)).spawn(len(masses))

    if cls_mode is None:
        cls_mode = str(globals().get("CLS_MODE", "asymptotic"))
    cls_mode = str(cls_mode)

    if refit_gp_on_toy is None:
        refit_gp_on_toy = bool(globals().get("UL_BANDS_REFIT_GP_ON_TOY", False))
    refit_gp_on_toy = bool(refit_gp_on_toy)

    if refit_restarts is None:
        refit_restarts = int(globals().get("UL_BANDS_REFIT_GP_RESTARTS", 0))
    refit_restarts = int(refit_restarts)

    if refit_optimize is None:
        refit_optimize = bool(globals().get("UL_BANDS_REFIT_GP_OPTIMIZE", True))
    refit_optimize = bool(refit_optimize)

    if train_exclude_nsigma is None:
        train_exclude_nsigma = globals().get(
            "UL_BANDS_TRAIN_EXCLUDE_NSIGMA",
            globals().get("GP_TRAIN_EXCLUDE_NSIGMA", BLIND_NSIGMA),
        )
    train_exclude_nsigma = float(train_exclude_nsigma)

    compute_eps2 = bool(use_eps2)

    # Respect blinding policy
    compute_obs = (str(DATA_VISIBILITY.get(ds.key, "observed")).lower().strip() == "observed")

    def _finite(a: np.ndarray) -> np.ndarray:
        a = np.asarray(a, float)
        return a[np.isfinite(a)]

    def _quantiles(a: np.ndarray) -> Tuple[float, float, float, float, float]:
        a = _finite(a)
        if a.size == 0:
            return (np.nan, np.nan, np.nan, np.nan, np.nan)
        return (
            float(np.quantile(a, 0.025)),
            float(np.quantile(a, 0.16)),
            float(np.quantile(a, 0.50)),
            float(np.quantile(a, 0.84)),
            float(np.quantile(a, 0.975)),
        )

    def _mean(a: np.ndarray) -> float:
        a = _finite(a)
        return float(np.mean(a)) if a.size else np.nan

    def _one_mass(i: int, m: float) -> dict:
        ss = child_ss[i]
        rng = np.random.default_rng(ss)

        with threadpool_limits(limits=int(threads_per_worker)):
            pred = estimate_background_for_dataset(ds, mass=m, restarts=int(restarts), train_exclude_nsigma=float(train_exclude_nsigma))

        tmpl = build_template(pred.edges, m, pred.sigma_val, norm=None)

        obs = np.asarray(pred.obs, int)
        mu = np.asarray(pred.mu, float)
        cov = np.asarray(pred.cov, float)

        # --- Analytic p0/Z diagnostic (exact profiled LRT; useful QA for background mismodelling) ---
        if compute_obs:
            try:
                p0_a, Z_a, _q0_a, _info_a = p0_profiled_gaussian_LRT(obs, mu, cov, tmpl)
            except Exception:
                p0_a, Z_a = (np.nan, np.nan)
        else:
            p0_a, Z_a = (np.nan, np.nan)

        # Observed UL at this mass (optional; depends on blinding policy)
        if compute_obs:
            if compute_eps2:
                eps2_obs, A_obs = cls_limit_for_template(
                    obs, mu, cov, tmpl,
                    alpha=float(alpha),
                    mode=str(cls_mode),
                    use_eps2=True,
                    ds=ds,
                    mass=float(m),
                    integral_density=float(pred.integral_density),
                    seed=int(rng.integers(1, 2**31 - 1)),
                )
            else:
                A_obs, _ = cls_limit_for_template(
                    obs, mu, cov, tmpl,
                    alpha=float(alpha),
                    mode=str(cls_mode),
                    use_eps2=False,
                    seed=int(rng.integers(1, 2**31 - 1)),
                )
                eps2_obs = np.nan
        else:
            eps2_obs, A_obs = (np.nan, np.nan)

        toy_A: List[float] = []
        toy_eps2: List[float] = []

        if not refit_gp_on_toy:
            # ---------- Mode (A): conditional MVN toys in blind window ----------
            lam_draws = draw_bkg_mvn_nonneg(
                mu, cov, int(n_toys), rng,
                method=MVN_TRUNC_METHOD,
                max_tries=MVN_TRUNC_MAX_TRIES,
            )
            n_draws = rng.poisson(lam_draws).astype(int)

            for n in n_draws:
                if compute_eps2:
                    eps2_t, A_t = cls_limit_for_template(
                        n, mu, cov, tmpl,
                        alpha=float(alpha),
                        mode=str(cls_mode),
                        use_eps2=True,
                        ds=ds,
                        mass=float(m),
                        integral_density=float(pred.integral_density),
                        seed=int(rng.integers(1, 2**31 - 1)),
                    )
                    toy_eps2.append(float(eps2_t))
                    toy_A.append(float(A_t))
                else:
                    A_t, _ = cls_limit_for_template(
                        n, mu, cov, tmpl,
                        alpha=float(alpha),
                        mode=str(cls_mode),
                        use_eps2=False,
                        seed=int(rng.integers(1, 2**31 - 1)),
                    )
                    toy_A.append(float(A_t))

        else:
            # ---------- Mode (B): full procedural toys with GP refit ----------
            x_full = np.asarray(pred.x_full, float).reshape(-1)
            mu_full = np.asarray(pred.mu_full, float).reshape(-1)

            blind = tuple(pred.blind)
            msk_blind = (x_full >= blind[0]) & (x_full <= blind[1])
            if int(np.sum(msk_blind)) <= 0:
                raise RuntimeError(f"[bands][{ds.key}] m={m:.6g}: blind window has no bins")

            x_win = x_full[msk_blind]

            train_nsig = float(train_exclude_nsigma)
            blind_train = (float(m) - train_nsig * float(pred.sigma_val), float(m) + train_nsig * float(pred.sigma_val))
            msk_train = (x_full < blind_train[0]) | (x_full > blind_train[1])

            ker = make_kernel_for_dataset(ds, mass=m, optimized_by_year=_OPTIMIZED_LS_BY_YEAR)
            # Use the data-fit best LS as an initial value (helps stability; esp. if refit_optimize=False)
            try:
                if hasattr(ker, "k2") and hasattr(ker.k2, "length_scale") and np.isfinite(getattr(pred, "ls_opt", np.nan)):
                    ker.k2.length_scale = float(pred.ls_opt)
            except Exception:
                pass

            for _ in range(int(n_toys)):
                y_full_toy = rng.poisson(np.clip(mu_full, 0.0, None)).astype(int)
                obs_toy = y_full_toy[msk_blind].astype(int)

                mu_fit = mu
                cov_fit = cov
                try:
                    X_train = x_full[msk_train]
                    y_train = y_full_toy[msk_train].astype(float)

                    gpr = _fit_gpr(
                        X_train, y_train,
                        restarts=int(refit_restarts),
                        kernel=ker,
                        optimize=bool(refit_optimize),
                    )
                    mu_fit, cov_fit = _predict_counts_from_log_gpr(gpr, x_win)
                except Exception:
                    pass

                if compute_eps2:
                    eps2_t, A_t = cls_limit_for_template(
                        obs_toy, np.asarray(mu_fit, float), np.asarray(cov_fit, float), tmpl,
                        alpha=float(alpha),
                        mode=str(cls_mode),
                        use_eps2=True,
                        ds=ds,
                        mass=float(m),
                        integral_density=float(pred.integral_density),
                        seed=int(rng.integers(1, 2**31 - 1)),
                    )
                    toy_eps2.append(float(eps2_t))
                    toy_A.append(float(A_t))
                else:
                    A_t, _ = cls_limit_for_template(
                        obs_toy, np.asarray(mu_fit, float), np.asarray(cov_fit, float), tmpl,
                        alpha=float(alpha),
                        mode=str(cls_mode),
                        use_eps2=False,
                        seed=int(rng.integers(1, 2**31 - 1)),
                    )
                    toy_A.append(float(A_t))

        toy_A = np.asarray(toy_A, float)
        toy_eps2 = np.asarray(toy_eps2, float)

        qA = _quantiles(toy_A)
        mA = _mean(toy_A)

        if compute_eps2:
            qE = _quantiles(toy_eps2)
            mE = _mean(toy_eps2)
        else:
            qE = (np.nan, np.nan, np.nan, np.nan, np.nan)
            mE = np.nan

        # p-values comparing observed UL to the toy UL distribution (eps2 only; optional)
        if compute_obs and compute_eps2 and np.isfinite(eps2_obs) and _finite(toy_eps2).size:
            te = _finite(toy_eps2)
            p_strong = float(np.mean(te <= float(eps2_obs)))
            p_weak = float(np.mean(te >= float(eps2_obs)))
            p_two = float(2.0 * min(p_strong, p_weak))
        else:
            p_strong = np.nan
            p_weak = np.nan
            p_two = np.nan

        return dict(
            dataset=ds.key,
            mass_GeV=float(m),

            # Publication-facing column names
            eps2_obs=float(eps2_obs) if np.isfinite(eps2_obs) else np.nan,
            A_obs=float(A_obs) if np.isfinite(A_obs) else np.nan,

            eps2_lo2=float(qE[0]), eps2_lo1=float(qE[1]), eps2_med=float(qE[2]), eps2_hi1=float(qE[3]), eps2_hi2=float(qE[4]), eps2_mean=float(mE),
            A_lo2=float(qA[0]),   A_lo1=float(qA[1]),   A_med=float(qA[2]),   A_hi1=float(qA[3]),   A_hi2=float(qA[4]),   A_mean=float(mA),

            # Backward-compatible aliases (kept for older plotting helpers)
            ul_eps2_obs=float(eps2_obs) if np.isfinite(eps2_obs) else np.nan,
            ul_A_obs=float(A_obs) if np.isfinite(A_obs) else np.nan,
            toy_eps2_uls_q02=float(qE[0]), toy_eps2_uls_q16=float(qE[1]), toy_eps2_uls_q50=float(qE[2]), toy_eps2_uls_q84=float(qE[3]), toy_eps2_uls_q97=float(qE[4]), toy_eps2_uls_mean=float(mE),
            toy_A_uls_q02=float(qA[0]),    toy_A_uls_q16=float(qA[1]),    toy_A_uls_q50=float(qA[2]),    toy_A_uls_q84=float(qA[3]),    toy_A_uls_q97=float(qA[4]),    toy_A_uls_mean=float(mA),

            p0_analytic=float(p0_a) if np.isfinite(p0_a) else np.nan,
            Z_analytic=float(Z_a) if np.isfinite(Z_a) else np.nan,

            p_strong=float(p_strong),
            p_weak=float(p_weak),
            p_two=float(p_two),

            # kernel bookkeeping from the real-data fit
            ls_opt=float(getattr(pred, "ls_opt", np.nan)),
            ls_lo=float(getattr(pred, "ls_lo", np.nan)),
            ls_hi=float(getattr(pred, "ls_hi", np.nan)),

            # provenance for bands
            bands_refit_gp_on_toy=bool(refit_gp_on_toy),
            bands_train_exclude_nsigma=float(train_exclude_nsigma),
            bands_refit_restarts=int(refit_restarts),
            bands_refit_optimize=bool(refit_optimize),
        )

    if int(n_workers) <= 1:
        rows = [_one_mass(i, m) for i, m in enumerate(masses)]
    else:
        rows = joblib.Parallel(n_jobs=int(n_workers), backend=str(backend))(
            joblib.delayed(_one_mass)(i, m) for i, m in enumerate(masses)
        )

    df = pd.DataFrame(rows).sort_values("mass_GeV").reset_index(drop=True)
    return df

def expected_ul_bands_for_combination(
    ds_keys: List[object],  # accepts List[str] or List[DatasetConfig]
    
    masses: List[float],
    *,
    cls_mode: Optional[str] = None,
    only_overlap: bool = False,
    alpha: float = CLS_ALPHA,
    n_toys: int = COMBINED_BANDS_N_TOYS,
    seed: int = COMBINED_BANDS_SEED,
    restarts: int = UL_BANDS_N_RESTARTS,
    use_eps2: bool = True,
    n_workers: int = UL_BANDS_N_WORKERS,
    backend: str = UL_BANDS_PARALLEL_BACKEND,
    threads_per_worker: int = UL_BANDS_THREADS_PER_WORKER,
    refit_gp_on_toy: Optional[bool] = None,
    refit_restarts: Optional[int] = None,
    refit_optimize: Optional[bool] = None,
    train_exclude_nsigma: Optional[float] = None,
) -> pd.DataFrame:
    """Expected ε² upper-limit (UL) bands for a dataset *combination* vs mass (v15.8).

    Returns the publication-facing band columns for ε²:
        eps2_lo2, eps2_lo1, eps2_med, eps2_hi1, eps2_hi2, eps2_mean
        eps2_obs  (if unblinded)

    Notes
    -----
    - The combined statistical model assumes dataset independence, so the covariance is block-diagonal.
    - `only_overlap=True` restricts to masses where >=2 datasets are simultaneously active.
    - Combination bands are defined in ε². If `use_eps2=False` is passed, we raise.
    """
    if not bool(use_eps2):
        raise ValueError("expected_ul_bands_for_combination computes ε² bands; set use_eps2=True")

    masses = [float(m) for m in masses]

    # Accept either dataset keys (str) or DatasetConfig objects
    ds_keys_raw = list(ds_keys)
    ds_keys = []
    for d in ds_keys_raw:
        if hasattr(d, "key"):
            ds_keys.append(str(getattr(d, "key")))
        else:
            ds_keys.append(str(d))

    # Full procedural mode toggles (defaults from config)
    if refit_gp_on_toy is None:
        refit_gp_on_toy = bool(globals().get("UL_BANDS_REFIT_GP_ON_TOY", False))
    refit_gp_on_toy = bool(refit_gp_on_toy)

    if refit_restarts is None:
        refit_restarts = int(globals().get("UL_BANDS_REFIT_GP_RESTARTS", 0))
    refit_restarts = int(refit_restarts)

    if refit_optimize is None:
        refit_optimize = bool(globals().get("UL_BANDS_REFIT_GP_OPTIMIZE", True))
    refit_optimize = bool(refit_optimize)

    if train_exclude_nsigma is None:
        train_exclude_nsigma = globals().get(
            "UL_BANDS_TRAIN_EXCLUDE_NSIGMA",
            globals().get("GP_TRAIN_EXCLUDE_NSIGMA", BLIND_NSIGMA),
        )
    train_exclude_nsigma = float(train_exclude_nsigma)

    # Which masses are valid for combination
    def active_set(m):
        return [d.key for d in active_datasets_for_mass(float(m), DATASETS)]

    if only_overlap:
        masses = [m for m in masses if len(set(active_set(m)).intersection(ds_keys)) >= 2]
    else:
        masses = [m for m in masses if len(set(active_set(m)).intersection(ds_keys)) >= 1]

    child_ss = np.random.SeedSequence(int(seed)).spawn(len(masses))
    if cls_mode is None:
        cls_mode = str(globals().get("CLS_MODE", "asymptotic"))
    cls_mode = str(cls_mode)

    # Respect blinding: only compute obs if all participating datasets are "observed"
    def can_compute_obs(keys_here):
        return all(str(DATA_VISIBILITY.get(k, "observed")).lower().strip() == "observed" for k in keys_here)

    def _finite(a: np.ndarray) -> np.ndarray:
        a = np.asarray(a, float)
        return a[np.isfinite(a)]

    def _quantiles(a: np.ndarray) -> Tuple[float, float, float, float, float]:
        a = _finite(a)
        if a.size == 0:
            return (np.nan, np.nan, np.nan, np.nan, np.nan)
        return (
            float(np.quantile(a, 0.025)),
            float(np.quantile(a, 0.16)),
            float(np.quantile(a, 0.50)),
            float(np.quantile(a, 0.84)),
            float(np.quantile(a, 0.975)),
        )

    def _mean(a: np.ndarray) -> float:
        a = _finite(a)
        return float(np.mean(a)) if a.size else np.nan

    def _empty_row(tag: str, m: float) -> dict:
        return dict(
            dataset_set=str(tag),
            mass_GeV=float(m),
            eps2_obs=np.nan,
            eps2_lo2=np.nan, eps2_lo1=np.nan, eps2_med=np.nan, eps2_hi1=np.nan, eps2_hi2=np.nan, eps2_mean=np.nan,
            p0_analytic=np.nan,
            Z_analytic=np.nan,
            ul_eps2_obs=np.nan,
            toy_eps2_uls_q02=np.nan, toy_eps2_uls_q16=np.nan, toy_eps2_uls_q50=np.nan, toy_eps2_uls_q84=np.nan, toy_eps2_uls_q97=np.nan, toy_eps2_uls_mean=np.nan,
            p_strong=np.nan, p_weak=np.nan, p_two=np.nan,
            meta=str([]),
            bands_refit_gp_on_toy=bool(refit_gp_on_toy),
            bands_train_exclude_nsigma=float(train_exclude_nsigma),
            bands_refit_restarts=int(refit_restarts),
            bands_refit_optimize=bool(refit_optimize),
        )

    def _one_mass(i: int, m: float) -> dict:
        ss = child_ss[i]
        rng = np.random.default_rng(ss)

        ds_here = [k for k in ds_keys if k in active_set(m)]
        if len(ds_here) == 0:
            return _empty_row("EMPTY", float(m))

        # Fit real-data predictions for each dataset at this mass (once)
        with threadpool_limits(limits=int(threads_per_worker)):
            preds = {k: estimate_background_for_dataset(DATASETS[k], mass=m, restarts=int(restarts), train_exclude_nsigma=float(train_exclude_nsigma)) for k in ds_here}

        meta = [
            dict(key=k, sigma=float(preds[k].sigma_val), dens=float(preds[k].integral_density))
            for k in ds_here
        ]
        ds_tag = "+".join(ds_here)

        compute_obs = can_compute_obs(ds_here)

        # Build combination components from the real-data GP predictions
        preds_list = [preds[k] for k in ds_here]
        ds_list = [DATASETS[k] for k in ds_here]
        obs_vec0, b_vec, cov_mat, s_unit = build_combined_components(float(m), ds_list, preds_list)

        # Observed combined UL (optional)
        if compute_obs:
            eps2_obs = combined_cls_limit_epsilon2_from_vectors(
                obs_vec0, b_vec, cov_mat, s_unit,
                alpha=float(alpha), mode=str(cls_mode), seed=int(rng.integers(1, 2**31 - 1)),
            )
        else:
            eps2_obs = np.nan


        # --- Analytic p0/Z diagnostic on the combined model (exact profiled LRT) ---
        if compute_obs:
            try:
                SCALE = float(EPS2_LRT_SCALE)
                tmpl_lrt = np.asarray(s_unit, float) / max(SCALE, 1e-300)
                p0_a, Z_a, _q0_a, _info_a = p0_profiled_gaussian_LRT(obs_vec0, b_vec, cov_mat, tmpl_lrt)
            except Exception:
                p0_a, Z_a = (np.nan, np.nan)
        else:
            p0_a, Z_a = (np.nan, np.nan)

        toy_eps2: List[float] = []

        if not refit_gp_on_toy:
            # ---------- Mode (A): conditional MVN toys in each dataset's blind window ----------
            lam_draws_list = [
                draw_bkg_mvn_nonneg(
                    preds[k].mu, preds[k].cov, int(n_toys), rng,
                    method=MVN_TRUNC_METHOD,
                    max_tries=MVN_TRUNC_MAX_TRIES,
                )
                for k in ds_here
            ]
            n_draws_list = [rng.poisson(lam).astype(int) for lam in lam_draws_list]

            if len(ds_here) == 1:
                # Single-dataset special case: just use the single-dataset CLs helper
                k = ds_here[0]
                tmpl = build_template(preds[k].edges, m, preds[k].sigma_val, norm=None)
                for t in range(int(n_toys)):
                    eps2_t, _A_t = cls_limit_for_template(
                        n_draws_list[0][t],
                        np.asarray(preds[k].mu, float),
                        np.asarray(preds[k].cov, float),
                        tmpl,
                        alpha=float(alpha),
                        mode=str(cls_mode),
                        use_eps2=True,
                        ds=DATASETS[k],
                        mass=float(m),
                        integral_density=float(preds[k].integral_density),
                        seed=int(rng.integers(1, 2**31 - 1)),
                    )
                    toy_eps2.append(float(eps2_t))
            else:
                # Concatenate toy observations and run the combined ε² CLs solver
                obs_vec_toys = np.stack(
                    [np.concatenate([n_draws_list[j][t] for j in range(len(ds_here))]) for t in range(int(n_toys))],
                    axis=0,
                )
                for t in range(int(n_toys)):
                    eps2_t = combined_cls_limit_epsilon2_from_vectors(
                        obs_vec_toys[t], b_vec, cov_mat, s_unit,
                        alpha=float(alpha), mode=str(cls_mode),
                        seed=int(rng.integers(1, 2**31 - 1)),
                    )
                    toy_eps2.append(float(eps2_t))

        else:
            # ---------- Mode (B): full procedural toys with GP refit ----------
            from types import SimpleNamespace

            # Precompute per-dataset full-range arrays and masks
            per = {}
            for k in ds_here:
                x_full = np.asarray(preds[k].x_full, float).reshape(-1)
                mu_full = np.asarray(preds[k].mu_full, float).reshape(-1)

                blind = tuple(preds[k].blind)
                msk_blind = (x_full >= blind[0]) & (x_full <= blind[1])
                if int(np.sum(msk_blind)) <= 0:
                    raise RuntimeError(f"[bands][combo] {ds_tag} m={m:.6g}: blind window has no bins for {k}")

                x_win = x_full[msk_blind]
                train_nsig = float(train_exclude_nsigma)
                blind_train = (float(m) - train_nsig * float(preds[k].sigma_val), float(m) + train_nsig * float(preds[k].sigma_val))
                msk_train = (x_full < blind_train[0]) | (x_full > blind_train[1])

                ker = make_kernel_for_dataset(DATASETS[k], mass=m, optimized_by_year=_OPTIMIZED_LS_BY_YEAR)
                try:
                    if hasattr(ker, "k2") and hasattr(ker.k2, "length_scale") and np.isfinite(getattr(preds[k], "ls_opt", np.nan)):
                        ker.k2.length_scale = float(preds[k].ls_opt)
                except Exception:
                    pass

                per[k] = dict(
                    x_full=x_full, mu_full=mu_full,
                    msk_blind=msk_blind, x_win=x_win,
                    msk_train=msk_train, ker=ker,
                )

            for _ in range(int(n_toys)):
                preds_toy_list = []
                obs_concat = []

                for k in ds_here:
                    x_full = per[k]["x_full"]
                    mu_full = per[k]["mu_full"]
                    y_full_toy = rng.poisson(np.clip(mu_full, 0.0, None)).astype(int)

                    obs_toy = y_full_toy[per[k]["msk_blind"]].astype(int)
                    obs_concat.append(obs_toy)

                    mu_fit = np.asarray(preds[k].mu, float)
                    cov_fit = np.asarray(preds[k].cov, float)
                    try:
                        X_train = x_full[per[k]["msk_train"]]
                        y_train = y_full_toy[per[k]["msk_train"]].astype(float)
                        gpr = _fit_gpr(
                            X_train, y_train,
                            restarts=int(refit_restarts),
                            kernel=per[k]["ker"],
                            optimize=bool(refit_optimize),
                        )
                        mu_fit, cov_fit = _predict_counts_from_log_gpr(gpr, per[k]["x_win"])
                    except Exception:
                        pass

                    # Minimal prediction object for build_combined_components (duck-typed)
                    preds_toy_list.append(SimpleNamespace(
                        obs=np.asarray(obs_toy, int),
                        mu=np.asarray(mu_fit, float),
                        cov=np.asarray(cov_fit, float),
                        edges=np.asarray(preds[k].edges, float),
                        sigma_val=float(preds[k].sigma_val),
                        integral_density=float(preds[k].integral_density),
                    ))

                obs_vec_toy, b_vec_toy, cov_mat_toy, s_unit_toy = build_combined_components(
                    float(m), ds_list, preds_toy_list
                )
                # b_vec_toy/cov_mat_toy/s_unit_toy are toy-specific here, but should be close to the
                # real-data versions; we pass the toy-specific ones for consistency.
                eps2_t = combined_cls_limit_epsilon2_from_vectors(
                    obs_vec_toy, b_vec_toy, cov_mat_toy, s_unit_toy,
                    alpha=float(alpha), mode=str(cls_mode), seed=int(rng.integers(1, 2**31 - 1)),
                )
                toy_eps2.append(float(eps2_t))

        toy_eps2 = np.asarray(toy_eps2, float)

        q02, q16, q50, q84, q97 = _quantiles(toy_eps2)
        mean = _mean(toy_eps2)

        if compute_obs and np.isfinite(eps2_obs) and _finite(toy_eps2).size:
            te = _finite(toy_eps2)
            p_strong = float(np.mean(te <= float(eps2_obs)))
            p_weak = float(np.mean(te >= float(eps2_obs)))
            p_two = float(2.0 * min(p_strong, p_weak))
        else:
            p_strong = np.nan
            p_weak = np.nan
            p_two = np.nan

        return dict(
            dataset_set=str(ds_tag),
            mass_GeV=float(m),

            # Publication-facing column names
            eps2_obs=float(eps2_obs) if np.isfinite(eps2_obs) else np.nan,
            eps2_lo2=float(q02), eps2_lo1=float(q16), eps2_med=float(q50), eps2_hi1=float(q84), eps2_hi2=float(q97), eps2_mean=float(mean),

            # Backward-compatible aliases
            ul_eps2_obs=float(eps2_obs) if np.isfinite(eps2_obs) else np.nan,
            toy_eps2_uls_q02=float(q02),
            toy_eps2_uls_q16=float(q16),
            toy_eps2_uls_q50=float(q50),
            toy_eps2_uls_q84=float(q84),
            toy_eps2_uls_q97=float(q97),
            toy_eps2_uls_mean=float(mean),

            p0_analytic=float(p0_a) if np.isfinite(p0_a) else np.nan,
            Z_analytic=float(Z_a) if np.isfinite(Z_a) else np.nan,

            p_strong=float(p_strong),
            p_weak=float(p_weak),
            p_two=float(p_two),

            meta=str(meta),

            # provenance
            bands_refit_gp_on_toy=bool(refit_gp_on_toy),
            bands_train_exclude_nsigma=float(train_exclude_nsigma),
            bands_refit_restarts=int(refit_restarts),
            bands_refit_optimize=bool(refit_optimize),
        )

    if int(n_workers) <= 1:
        rows = [_one_mass(i, m) for i, m in enumerate(masses)]
    else:
        rows = joblib.Parallel(n_jobs=int(n_workers), backend=str(backend))(
            joblib.delayed(_one_mass)(i, m) for i, m in enumerate(masses)
        )

    df = pd.DataFrame(rows).sort_values("mass_GeV").reset_index(drop=True)
    return df

def plot_ul_bands(df: pd.DataFrame, use_eps2: bool = True, title: str = "", outpath: Optional[str] = None):
    """Plot observed + expected UL bands (±1σ/±2σ) vs mass with robust log scaling (v15.8).

    This plotter accepts either the v15.8 "publication" column names:
      - eps2_obs, eps2_lo2/lo1/med/hi1/hi2
      - A_obs,   A_lo2/lo1/med/hi1/hi2
    or the older internal names:
      - ul_eps2_obs, toy_eps2_uls_q*   (and optionally toy_A_uls_q*).

    It will automatically fall back to linear y-scale if there are no finite positive values,
    preventing Matplotlib log-scale failures during `tight_layout()` or interactive drawing.
    """
    if df is None or len(df) == 0:
        print("[plot_ul_bands] empty dataframe")
        return None

    d = df.copy()
    if "mass_GeV" not in d.columns:
        raise ValueError("plot_ul_bands requires a 'mass_GeV' column")
    d = d.sort_values("mass_GeV")

    x = np.asarray(d["mass_GeV"], float)

    def _pick(*names, default=np.nan):
        for n in names:
            if n in d.columns:
                return np.asarray(pd.to_numeric(d[n], errors="coerce"), float)
        return np.full(len(d), float(default))

    if bool(use_eps2):
        y_obs = _pick("eps2_obs", "ul_eps2_obs")
        y_lo2 = _pick("eps2_lo2", "toy_eps2_uls_q02")
        y_lo1 = _pick("eps2_lo1", "toy_eps2_uls_q16")
        y_med = _pick("eps2_med", "toy_eps2_uls_q50")
        y_hi1 = _pick("eps2_hi1", "toy_eps2_uls_q84")
        y_hi2 = _pick("eps2_hi2", "toy_eps2_uls_q97")
        y_mean = _pick("eps2_mean", "toy_eps2_uls_mean")

        ylab = r"$\varepsilon^2_{95}$"
    else:
        y_obs = _pick("A_obs", "ul_A_obs")
        y_lo2 = _pick("A_lo2", "toy_A_uls_q02")
        y_lo1 = _pick("A_lo1", "toy_A_uls_q16")
        y_med = _pick("A_med", "toy_A_uls_q50")
        y_hi1 = _pick("A_hi1", "toy_A_uls_q84")
        y_hi2 = _pick("A_hi2", "toy_A_uls_q97")
        y_mean = _pick("A_mean", "toy_A_uls_mean")

        ylab = r"$A_{95}$ (events)"

    # Decide whether log scale is valid
    y_stack = np.concatenate([y_obs, y_lo2, y_lo1, y_med, y_hi1, y_hi2, y_mean])
    y_pos = y_stack[np.isfinite(y_stack) & (y_stack > 0)]
    log_ok = (y_pos.size > 0)

    # For log plots, mask non-positive values to avoid crashes
    def _mask_nonpos(a):
        a = np.asarray(a, float)
        if not log_ok:
            return a
        a2 = a.copy()
        a2[~np.isfinite(a2)] = np.nan
        a2[a2 <= 0] = np.nan
        return a2

    y_obs_p = _mask_nonpos(y_obs)
    y_lo2_p = _mask_nonpos(y_lo2)
    y_lo1_p = _mask_nonpos(y_lo1)
    y_med_p = _mask_nonpos(y_med)
    y_hi1_p = _mask_nonpos(y_hi1)
    y_hi2_p = _mask_nonpos(y_hi2)

    fig, ax = plt.subplots(figsize=(9.6, 5.6))

    # Expected bands (HEP convention: green=1σ, yellow=2σ)
    if np.isfinite(y_lo2_p).any() and np.isfinite(y_hi2_p).any():
        ax.fill_between(x, y_lo2_p, y_hi2_p, label=r"Expected $\pm 2\sigma$", alpha=0.7, color="gold")
    if np.isfinite(y_lo1_p).any() and np.isfinite(y_hi1_p).any():
        ax.fill_between(x, y_lo1_p, y_hi1_p, label=r"Expected $\pm 1\sigma$", alpha=0.7, color="limegreen")

    # Median + observed
    if np.isfinite(y_med_p).any():
        ax.plot(x, y_med_p, ls="--", lw=2.0, color="k", label="Expected (median)")
    if np.isfinite(y_obs_p).any():
        ax.plot(x, y_obs_p, ls="-", marker="o", lw=2.0, color="k", label="Observed")

    ax.set_xlabel(r"$m$ [GeV]")
    ax.set_ylabel(ylab)
    if title:
        ax.set_title(str(title), pad=float(globals().get("TITLE_PAD", 12.0)))

    if log_ok:
        ax.set_yscale("log")
        # A gentle lower bound to keep LogLocator happy even with masked segments
        y_min = float(np.nanmin(y_pos))
        if np.isfinite(y_min) and y_min > 0:
            ax.set_ylim(bottom=max(y_min * 0.5, y_min / 10.0))
    else:
        ax.set_yscale("linear")
        ax.text(
            0.02, 0.02,
            "No finite positive UL values available → linear scale",
            transform=ax.transAxes,
            fontsize=10,
            va="bottom",
            ha="left",
            alpha=0.8,
        )

    ax.legend(loc="best", frameon=True)

    # Robust layout
    try:
        fig.tight_layout()
    except Exception:
        try:
            fig.set_layout_engine("constrained")
        except Exception:
            pass

    if outpath:
        fig.savefig(outpath, dpi=int(globals().get("SAVEFIG_DPI", 300)), bbox_inches="tight")
        plt.close(fig)
    return fig, ax

def _lee_trials_from_grid(
    masses: np.ndarray,
    ds_keys: List[object],  # accepts List[str] or List[DatasetConfig]
    
    *,
    datasets: Dict[str, DatasetConfig],
    indep_width_sigma: float = 2.355,
    combo_sigma_method: str = "min",
) -> float:
    """Approximate an *effective* trials factor N_eff for the look-elsewhere effect.

    We treat neighboring mass hypotheses as correlated over a characteristic width ~ W·σ(m),
    and estimate the number of independent resolution elements across the scan:

        N_eff ≈ ∫ dm / (W · σ_eff(m))

    where σ_eff(m) is the relevant mass resolution for a single dataset or a chosen
    combination rule for multiple datasets.

    Notes
    -----
    - This is an *effective trials factor* approximation, not a full Gross–Vitells upcrossing
      calculation. It is meant for quick reviewer-facing diagnostics.
    - We cap N_eff by the number of tested mass points, since the scan grid is discrete.
    """
    m = np.asarray(masses, float)
    m = m[np.isfinite(m)]
    if m.size < 2:
        return 1.0
    m = np.sort(m)

    W = max(float(indep_width_sigma), 1e-6)

    def sigma_eff(mi: float) -> float:
        sigs = []
        for k in ds_keys:
            try:
                sigs.append(float(datasets[k].sigma(float(mi))))
            except Exception:
                continue
        sigs = [s for s in sigs if np.isfinite(s) and s > 0]
        if len(sigs) == 0:
            return float("nan")
        meth = str(combo_sigma_method).lower().strip()
        if meth == "min":
            return float(np.min(sigs))
        if meth == "mean":
            return float(np.mean(sigs))
        if meth == "harmonic":
            return float(len(sigs) / np.sum(1.0 / np.asarray(sigs)))
        return float(np.min(sigs))

    dm = np.diff(m)
    mids = 0.5 * (m[:-1] + m[1:])

    Neff = 0.0
    for dmi, mi in zip(dm, mids):
        sig = sigma_eff(float(mi))
        if not np.isfinite(sig) or sig <= 0:
            continue
        Neff += float(dmi) / (W * float(sig))

    # Effective trials factor must be >=1 and <= number of tested hypotheses
    Neff = max(float(Neff), 1.0)
    Neff = min(float(Neff), float(m.size))
    return float(Neff)

def _p_global_from_local(p_local: np.ndarray, *, Neff: float, method: str = "sidak") -> np.ndarray:
    """Convert local one-sided p-values to global p-values via a trials factor approximation."""
    p = np.asarray(p_local, float)
    p = np.clip(p, 0.0, 1.0)
    N = max(float(Neff), 1.0)
    meth = str(method).lower().strip()
    if meth == "bonferroni":
        return np.clip(N * p, 0.0, 1.0)
    # Sidak: p_global = 1 - (1 - p)^N (stable for tiny p via log1p)
    return np.clip(-np.expm1(N * np.log1p(-p)), 0.0, 1.0)


def _z_from_p_one_sided(p: np.ndarray) -> np.ndarray:
    """One-sided Gaussian Z corresponding to p (Z = Φ^{-1}(1-p))."""
    from scipy.stats import norm
    p = np.asarray(p, float)
    p = np.clip(p, 1e-300, 1.0)
    return norm.isf(p)


def _p_from_z_one_sided(z: float) -> float:
    from scipy.stats import norm
    return float(norm.sf(float(z)))


def _infer_ds_keys_from_df(df: pd.DataFrame) -> List[str]:
    """Infer dataset keys from a scan/bands dataframe."""
    if df is None or df.empty:
        return []
    if "dataset" in df.columns:
        vals = [str(v) for v in df["dataset"].dropna().unique().tolist()]
        if len(vals) == 1:
            return vals
    if "dataset_set" in df.columns:
        tag = str(df["dataset_set"].dropna().iloc[0])
        keys = [t.strip() for t in tag.split("+") if t.strip()]
        return keys
    return []


def plot_ul_pvalues(df: pd.DataFrame, title: str = "", outpath: Optional[str] = None):
    """Plot UL p-values vs mass (v15.8, robust to missing/non-finite values)."""
    if df is None or len(df) == 0:
        print("[plot_ul_pvalues] empty dataframe")
        return None

    d = df.copy()
    if "mass_GeV" not in d.columns:
        raise ValueError("plot_ul_pvalues requires a 'mass_GeV' column")
    d = d.sort_values("mass_GeV")
    x = np.asarray(d["mass_GeV"], float)

    def _pick(name, default=np.nan):
        if name in d.columns:
            return np.asarray(pd.to_numeric(d[name], errors="coerce"), float)
        return np.full(len(d), float(default))

    p_strong = _pick("p_strong", 1.0)
    p_weak   = _pick("p_weak", 1.0)
    p_two    = _pick("p_two", 1.0)

    # Replace non-finite by 1 (harmless on log scale after clipping)
    for arr in (p_strong, p_weak, p_two):
        arr[~np.isfinite(arr)] = 1.0

    p_strong = np.clip(p_strong, 1e-300, 1.0)
    p_weak   = np.clip(p_weak,   1e-300, 1.0)
    p_two    = np.clip(p_two,    1e-300, 1.0)

    fig, ax = plt.subplots(figsize=(9.6, 5.0))
    ax.plot(x, p_strong, marker="o", label="P(toy UL ≤ observed UL)")
    ax.plot(x, p_weak,   marker="o", label="P(toy UL ≥ observed UL)")
    ax.plot(x, p_two,    marker="o", label="Two-sided (2×min)")

    ax.set_xlabel(r"$m$ [GeV]")
    ax.set_ylabel("p-value")
    if title:
        ax.set_title(str(title), pad=float(globals().get("TITLE_PAD", 12.0)))
    ax.set_yscale("log")
    ax.set_ylim(1e-6, 1.05)
    ax.legend(loc="best", frameon=True)

    try:
        fig.tight_layout()
    except Exception:
        pass

    if outpath:
        fig.savefig(outpath, dpi=int(globals().get("SAVEFIG_DPI", 300)), bbox_inches="tight")
        plt.close(fig)
    return fig, ax

def plot_analytic_p0(
    df: pd.DataFrame,
    *,
    title: str = "",
    outpath: Optional[str] = None,
    apply_lee: Optional[bool] = None,
    lee_sigma_lines: Optional[List[int]] = None,
    Neff: Optional[float] = None,
):
    """Plot analytic bump-hunt p0 (from profiled LRT) vs mass.

    If `apply_lee` is True, overlay horizontal lines corresponding to *global* significance
    thresholds using a trials-factor approximation.
    """
    if df is None or df.empty or ("p0_analytic" not in df.columns):
        print("[bands] empty dataframe (or missing p0_analytic); nothing to plot")
        return

    if apply_lee is None:
        apply_lee = bool(globals().get("LEE_ENABLE", False))
    apply_lee = bool(apply_lee)

    method = str(globals().get("LEE_METHOD", "sidak"))
    indepW = float(globals().get("LEE_INDEP_WIDTH_SIGMA", 2.355))
    combo_method = str(globals().get("LEE_COMBO_SIGMA_METHOD", "min"))

    if lee_sigma_lines is None:
        lee_sigma_lines = list(globals().get("LEE_SIGMA_LINES", [3, 5]))
    lee_sigma_lines = [int(z) for z in lee_sigma_lines]

    x = df["mass_GeV"].to_numpy(float)
    order = np.argsort(x)
    x = x[order]
    p0 = np.clip(df["p0_analytic"].to_numpy(float)[order], 1e-300, 1.0)

    # Trials factor (optional)
    if apply_lee and (Neff is None):
        keys = _infer_ds_keys_from_df(df)
        if len(keys) > 0:
            Neff = _lee_trials_from_grid(x, keys, datasets=DATASETS, indep_width_sigma=indepW, combo_sigma_method=combo_method)
        else:
            Neff = 1.0
    if Neff is None:
        Neff = 1.0
    Neff = float(max(Neff, 1.0))

    fig, ax = plt.subplots(figsize=(9.6, 5.0))
    ax.semilogy(x, p0, label=r"local analytic $p_0$")
    ax.axhline(0.05, linestyle=":", label="0.05")

    if apply_lee and Neff > 1.0:
        for z in lee_sigma_lines:
            pglob = _p_from_z_one_sided(float(z))
            if str(method).lower().strip() == "bonferroni":
                ploc = min(1.0, pglob / Neff)
            else:
                ploc = 1.0 - (1.0 - pglob) ** (1.0 / Neff)
            ploc = float(np.clip(ploc, 1e-300, 1.0))
            ax.axhline(ploc, linestyle="--", linewidth=1.0, label=fr"{z}$\sigma$ global (LEE approx)")

        ax.text(
            0.02, 0.03,
            fr"$N_{{\rm eff}} \approx {Neff:.1f}$ ({method})",
            transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle="round,pad=0.25", facecolor="white", alpha=0.85, edgecolor="0.4"),
        )

    ax.set_xlabel("mass (GeV)")
    ax.set_ylabel(r"$p_0$")
    ax.set_title(title or r"Analytic $p_0$ vs mass")
    ax.grid(True, which="both", alpha=0.25)
    ax.legend(loc="best", frameon=True)
    fig.tight_layout()
    if outpath:
        fig.savefig(outpath, dpi=int(globals().get("SAVEFIG_DPI", 300)))
    plt.show()
    plt.close(fig)


def plot_Z_local_global(
    df: pd.DataFrame,
    *,
    title: str = "",
    outpath: Optional[str] = None,
    apply_lee: Optional[bool] = None,
    Neff: Optional[float] = None,
):
    """Plot local and (approximate) global Z vs mass derived from analytic p0."""
    if df is None or df.empty or ("p0_analytic" not in df.columns):
        print("[bands] empty dataframe (or missing p0_analytic); nothing to plot")
        return

    if apply_lee is None:
        apply_lee = bool(globals().get("LEE_ENABLE", False))
    apply_lee = bool(apply_lee)

    method = str(globals().get("LEE_METHOD", "sidak"))
    indepW = float(globals().get("LEE_INDEP_WIDTH_SIGMA", 2.355))
    combo_method = str(globals().get("LEE_COMBO_SIGMA_METHOD", "min"))

    x = df["mass_GeV"].to_numpy(float)
    order = np.argsort(x)
    x = x[order]
    p0 = np.clip(df["p0_analytic"].to_numpy(float)[order], 1e-300, 1.0)

    Zloc = _z_from_p_one_sided(p0)

    if apply_lee and (Neff is None):
        keys = _infer_ds_keys_from_df(df)
        if len(keys) > 0:
            Neff = _lee_trials_from_grid(x, keys, datasets=DATASETS, indep_width_sigma=indepW, combo_sigma_method=combo_method)
        else:
            Neff = 1.0
    if Neff is None:
        Neff = 1.0
    Neff = float(max(Neff, 1.0))

    if apply_lee and Neff > 1.0:
        pglob = _p_global_from_local(p0, Neff=Neff, method=method)
        Zglob = _z_from_p_one_sided(pglob)
    else:
        Zglob = None

    fig, ax = plt.subplots(figsize=(9.6, 5.0))
    ax.plot(x, Zloc, linewidth=1.5, label=r"local $Z$")
    if Zglob is not None:
        ax.plot(x, Zglob, linestyle="--", linewidth=1.5, label=r"global $Z$ (LEE approx)")
        ax.text(
            0.02, 0.03,
            fr"$N_{{\rm eff}} \approx {Neff:.1f}$ ({method})",
            transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle="round,pad=0.25", facecolor="white", alpha=0.85, edgecolor="0.4"),
        )

    ax.axhline(0.0, color="k", linewidth=0.8, alpha=0.5)
    ax.set_xlabel("mass (GeV)")
    ax.set_ylabel(r"$Z$")
    ax.set_title(title or "Analytic local/global significance vs mass")
    ax.grid(True, which="both", alpha=0.25)
    ax.legend(loc="best", frameon=True)
    fig.tight_layout()
    if outpath:
        fig.savefig(outpath, dpi=int(globals().get("SAVEFIG_DPI", 300)))
    plt.show()
    plt.close(fig)

print("Expected-band utilities ready (v15.8).")


## Systematics and robustness studies (v15.8)

This notebook is designed to produce *statistically correct* local test statistics and CL\(_s\) upper limits **given** a background model (the GP) and a signal model (the Gaussian template with dataset-specific conversions to \(\varepsilon^2\)). For a publication-quality result, you should explicitly evaluate how sensitive the conclusions are to modelling choices and approximations.

### Why we use a GP training exclusion window ("exclusion zone")
It is common in resonance searches to **exclude** the signal region (or a wider window) from the background fit and then interpolate from the sidebands. In traditional analyses this appears as a *sideband-only parametric fit*; here it is implemented as a *sideband-only GP fit*. This:
- reduces bias from potential signal contamination in the fitted background,
- makes the prediction in the blind window an interpolation (with larger uncertainty than in the sidebands),
- naturally increases the GP predictive variance inside the excluded region (especially for wider exclusions).

**Yes:** widening the exclusion window increases interpolation uncertainty.  
**But:** the analysis is already built to propagate that uncertainty via the GP posterior covariance \(\Sigma\) in the likelihood.

The *choice* of exclusion width (\(n_\sigma\)) is itself a modelling decision, so it should be treated as a **background-modelling systematic** (see suggested studies below).

### Recommended systematic studies to report (minimum set)

#### A) GP background-model systematics
1. **Training exclusion width**: vary `GP_TRAIN_EXCLUDE_NSIGMA` (and/or `UL_BANDS_TRAIN_EXCLUDE_NSIGMA`) across a reasonable range (e.g. 1.0, 1.64, 2.0, 3.0) and propagate to limits/p\(_0\).
2. **Kernel / length-scale policy**:
   - compare `KERNEL_POLICY="local"` vs `"global"` (if used),
   - vary `KERNEL_LS_RES_*` bounds/floor choices within motivated ranges,
   - check that results are stable against modest changes in restarts (`N_RESTARTS`).
3. **Hyperparameter optimization stability**:
   - monitor convergence warnings (LBFGS) and refit with larger `maxiter` or additional restarts as a robustness check,
   - consider a “frozen-hyperparameter” variant (optimize once on data, then hold fixed).
4. **Binning / rebinning sensitivity**: vary `NEIGHBORHOOD_REBIN` (and/or the histogram binning if possible) and check UL stability.
5. **Injection/extraction closure** (already supported): demonstrate negligible bias and correct pull widths/coverage across the mass range.

#### B) Statistical-procedure systematics
1. **Asymptotic vs toy CL\(_s\)**: compare `CLS_MODE="asymptotic"` to a toy-based mode at a subset of masses (especially where counts are low).
2. **Local-to-global conversion (LEE)**:
   - report the local p\(_0\) curve,
   - quote a global p-value using a clearly stated approximation (effective trials factor),
   - treat the trials-factor estimate as an approximation and (if possible) validate with toys.

#### C) Signal-model & conversion systematics
1. **Resolution model** \(\sigma(m)\): vary \(\sigma(m)\) within its calibration uncertainty and propagate to limits.
2. **Normalization systematics**: uncertainties on efficiency, luminosity, and the radiative fraction model (entering `A_from_epsilon2`) act like a multiplicative uncertainty on the signal yield per \(\varepsilon^2\). Treat these as a nuisance parameter, or conservatively propagate by rescaling the signal normalization.

#### D) Combination systematics (2015+2016, etc.)
The combined likelihood assumes:
- **statistical independence** of datasets (no shared events),
- **block-diagonal** background covariance (no cross-dataset GP correlations),
- a shared parameter of interest \(\varepsilon^2\), with dataset-specific signal normalizations.

Systematics to consider in combinations:
1. correlated normalization uncertainties across years (e.g., luminosity, common efficiencies),
2. year-specific uncertainties (trigger, detector conditions),
3. consistency of the resolution model and template across years.

A simple conservative cross-check is to apply correlated normalization shifts coherently to all datasets and take an envelope of the combined limit.



In [ ]:
# =============================================================================
# Systematics helpers (v15.8)
# =============================================================================
#
# These helpers are intentionally lightweight. They provide a framework to quantify
# the sensitivity of limits/p0 to key modelling knobs without imposing a full nuisance-parameter
# machinery inside the notebook.
#
# Typical usage:
#   - pick a small set of mass points (e.g. 5–10) spanning the analysis range
#   - run each systematic variant on that subset
#   - inspect ratios / envelopes
#
from contextlib import contextmanager

@contextmanager
def temp_config(**overrides):
    """Temporarily override notebook-global configuration values."""
    old = {}
    missing = object()
    for k, v in overrides.items():
        old[k] = globals().get(k, missing)
        globals()[k] = v
    try:
        yield
    finally:
        for k, v0 in old.items():
            if v0 is missing:
                try:
                    del globals()[k]
                except Exception:
                    pass
            else:
                globals()[k] = v0

def syst_exclusion_width_observed_limits(
    ds: DatasetConfig,
    masses: List[float],
    widths_nsigma: List[float],
    *,
    restarts: int = N_RESTARTS,
) -> pd.DataFrame:
    """Observed-only UL/p0 sensitivity to the GP training exclusion width.

    This reruns the *data* GP fit at each mass for each exclusion width and records:
      - eps2_up, A_up
      - p0_analytic, Z_analytic

    It does NOT run expected bands (to keep it fast). Use it as a robustness diagnostic.
    """
    rows = []
    for w in widths_nsigma:
        with temp_config(GP_TRAIN_EXCLUDE_NSIGMA=float(w)):
            for m in masses:
                try:
                    res, pred, _ = evaluate_single_dataset(ds, float(m), do_extraction=False)
                    rows.append({
                        "dataset": ds.key,
                        "mass_GeV": float(m),
                        "train_exclude_nsigma": float(w),
                        "A_up": float(res.A_up),
                        "eps2_up": float(res.eps2_up),
                        "p0_analytic": float(res.p0_analytic),
                        "Z_analytic": float(res.Z_analytic),
                        "ls_opt": float(getattr(pred, "ls_opt", np.nan)),
                    })
                except Exception as e:
                    rows.append({
                        "dataset": ds.key,
                        "mass_GeV": float(m),
                        "train_exclude_nsigma": float(w),
                        "A_up": np.nan,
                        "eps2_up": np.nan,
                        "p0_analytic": np.nan,
                        "Z_analytic": np.nan,
                        "ls_opt": np.nan,
                        "error": str(e),
                    })
    return pd.DataFrame(rows)

def apply_signal_norm_syst_to_eps2_limit(eps2_up: float, rel_unc: float, *, direction: str = "up") -> float:
    """Propagate a multiplicative signal-normalization uncertainty to an eps2 limit.

    If the expected signal yield per eps2 is uncertain by a factor (1 ± rel_unc), then
    eps2_up scales inversely with that factor.
    """
    e = float(eps2_up)
    du = float(abs(rel_unc))
    if (not np.isfinite(e)) or e <= 0 or (not np.isfinite(du)):
        return float("nan")
    if direction.lower().strip() in ("down", "-", "minus", "lower"):
        # smaller signal yield -> weaker (larger) eps2 limit
        scale = max(1.0 - du, 1e-6)
        return float(e / scale)
    # larger signal yield -> stronger (smaller) eps2 limit
    scale = 1.0 + du
    return float(e / scale)

print("Systematics helpers ready (v15.8).")

## Signal injection + extraction (closure tests)

This section implements the pseudo-experiment workflow in your note (Sec. 8):

For each dataset \(d\), mass hypothesis \(m_0\), and injected strength \(N_{\mathrm{inj}}\):

1. **Sideband-only GP fit** (already done in `estimate_background_for_dataset`) provides the blind-window predictive mean and covariance \((\bar b,\,\mathrm{Cov}_b)\).
2. For each toy, draw a *toy background mean* (truth model)
   \[
   b \sim \mathcal N(\bar b,\,\mathrm{Cov}_b)\quad \text{with truncation } b_i\ge 0,
   \]
   then draw **Poisson** counts in each blind bin: \(n^{(b)}_i\sim\mathrm{Pois}(b_i)\).
3. Construct the **unit-normalized Gaussian template** \(w_i(m_0,\sigma(m_0))\) by integrating a Gaussian over the blind-window bin edges.
4. Inject signal counts using **Gaussian-template injection only**:
   - `INJ_MODE="poisson"`: \(n^{(s)}_i \sim \mathrm{Pois}(N_{\mathrm{inj}}\,w_i)\) (total injected signal fluctuates; gives horizontal Poisson bands)
   - `INJ_MODE="multinomial"`: \(n^{(s)} \sim \mathrm{Multinomial}(N_{\mathrm{inj}}, w)\) (total fixed)
5. Fit each pseudo-dataset with the **profiled Gaussian-prior extraction** (additive model), **always enforcing \(A\ge 0\)** in toys.

We report toy-based vertical and horizontal uncertainties (central 68% intervals) and produce residual/linearity plots.


In [ ]:
# =============================================================================
# Cell 19 — Signal injection + extraction toy suite (v15)
# =============================================================================
#
# Purpose:
#   - Validate extraction linearity and bias vs injected signal yield
#   - Provide "realistic" pull plots: (Ahat − Ainj) / σ_A
#   - Study detection significance Zhat = Ahat/σ_A as a function of injection
#
# Notes on template normalization:
#   - In v14, TEMPLATE_NORM_MODE defaults to 'pdf' so Σw_win < 1 generally.
#   - In multinomial injection mode we first draw N_in_window ~ Binomial(Ainj, Σw_win)
#     and then distribute across bins according to w / Σw_win.
# =============================================================================


def _inject_counts_weighted(weights: np.ndarray, total: int, rng: np.random.Generator) -> np.ndarray:
    """Multinomial draw across bins (weights must sum to 1)."""
    total = int(total)
    if total <= 0:
        return np.zeros_like(weights, dtype=int)
    w = np.asarray(weights, float)
    wsum = float(np.sum(w))
    if wsum <= 0:
        return np.zeros_like(weights, dtype=int)
    w = w / wsum
    return rng.multinomial(total, w)


def _inject_counts_from_template(
    template: np.ndarray,
    strength: float,
    rng: np.random.Generator,
    mode: str = "multinomial",
) -> Tuple[np.ndarray, int, float]:
    """Inject a signal distributed like `template` (bin weights) at amplitude `strength`.

    Args:
      template: bin weights w_i (not necessarily normalized); in v14 'pdf' mode Σw is the
                fraction captured by the histogram / blind window.
      strength: injected *total-yield* parameter A_inj (float allowed).
      mode:
        - 'multinomial': fixed total yield A_inj (rounded to int) with binomial thinning if Σw<1
        - 'poisson'    : independent Poisson per bin with mean = A_inj * w_i

    Returns:
      sig_counts: injected signal counts per bin (int array)
      Nsig_win  : realized total injected signal counts in the provided bin set (sum(sig_counts))
      f_sumw    : Σw (useful diagnostic; in pdf-mode this is a fraction)
    """
    w = np.asarray(template, float)
    f = float(np.sum(w))
    if (not np.isfinite(f)) or (f <= 0.0) or (not np.isfinite(strength)) or (strength <= 0.0):
        return np.zeros_like(w, dtype=int), 0, f

    mode = str(mode).lower().strip()
    if mode == "poisson":
        sig = rng.poisson(float(strength) * w)
        return sig.astype(int), int(np.sum(sig)), f

    # multinomial: interpret strength as total yield parameter; if Σw<1, thin first.
    Ntot = int(np.round(float(strength)))
    if Ntot <= 0:
        return np.zeros_like(w, dtype=int), 0, f

    if f < 1.0:
        Nwin = int(rng.binomial(Ntot, f))
        if Nwin <= 0:
            return np.zeros_like(w, dtype=int), 0, f
        w_norm = w / f
        sig = _inject_counts_weighted(w_norm, Nwin, rng)
        return sig.astype(int), int(np.sum(sig)), f

    # f≈1: distribute all Ntot across bins
    w_norm = w / f
    sig = _inject_counts_weighted(w_norm, Ntot, rng)
    return sig.astype(int), int(np.sum(sig)), f


def _sigmaA_reference(
    pred: BlindPrediction,
    mass: float,
    *,
    source: str = "asimov",
    rng: Optional[np.random.Generator] = None,
) -> float:
    """Estimate σ_A(m) under background-only for 'sigma-level' injection scaling.

    source:
      - 'asimov' : n_obs := b_mean
      - 'poisson': n_obs := Poisson(b_mean)  (single draw)
    """
    tmpl = build_template(pred.edges, mass, pred.sigma_val, norm=None)
    b = np.asarray(pred.mu, float)
    if str(source).lower().strip() == "poisson":
        if rng is None:
            rng = np.random.default_rng(12345)
        n_ref = rng.poisson(b)
    else:
        n_ref = b  # Asimov

    d = fit_A_profiled_gaussian_details(n_ref, pred.mu, pred.cov, tmpl, allow_negative=True)
    return float(d.get("sigma_A", np.nan))



def run_injection_extraction_toys(
    ds: DatasetConfig,
    *,
    masses: List[float],
    strengths: List[float],
    n_toys: int,
    outdir: str,
    seed: int = 1,
    inj_mode: str = "multinomial",
    strengths_mode: Optional[str] = None,
    sigma_multipliers: Optional[List[float]] = None,
    sigma_source: Optional[str] = None,
    refit_gp_on_toy: Optional[bool] = None,
    refit_restarts: Optional[int] = None,
    refit_optimize: Optional[bool] = None,
    inj_shape_mode: Optional[str] = None,
    train_exclude_nsigma: Optional[float] = None,
) -> pd.DataFrame:
    """Run injection+extraction toys for one dataset.

    strengths_mode:
      - 'absolute': interpret `strengths` as absolute A_inj values (counts)
      - 'sigmaA'  : interpret `strengths` as σ_A multipliers (use sigma_multipliers if provided)

    Two toy-generation modes are supported:

    (1) Conditional blind-window toys (fast; matches v14 behavior):
        - Draw background truth only in the blind window from the conditional GP posterior MVN (μ, Σ)
          obtained from the *real-data* sideband GP fit.
        - Poisson-sample the observed blind-window counts.
        - Inject signal ONLY into the blind window (by construction, no sideband leakage).

        This mode is ideal for validating the profiled-A machinery and pull math.

    (2) Full procedural toys with GP refit (slow; reviewer/coverage studies):
        - Draw a full-range pseudo-dataset: Poisson( μ_full ) (+ injected signal).
        - Refit the GP on the toy *sidebands*, excluding a configurable training window.
        - Predict (μ, Σ) in the blind window using the toy-trained GP.
        - Extract A in the blind window using that prediction.

        v15 additions for this mode:
          - inj_shape_mode: "full" (inject full Gaussian line-shape across full histogram)
                           "window" (inject only in blind window; diagnostic)
          - train_exclude_nsigma: can be set > BLIND_NSIGMA to reduce sideband signal leakage bias.

    Notes
    -----
    In full-refit mode, if the injected signal has non-negligible tails outside the blind window,
    those tails can enter the GP sideband training region unless the training exclusion is wide.
    This can bias A_hat low (the GP absorbs part of the signal). The `f_train_frac` and `Nsig_train`
    outputs quantify this effect per mass point and toy.
    """
    ensure_dir(outdir)
    rng = np.random.default_rng(int(seed))

    strengths_mode = (strengths_mode or globals().get("INJ_STRENGTH_MODE", "absolute")).lower().strip()
    sigma_source = (sigma_source or globals().get("INJ_SIGMA_A_SOURCE", "asimov")).lower().strip()

    # Mode toggles
    if refit_gp_on_toy is None:
        refit_gp_on_toy = bool(globals().get("INJ_REFIT_GP_ON_TOY", False))
    refit_gp_on_toy = bool(refit_gp_on_toy)

    if refit_restarts is None:
        refit_restarts = int(globals().get("INJ_REFIT_GP_RESTARTS", 0))
    refit_restarts = int(refit_restarts)

    if refit_optimize is None:
        refit_optimize = bool(globals().get("INJ_REFIT_GP_OPTIMIZE", True))
    refit_optimize = bool(refit_optimize)

    inj_shape_mode = (inj_shape_mode or globals().get("INJ_SHAPE_MODE", "full")).lower().strip()
    if inj_shape_mode not in ("full", "window", "blind", "blind_only"):
        inj_shape_mode = "full"

    if train_exclude_nsigma is None:
        train_exclude_nsigma = globals().get("INJ_TRAIN_EXCLUDE_NSIGMA", None)
    train_exclude_nsigma = float(train_exclude_nsigma) if train_exclude_nsigma is not None else None

    # Strength grid
    if strengths_mode == "sigmaa":
        mults = sigma_multipliers if sigma_multipliers is not None else globals().get("INJ_SIGMA_MULTIPLIERS", [0.0, 1.0, 2.0, 3.0])
        strength_tags = [float(x) for x in mults]
    else:
        strength_tags = [float(x) for x in strengths]
    n_strength = len(strength_tags)

    rows: List[Dict[str, object]] = []

    for m in masses:
        m = float(m)
        pred = estimate_background_for_dataset(ds, m)

        # Template / fraction in blind window (uses global TEMPLATE_NORM_MODE when norm=None)
        tmpl_win = build_template(pred.edges, m, pred.sigma_val, norm=None)
        f_win = float(np.sum(tmpl_win))

        # Reference σ_A for sigma-level injection scaling (and for axis conversion)
        sigmaA_ref = _sigmaA_reference(pred, m, source=str(sigma_source), rng=rng)

        # Determine actual injection strengths for this mass
        if strengths_mode == "sigmaa":
            A_inj_list = [float(tag) * float(sigmaA_ref) for tag in strength_tags]
            inj_nsigma_list = list(strength_tags)
        else:
            A_inj_list = list(strength_tags)
            inj_nsigma_list = [float(A) / float(sigmaA_ref) if np.isfinite(sigmaA_ref) and sigmaA_ref > 0 else np.nan for A in A_inj_list]

        # Full-range arrays (available in both toy modes; used for leakage diagnostics)
        x_full = np.asarray(pred.x_full, float).reshape(-1)
        edges_full = np.asarray(pred.edges_full, float)
        tmpl_full = build_template(edges_full, m, pred.sigma_val, norm=None)
        f_full = float(np.sum(tmpl_full))

        blind = tuple(pred.blind)

        # Decide the sideband training exclusion window (used only in full_refit mode).
        if refit_gp_on_toy and (train_exclude_nsigma is not None):
            train_nsig = float(train_exclude_nsigma)
            blind_train = (m - train_nsig * float(pred.sigma_val), m + train_nsig * float(pred.sigma_val))
        else:
            # Use the training-exclusion choice used for the real-data GP prediction
            train_nsig = float(getattr(pred, "train_exclude_nsigma", float(globals().get("GP_TRAIN_EXCLUDE_NSIGMA", BLIND_NSIGMA))))
            blind_train = tuple(getattr(pred, "blind_train", blind))

        msk_blind = (x_full >= blind[0]) & (x_full <= blind[1])
        if int(np.sum(msk_blind)) <= 0:
            raise RuntimeError(f"[inj][{ds.key}] m={m:.6g}: blind window has no bins; cannot run toys")
        x_win = x_full[msk_blind]

        msk_train = (x_full < blind_train[0]) | (x_full > blind_train[1])
        # Gaussian template fraction that *lands in the GP training region* (potential leakage)
        try:
            f_train = float(np.sum(tmpl_full[msk_train])) if tmpl_full.shape[0] == x_full.shape[0] else np.nan
        except Exception:
            f_train = np.nan
        f_train_frac = float(f_train / f_full) if (np.isfinite(f_train) and np.isfinite(f_full) and f_full > 0) else np.nan

        # Precompute objects used by the chosen toy mode
        if refit_gp_on_toy:
            toy_mode = "full_refit"

            mu_full = np.asarray(pred.mu_full, float).reshape(-1)

            # Kernel for the GP refit (clone happens inside _fit_gpr)
            ker = make_kernel_for_dataset(ds, mass=m, optimized_by_year=_OPTIMIZED_LS_BY_YEAR)

            # Use the data-fit best LS as an initial value (helps stability; esp. if refit_optimize=False)
            try:
                if hasattr(ker, "k2") and hasattr(ker.k2, "length_scale") and np.isfinite(getattr(pred, "ls_opt", np.nan)):
                    ker.k2.length_scale = float(pred.ls_opt)
            except Exception:
                pass

        else:
            toy_mode = "conditional_gp"

            # Background MVN draws in blind window (conditional on real-data sidebands)
            b_draws = draw_bkg_mvn_nonneg(
                pred.mu, pred.cov, int(n_toys), rng,
                method=MVN_TRUNC_METHOD,
                max_tries=MVN_TRUNC_MAX_TRIES,
            )

        for j, (A_inj, inj_nsigma) in enumerate(zip(A_inj_list, inj_nsigma_list)):
            A_inj = float(A_inj)

            for i in range(int(n_toys)):
                refit_ok = np.nan
                nll = np.nan

                if refit_gp_on_toy:
                    # --- Full-range pseudo-dataset toy (background + injected signal) ---
                    bkg_full = rng.poisson(np.clip(mu_full, 0.0, None)).astype(int)

                    # Inject signal: full-shape or window-only diagnostic
                    if inj_shape_mode in ("window", "blind", "blind_only"):
                        sig_full = np.zeros_like(bkg_full, dtype=int)
                        sig_win, _, _ = _inject_counts_from_template(tmpl_win, A_inj, rng, mode=inj_mode)
                        idx_blind = np.where(msk_blind)[0]
                        sig_win = np.asarray(sig_win, int).reshape(-1)
                        if sig_win.shape[0] == idx_blind.shape[0]:
                            sig_full[idx_blind] = sig_win
                        else:
                            # best-effort alignment (should not happen in normal binning)
                            n = min(sig_win.shape[0], idx_blind.shape[0])
                            sig_full[idx_blind[:n]] = sig_win[:n]
                        Nsig_full = int(np.sum(sig_full))
                    else:
                        sig_full, Nsig_full, _ = _inject_counts_from_template(tmpl_full, A_inj, rng, mode=inj_mode)

                    y_full_toy = (bkg_full + sig_full).astype(int)

                    obs = y_full_toy[msk_blind].astype(int)
                    Nsig_win = int(np.sum(sig_full[msk_blind]))
                    Nsig_train = int(np.sum(sig_full[msk_train]))

                    # --- GP refit on toy sidebands (excluding blind_train) ---
                    mu_fit = np.asarray(pred.mu, float)
                    cov_fit = np.asarray(pred.cov, float)
                    try:
                        X_train = x_full[msk_train]
                        y_train = y_full_toy[msk_train].astype(float)

                        gpr = _fit_gpr(
                            X_train, y_train,
                            restarts=int(refit_restarts),
                            kernel=ker,
                            optimize=bool(refit_optimize),
                        )
                        mu_fit, cov_fit = _predict_counts_from_log_gpr(gpr, x_win)
                        mu_fit = np.asarray(mu_fit, float)
                        cov_fit = np.asarray(cov_fit, float)
                        refit_ok = True
                    except Exception:
                        # Fall back to the original (real-data-conditioned) prediction
                        refit_ok = False

                    fit = fit_A_profiled_gaussian(
                        obs, mu_fit, cov_fit, tmpl_win,
                        allow_negative=bool(globals().get("INJ_EXTRACT_ALLOW_NEGATIVE", True)),
                    )

                else:
                    # --- Conditional GP-posterior toy (v14 behavior; blind window only) ---
                    b = b_draws[i]

                    sig_counts, Nsig_win, _ = _inject_counts_from_template(tmpl_win, A_inj, rng, mode=inj_mode)

                    lam = np.clip(np.asarray(b, float), 0.0, None) + np.clip(np.asarray(sig_counts, float), 0.0, None)
                    obs = rng.poisson(lam).astype(int)
                    Nsig_train = 0

                    fit = fit_A_profiled_gaussian(
                        obs, np.asarray(pred.mu, float), np.asarray(pred.cov, float), tmpl_win,
                        allow_negative=bool(globals().get("INJ_EXTRACT_ALLOW_NEGATIVE", True)),
                    )

                A_hat = float(fit.get("A_hat", np.nan))
                sigma_A = float(fit.get("sigma_A", np.nan))
                success = bool(fit.get("success", False))
                nll = float(fit.get("nll", np.nan))

                Zhat = A_hat / sigma_A if (np.isfinite(sigma_A) and sigma_A > 0) else np.nan
                pull_param = (A_hat - A_inj) / sigma_A if (np.isfinite(sigma_A) and sigma_A > 0) else np.nan

                rows.append({
                    "dataset": ds.key,
                    "mass_GeV": m,
                    "toy": int(i),
                    "strength": A_inj,
                    "inj_nsigma": float(inj_nsigma),
                    "sigmaA_ref": float(sigmaA_ref),
                    "sigma_val": float(pred.sigma_val),
                    "sigma_x": float(pred.sigma_x),

                    # Template fractions (useful for leakage studies)
                    "f_win": float(f_win),
                    "f_full": float(f_full),
                    "f_train": float(f_train),
                    "f_train_frac": float(f_train_frac),

                    # Window choices (for provenance)
                    "blind_nsigma": float(globals().get("BLIND_NSIGMA", BLIND_NSIGMA)),
                    "train_exclude_nsigma": float(train_nsig),
                    "inj_shape_mode": str(inj_shape_mode),

                    # Fit outputs
                    "A_hat": float(A_hat),
                    "sigma_A": float(sigma_A),
                    "Zhat": float(Zhat),
                    "pull_param": float(pull_param),
                    "Nsig_win": int(Nsig_win),
                    "Nsig_train": int(Nsig_train),
                    "success": bool(success),
                    "nll": float(nll),

                    # bookkeeping
                    "toy_mode": str(toy_mode),
                    "refit_gp_on_toy": bool(refit_gp_on_toy),
                    "refit_ok": refit_ok,
                    "refit_restarts": int(refit_restarts),
                    "refit_optimize": bool(refit_optimize),
                })

        print(f"[inj] {ds.key} m={m:.6g} GeV done ({toy_mode}; {n_strength} strengths × {n_toys} toys)")

    df_toys = pd.DataFrame(rows)

    # Save a single CSV (compact)
    out_csv = os.path.join(outdir, f"inj_extract_toys_{ds.key}.csv")
    df_toys.to_csv(out_csv, index=False)
    print("[inj] wrote", out_csv)

    return df_toys


def summarize_injection_grid(df_toys: pd.DataFrame) -> pd.DataFrame:
    """Summarize injection toys by (mass, strength)."""
    if df_toys.empty:
        return pd.DataFrame()

    g = df_toys.groupby(["dataset", "mass_GeV", "strength"], dropna=False)

    def q(x, p):
        return float(np.quantile(np.asarray(x, float), p))

    def _col_mean(sub, name):
        if name not in sub.columns:
            return np.nan
        return float(np.nanmean(sub[name].to_numpy(float)))

    def _col_first(sub, name):
        if name not in sub.columns:
            return None
        vals = [v for v in sub[name].to_numpy() if v is not None and str(v) != "nan"]
        return vals[0] if vals else None

    def _bool_rate(sub, name):
        if name not in sub.columns:
            return np.nan
        arr = sub[name].to_numpy()
        num = []
        for v in arr:
            if v is True:
                num.append(1.0)
            elif v is False:
                num.append(0.0)
            else:
                num.append(np.nan)
        return float(np.nanmean(np.asarray(num, float))) if len(num) else np.nan

    rows = []
    for (ds, m, Ainj), sub in g:
        Ahat = sub["A_hat"].to_numpy(float)
        sigA = sub["sigma_A"].to_numpy(float)
        pull = sub["pull_param"].to_numpy(float)
        Zhat = sub["Zhat"].to_numpy(float)

        rows.append({
            "dataset": ds,
            "mass_GeV": float(m),
            "strength": float(Ainj),
            "inj_nsigma": _col_mean(sub, "inj_nsigma"),
            "sigmaA_ref": _col_mean(sub, "sigmaA_ref"),

            # Template leakage diagnostics (v15)
            "f_win": _col_mean(sub, "f_win"),
            "f_full": _col_mean(sub, "f_full"),
            "f_train_frac": _col_mean(sub, "f_train_frac"),
            "Nsig_train_mean": _col_mean(sub, "Nsig_train"),
            "refit_ok_rate": _bool_rate(sub, "refit_ok"),
            "inj_shape_mode": _col_first(sub, "inj_shape_mode"),
            "toy_mode": _col_first(sub, "toy_mode"),

            # Fit summaries
            "A_hat_mean": float(np.nanmean(Ahat)),
            "A_hat_std": float(np.nanstd(Ahat, ddof=1)),
            "sigma_A_mean": float(np.nanmean(sigA)),

            "pull_mean": float(np.nanmean(pull)),
            "pull_std": float(np.nanstd(pull, ddof=1)),
            "pull_q16": q(pull, 0.16),
            "pull_q84": q(pull, 0.84),
            "pull_q02": q(pull, 0.025),
            "pull_q97": q(pull, 0.975),

            "cov_1sigma": float(np.nanmean(np.abs(pull) < 1.0)),
            "cov_2sigma": float(np.nanmean(np.abs(pull) < 2.0)),

            "Zhat_mean": float(np.nanmean(Zhat)),
            "Zhat_q16": q(Zhat, 0.16),
            "Zhat_q84": q(Zhat, 0.84),
        })

    df_sum = pd.DataFrame(rows).sort_values(["dataset", "mass_GeV", "strength"]).reset_index(drop=True)
    return df_sum


def _grid(ax):
    ax.grid(True, which="major", alpha=0.25)
    try:
        ax.minorticks_on()
        ax.grid(True, which="minor", alpha=0.12)
    except Exception:
        pass


def plot_linearity(df_sum: pd.DataFrame, *, xvar: str = "inj_nsigma", title: str = "", outpath: Optional[str] = None):
    """Linearity: Ahat_mean vs injected strength."""
    if df_sum.empty:
        return

    fig, ax = plt.subplots(figsize=(9.2, 5.4))
    for (ds, m), sub in df_sum.groupby(["dataset", "mass_GeV"]):
        x = sub[xvar].to_numpy(float) if xvar in sub.columns else sub["strength"].to_numpy(float)
        y = sub["A_hat_mean"].to_numpy(float)
        yerr = sub["A_hat_std"].to_numpy(float)
        ax.errorbar(x, y, yerr=yerr, fmt="o-", label=f"{ds} @ {m:.3f} GeV")

    ax.set_xlabel("Injected strength" + (r" ($A_{inj}/\sigma_A$)" if xvar == "inj_nsigma" else " (counts)"))
    ax.set_ylabel(r"Mean extracted $\hat A$")
    ax.set_title(title or "Injection/extraction linearity")
    _grid(ax)
    ax.legend(loc="best", frameon=True, fontsize=9)
    fig.tight_layout()
    if outpath:
        fig.savefig(outpath, dpi=int(globals().get("SAVEFIG_DPI", 300)))
    plt.show()
    plt.close(fig)


def plot_residual_pull(df_sum: pd.DataFrame, *, xvar: str = "inj_nsigma", title: str = "", outpath: Optional[str] = None):
    """Mean pull (Ahat−Ainj)/σ_A vs injected strength."""
    if df_sum.empty:
        return

    fig, ax = plt.subplots(figsize=(9.2, 5.2))
    for (ds, m), sub in df_sum.groupby(["dataset", "mass_GeV"]):
        x = sub[xvar].to_numpy(float) if xvar in sub.columns else sub["strength"].to_numpy(float)
        y = sub["pull_mean"].to_numpy(float)
        yerr = sub["pull_std"].to_numpy(float) / np.sqrt(np.maximum(1.0, float(n_toys_for_summary(df_sum, ds, m))))
        ax.errorbar(x, y, yerr=yerr, fmt="o-", label=f"{ds} @ {m:.3f} GeV")

    ax.axhline(0.0, color="k", linewidth=1.0, alpha=0.5)
    ax.axhline(+0.2, color="k", linewidth=0.8, alpha=0.25, linestyle=":")
    ax.axhline(-0.2, color="k", linewidth=0.8, alpha=0.25, linestyle=":")
    ax.set_xlabel("Injected strength" + (r" ($A_{inj}/\sigma_A$)" if xvar == "inj_nsigma" else " (counts)"))
    ax.set_ylabel(r"Mean pull $(\hat A-A_{inj})/\sigma_A$")
    ax.set_title(title or "Residual pull vs injected strength")
    _grid(ax)
    ax.legend(loc="best", frameon=True, fontsize=9)
    fig.tight_layout()
    if outpath:
        fig.savefig(outpath, dpi=int(globals().get("SAVEFIG_DPI", 300)))
    plt.show()
    plt.close(fig)


def plot_pull_width(df_sum: pd.DataFrame, *, xvar: str = "inj_nsigma", title: str = "", outpath: Optional[str] = None):
    """Pull width (std) vs injected strength."""
    if df_sum.empty:
        return

    fig, ax = plt.subplots(figsize=(9.2, 5.2))
    for (ds, m), sub in df_sum.groupby(["dataset", "mass_GeV"]):
        x = sub[xvar].to_numpy(float) if xvar in sub.columns else sub["strength"].to_numpy(float)
        y = sub["pull_std"].to_numpy(float)
        ax.plot(x, y, marker="o", label=f"{ds} @ {m:.3f} GeV")

    ax.axhline(1.0, color="k", linewidth=1.0, alpha=0.5, linestyle="--", label="ideal width=1")
    ax.set_xlabel("Injected strength" + (r" ($A_{inj}/\sigma_A$)" if xvar == "inj_nsigma" else " (counts)"))
    ax.set_ylabel("Pull width (std)")
    ax.set_title(title or "Pull width vs injected strength")
    _grid(ax)
    ax.legend(loc="best", frameon=True, fontsize=9)
    fig.tight_layout()
    if outpath:
        fig.savefig(outpath, dpi=int(globals().get("SAVEFIG_DPI", 300)))
    plt.show()
    plt.close(fig)


def plot_detection_significance(df_sum: pd.DataFrame, *, xvar: str = "inj_nsigma", title: str = "", outpath: Optional[str] = None):
    """Mean Zhat = Ahat/σ_A vs injected strength."""
    if df_sum.empty:
        return

    fig, ax = plt.subplots(figsize=(9.2, 5.2))
    for (ds, m), sub in df_sum.groupby(["dataset", "mass_GeV"]):
        x = sub[xvar].to_numpy(float) if xvar in sub.columns else sub["strength"].to_numpy(float)
        y = sub["Zhat_mean"].to_numpy(float)
        ax.plot(x, y, marker="o", label=f"{ds} @ {m:.3f} GeV")

    ax.axhline(3.0, color="k", linewidth=0.9, alpha=0.35, linestyle=":", label="3σ")
    ax.axhline(5.0, color="k", linewidth=0.9, alpha=0.35, linestyle=":", label="5σ")
    ax.set_xlabel("Injected strength" + (r" ($A_{inj}/\sigma_A$)" if xvar == "inj_nsigma" else " (counts)"))
    ax.set_ylabel(r"Mean $\hat Z = \hat A/\sigma_A$")
    ax.set_title(title or "Detection significance vs injected strength")
    _grid(ax)
    ax.legend(loc="best", frameon=True, fontsize=9)
    fig.tight_layout()
    if outpath:
        fig.savefig(outpath, dpi=int(globals().get("SAVEFIG_DPI", 300)))
    plt.show()
    plt.close(fig)


def n_toys_for_summary(df_sum: pd.DataFrame, ds: str, m: float) -> int:
    """Best-effort retrieval of n_toys used for SEM scaling (if the caller wants it)."""
    # If df_sum came from a single df_toys, n_toys is constant; this helper is a no-op placeholder.
    return int(globals().get("INJ_NUM_TOYS", globals().get("INJ_N_TOYS", 1)))




def plot_coverage(df_sum: pd.DataFrame, *, xvar: str = "inj_nsigma", title: str = "", outpath: Optional[str] = None):
    """Coverage vs injection strength using pull-based coverage metrics."""
    if df_sum.empty or ("cov_1sigma" not in df_sum.columns):
        return

    fig, ax = plt.subplots(figsize=(9.2, 5.2))
    for (ds, m), sub in df_sum.groupby(["dataset", "mass_GeV"]):
        x = sub[xvar].to_numpy(float) if xvar in sub.columns else sub["strength"].to_numpy(float)
        ax.plot(x, sub["cov_1sigma"].to_numpy(float), marker="o", label=f"{ds} @ {m:.3f} GeV (|pull|<1)")
        ax.plot(x, sub["cov_2sigma"].to_numpy(float), marker="o", linestyle="--", label=f"{ds} @ {m:.3f} GeV (|pull|<2)")

    ax.axhline(0.6827, color="k", linewidth=1.0, alpha=0.35, linestyle=":", label="Gaussian 1σ (68.27%)")
    ax.axhline(0.9545, color="k", linewidth=1.0, alpha=0.35, linestyle=":", label="Gaussian 2σ (95.45%)")
    ax.set_xlabel("Injected strength" + (r" ($A_{inj}/\sigma_A$)" if xvar == "inj_nsigma" else " (counts)"))
    ax.set_ylabel("Coverage fraction")
    ax.set_ylim(0.0, 1.02)
    ax.set_title(title or "Pull-based coverage vs injected strength")
    _grid(ax)
    ax.legend(loc="best", frameon=True, fontsize=8)
    fig.tight_layout()
    if outpath:
        fig.savefig(outpath, dpi=int(globals().get("SAVEFIG_DPI", 300)))
    plt.show()
    plt.close(fig)




def plot_bias_vs_injected_strength(df_sum: pd.DataFrame, *, xvar: str = "inj_nsigma", title: str = "", outpath: Optional[str] = None):
    """Bias = <A_hat> - A_inj vs injected strength."""
    if df_sum.empty:
        return

    fig, ax = plt.subplots(figsize=(9.2, 5.2))
    for (ds, m), sub in df_sum.groupby(["dataset", "mass_GeV"]):
        x = sub[xvar].to_numpy(float) if xvar in sub.columns else sub["strength"].to_numpy(float)
        bias = sub["A_hat_mean"].to_numpy(float) - sub["strength"].to_numpy(float)
        ax.plot(x, bias, marker="o", label=f"{ds} @ {m:.3f} GeV")

    ax.axhline(0.0, color="k", linewidth=1.0, alpha=0.5, linestyle="--")
    ax.set_xlabel("Injected strength" + (r" ($A_{inj}/\sigma_A$)" if xvar == "inj_nsigma" else " (counts)"))
    ax.set_ylabel(r"Bias $\langle \hat A \rangle - A_{inj}$ (counts)")
    ax.set_title(title or "Extraction bias vs injected strength")
    _grid(ax)
    ax.legend(loc="best", frameon=True, fontsize=9)
    fig.tight_layout()
    if outpath:
        fig.savefig(outpath, dpi=int(globals().get("SAVEFIG_DPI", 300)))
    plt.show()
    plt.close(fig)


def plot_leakage_vs_mass(df_sum: pd.DataFrame, *, title: str = "", outpath: Optional[str] = None):
    """Template leakage fraction into the GP training region vs mass (v15)."""
    if df_sum.empty or ("f_train_frac" not in df_sum.columns):
        return

    # f_train_frac is constant for a given (dataset, mass) at fixed train_exclude_nsigma; drop duplicates.
    df_u = (
        df_sum.sort_values(["dataset", "mass_GeV", "inj_nsigma"])
        .drop_duplicates(subset=["dataset", "mass_GeV"])
        .copy()
    )

    fig, ax = plt.subplots(figsize=(9.2, 5.2))
    for ds, sub in df_u.groupby("dataset"):
        ax.plot(sub["mass_GeV"].to_numpy(float), sub["f_train_frac"].to_numpy(float), marker="o", label=str(ds))

    ax.set_xlabel("Mass [GeV]")
    ax.set_ylabel("Template fraction in GP training region (f_train / f_full)")
    ax.set_ylim(0.0, 1.02)
    ax.set_title(title or "Signal-template leakage into GP training region vs mass")
    _grid(ax)
    ax.legend(loc="best", frameon=True, fontsize=9)
    fig.tight_layout()
    if outpath:
        fig.savefig(outpath, dpi=int(globals().get("SAVEFIG_DPI", 300)))
    plt.show()
    plt.close(fig)


print("Injection/extraction toy suite ready (v15).")


## v10 additions: injection/extraction toy *sigma-level* diagnostics

The v9 injection/extraction suite already produced **residuals** and **linearity** plots.

In v10 we add two extra diagnostics that are often the quickest way to interpret both *bias* and *sensitivity*:

- **Detection significance**: \(Z \approx \hat A/\sigma_A\) vs injected strength.
- **Residual pull**: \((\hat A - A_{\rm truth})/\sigma_A\) vs injected strength (bias-in-\(\sigma\) units).

**Important interpretation note**

If you are plotting the *residual pull* for injections of 5000 vs 10000 events, and the extraction is unbiased + well-calibrated, the residual-pull distributions should look *very similar* (centered near 0 with width ~1). This is expected.  
To see sensitivity (i.e. how detectable the injected signal is), look at **\(Z\)** or **analytic \(p_0\)/\(Z\)** instead.


## Execution cells

1) Run the scan:
- produces `results_single.csv` and `results_combined.csv`
- produces per-mass folders with plots and JSONs (if enabled)

2) Make summary plots

3) Optionally compute expected bands

4) Optionally run injection/extraction


In [ ]:
# =============================================================================
# Cell 15b — Smoke test one mass point (prints errors immediately)
# =============================================================================
# Pick a representative mass inside the overlap (if any), otherwise inside the first dataset range.
def pick_test_mass(datasets: Dict[str, DatasetConfig]) -> float:
    lows = [d.m_low for d in datasets.values()]
    highs = [d.m_high for d in datasets.values()]
    # prefer an overlap if possible
    lo = max(lows); hi = min(highs)
    if lo < hi:
        return float(np.round(0.5*(lo+hi), 3))
    # else pick mid of first dataset
    d0 = list(datasets.values())[0]
    return float(np.round(0.5*(d0.m_low+d0.m_high), 3))

TEST_MASS = pick_test_mass(DATASETS)
print("TEST_MASS =", TEST_MASS, "GeV")

for key, ds in DATASETS.items():
    if ds.m_low <= TEST_MASS <= ds.m_high:
        print("\n--- testing", key, ds.label, "---")
        try:
            #res, pred = evaluate_single_dataset(ds, TEST_MASS, do_extraction=True)
            res, pred, _ = evaluate_single_dataset(ds, TEST_MASS, do_extraction=True)
            print("A_up =", res.A_up, " eps2_up =", res.eps2_up, " p0 =", res.p0_analytic, " Z =", res.Z_analytic)
            print("A_hat =", res.A_hat, "+/-", res.sigma_A, " success=", res.extract_success)
            # Make one plot to verify matplotlib write permissions
            tmp_dir = os.path.join(OUTPUT_DIR, "smoke_test")
            ensure_dir(tmp_dir)
            plot_full_range(ds, TEST_MASS, pred, os.path.join(tmp_dir, f"{key}_fit_full.png"))
            print("wrote", os.path.join(tmp_dir, f"{key}_fit_full.png"))
        except Exception as e:
            print("FAILED:", e)


In [ ]:
# =============================================================================
# Cell 16 — RUN: scan
# =============================================================================

df_single, df_comb = run_scan(DATASETS)

print("Scan complete. DataFrames: df_single, df_comb")


In [ ]:
# =============================================================================
# Cell 15 — RUN: summary plots (v15)
# =============================================================================
#
# Produces:
#   - Per-dataset observed limit curves (A_up and eps2_up), with expected bands overlay if available
#   - Combined eps2 curve (observed), with expected bands overlay if available
#   - Overlay plots comparing observed limits across datasets
#   - GP hyper-parameter QA plots
# =============================================================================

summary_dir = os.path.join(OUTPUT_DIR, "summary_plots")
ensure_dir(summary_dir)

# --- Per-dataset plots ---
for ds in DATASETS.values():
    if not ds.enabled:
        continue
    plot_dataset_limit_curves(df_single, ds, outdir=summary_dir)

# --- Overlay observed limits across datasets ---
plot_observed_overlay(
    df_single,
    outdir=summary_dir,
    ycol="eps2_up",
    ylabel=r"$\varepsilon^2_{\rm up}$",
    title="Observed ε² upper limits (overlay)",
    outname="overlay_eps2_up.png",
)
plot_observed_overlay(
    df_single,
    outdir=summary_dir,
    ycol="A_up",
    ylabel=r"$A_{\rm up}$ (events)",
    title="Observed signal-yield upper limits (overlay)",
    outname="overlay_A_up.png",
)

# --- Combined (eps2 only) ---
if df_comb is not None and (not df_comb.empty):
    # best-effort: pick an existing combined expected-band file (if present)
    import glob
    cand = sorted(glob.glob(os.path.join(OUTPUT_DIR, "ul_bands_combined_*.csv")))
    bands_csv = cand[-1] if len(cand) > 0 else None
    label = "combined"
    if "datasets" in df_comb.columns and df_comb["datasets"].notna().any():
        label = "combined_" + str(df_comb["datasets"].dropna().iloc[0]).replace("+", "_")

    plot_combined_eps2_curve(df_comb, outdir=summary_dir, bands_csv=bands_csv, label=label)

# --- GP hyperparameter QA ---
gp_dir = os.path.join(summary_dir, "gp_hyperparameters")
plot_gp_hyperparameters(df_single, outdir=gp_dir)

print(f"[summary] wrote plots to {summary_dir}")


In [ ]:
# =============================================================================
# Cell 18 — RUN: expected bands (optional, v14)
# =============================================================================
#
# This produces *expected* bands under background-only.
#
# v14 update:
#   - For single-dataset bands, we now write and plot BOTH:
#       (1) A_up expected bands (signal-yield UL)
#       (2) eps2_up expected bands
#   - For combined bands, we only plot eps2_up bands (A_up is not meaningful for a combination).
# =============================================================================

# --- Single-dataset expected bands ---
if bool(globals().get("MAKE_UL_BANDS", False)) and (RUN_LIMIT_BANDS_ON in DATASETS):
    ds_b = DATASETS[RUN_LIMIT_BANDS_ON]
    masses = np.round(np.arange(ds_b.m_low, ds_b.m_high + MASS_STEP_GEV / 2, MASS_STEP_GEV), 3)

    print(f"[bands] running expected UL bands for dataset={ds_b.key} with {masses.size} mass points")
    print(f"[bands] n_toys={UL_BANDS_TOYS}  restarts={UL_BANDS_N_RESTARTS}  "
          f"workers={UL_BANDS_N_WORKERS} backend={UL_BANDS_PARALLEL_BACKEND}  "
          f"threads/worker={UL_BANDS_THREADS_PER_WORKER}")

    df_bands = expected_ul_bands_for_dataset(
        ds_b, masses, n_toys=UL_BANDS_TOYS,
        restarts=UL_BANDS_N_RESTARTS,
        n_workers=UL_BANDS_N_WORKERS,
        backend=UL_BANDS_PARALLEL_BACKEND,
        threads_per_worker=UL_BANDS_THREADS_PER_WORKER,
        seed=UL_BANDS_SEED,
        cls_mode="asymptotic",
    )

    out_csv = os.path.join(OUTPUT_DIR, f"ul_bands_{ds_b.key}.csv")
    df_bands.to_csv(out_csv, index=False)
    print("[bands] wrote", out_csv)

    # plots
    out_band_A   = os.path.join(OUTPUT_DIR, f"ul_bands_A_{ds_b.key}.png")
    out_band_eps = os.path.join(OUTPUT_DIR, f"ul_bands_eps2_{ds_b.key}.png")
    out_pval = os.path.join(OUTPUT_DIR, f"ul_pvals_{ds_b.key}.png")
    out_p0   = os.path.join(OUTPUT_DIR, f"p0_analytic_{ds_b.key}.png")

    plot_ul_bands(df_bands, use_eps2=False,
                  title=f"Expected UL bands on A (signal yield) — {ds_b.key}", outpath=out_band_A)
    plot_ul_bands(df_bands, use_eps2=True,
                  title=f"Expected UL bands on ε² — {ds_b.key}", outpath=out_band_eps)

    plot_ul_pvalues(df_bands, title=f"UL p-values ({ds_b.key})", outpath=out_pval)
    plot_analytic_p0(df_bands, title=f"Analytic p0 ({ds_b.key})", outpath=out_p0)
    plot_Z_local_global(df_bands, title=f"Analytic local/global Z ({ds_b.key})", outpath=os.path.join(OUTPUT_DIR, f"Z_local_global_{ds_b.key}.png"))

else:
    print("[bands] Single-dataset bands: MAKE_UL_BANDS=False or RUN_LIMIT_BANDS_ON not enabled / set to 'none'")

# --- Combined expected bands (eps2 only) ---
if bool(globals().get("RUN_COMBINED_BANDS", False)):
    keys = [k for k in list(globals().get("RUN_COMBINED_BANDS_DATASETS", [])) if k in DATASETS]
    ds_combo = [DATASETS[k] for k in keys]
    if len(ds_combo) == 0:
        print("[bands][combined] no valid datasets selected in RUN_COMBINED_BANDS_DATASETS")
    else:
        lo = min([d.m_low for d in ds_combo])
        hi = max([d.m_high for d in ds_combo])
        masses = np.round(np.arange(lo, hi + MASS_STEP_GEV / 2, MASS_STEP_GEV), 3)

        if bool(globals().get("RUN_COMBINED_BANDS_ONLY_OVERLAP", True)):
            def _n_active(m):
                return sum([(m >= d.m_low) and (m <= d.m_high) and bool(getattr(d, "enabled", True)) for d in ds_combo])
            masses = np.array([m for m in masses if _n_active(float(m)) >= 2], dtype=float)

        tag = "_".join([d.key for d in ds_combo])
        print(f"[bands][combined] running combined expected bands for {tag} with {masses.size} mass points")
        print(f"[bands][combined] n_toys={COMBINED_BANDS_N_TOYS}  restarts={UL_BANDS_N_RESTARTS}  "
              f"workers={UL_BANDS_N_WORKERS} backend={UL_BANDS_PARALLEL_BACKEND}  "
              f"threads/worker={UL_BANDS_THREADS_PER_WORKER}")

        df_cb = expected_ul_bands_for_combination(
            ds_combo, masses, n_toys=COMBINED_BANDS_N_TOYS,
            restarts=UL_BANDS_N_RESTARTS,
            n_workers=UL_BANDS_N_WORKERS,
            backend=UL_BANDS_PARALLEL_BACKEND,
            threads_per_worker=UL_BANDS_THREADS_PER_WORKER,
            seed=COMBINED_BANDS_SEED,
            cls_mode="asymptotic",
        )

        out_csv = os.path.join(OUTPUT_DIR, f"ul_bands_combined_{tag}.csv")
        df_cb.to_csv(out_csv, index=False)
        print("[bands][combined] wrote", out_csv)

        out_band_eps = os.path.join(OUTPUT_DIR, f"ul_bands_eps2_combined_{tag}.png")
        out_pval = os.path.join(OUTPUT_DIR, f"ul_pvals_combined_{tag}.png")
        out_p0   = os.path.join(OUTPUT_DIR, f"p0_analytic_combined_{tag}.png")

        plot_ul_bands(df_cb, use_eps2=True,
                      title=f"Combined expected UL bands on ε² — {tag}", outpath=out_band_eps)
        plot_ul_pvalues(df_cb, title=f"Combined UL p-values ({tag})", outpath=out_pval)
        plot_analytic_p0(df_cb, title=f"Combined analytic p0 ({tag})", outpath=out_p0)
        plot_Z_local_global(df_cb, title=f"Combined local/global Z ({tag})", outpath=os.path.join(OUTPUT_DIR, f"Z_local_global_combined_{tag}.png"))

else:
    print("[bands] Combined bands: RUN_COMBINED_BANDS=False")

print("Ready to run expected bands if desired (v15).")


In [ ]:
# =============================================================================
# Cell 19 — RUN: injection/extraction toys + plot suite (optional) (v15)
# =============================================================================

if INJECT_SIGNAL and (INJ_DATASET_KEY in DATASETS):
    ds_inj = DATASETS[INJ_DATASET_KEY]
    inj_dir = os.path.join(OUTPUT_DIR, "injection_extraction_v15")
    ensure_dir(inj_dir)

    df_inj_toys = run_injection_extraction_toys(
        ds=ds_inj,
        masses=INJ_MASSES_GEV,
        strengths=INJ_STRENGTHS,
        n_toys=int(INJ_NUM_TOYS),
        outdir=inj_dir,
        seed=int(INJ_SEED),
        inj_mode=str(INJ_MODE),
        strengths_mode=str(INJ_STRENGTH_MODE),
        sigma_multipliers=list(INJ_SIGMA_MULTIPLIERS),
        sigma_source=str(INJ_SIGMA_A_SOURCE),
    )

    df_inj_sum = summarize_injection_grid(df_inj_toys)
    out_sum = os.path.join(inj_dir, f"inj_extract_summary_{ds_inj.key}.csv")
    df_inj_sum.to_csv(out_sum, index=False)
    print("[inj] wrote", out_sum)
    display(df_inj_sum.head(12))

    # Suite plots (publication-friendly defaults)
    plot_linearity(df_inj_sum, outpath=os.path.join(inj_dir, f"linearity_{ds_inj.key}.png"))
    plot_bias_vs_injected_strength(df_inj_sum, xvar='inj_nsigma', outpath=os.path.join(inj_dir, 'bias_vs_inj_nsigma.png'))
    plot_leakage_vs_mass(df_inj_sum, outpath=os.path.join(inj_dir, 'leakage_vs_mass.png'))
    plot_residual_pull(df_inj_sum, outpath=os.path.join(inj_dir, f"pull_mean_{ds_inj.key}.png"))
    plot_pull_width(df_inj_sum, outpath=os.path.join(inj_dir, f"pull_width_{ds_inj.key}.png"))
    plot_coverage(df_inj_sum, outpath=os.path.join(inj_dir, f"coverage_{ds_inj.key}.png"))
    plot_detection_significance(df_inj_sum, outpath=os.path.join(inj_dir, f"Zhat_{ds_inj.key}.png"))

    print(f"[inj] wrote CSVs + plots to: {inj_dir}")
else:
    print("Injection disabled or dataset key not enabled")

print("Ready to run injection/extraction toys if desired.")


In [ ]:
# =============================================================================
# Cell 16 — EXTRACTION_DISPLAY utilities (v15)
# =============================================================================
#
# Goal: publication-quality, reviewer-facing "single-point" extraction visuals that:
#   - show full-range context and GP mean
#   - show the blind-window fit details (profiled background and fitted λ)
#   - show the extracted signal component with injected truth overlay
#
# This is based on the v12.5 display style, updated to use the v14 template convention.
# =============================================================================



# --- Fallback: define injection helpers if the injection-toy cell has not been executed ---
if "_inject_counts_from_template" not in globals():
    def _inject_counts_weighted(weights: np.ndarray, total: int, rng: np.random.Generator) -> np.ndarray:
        total = int(total)
        if total <= 0:
            return np.zeros_like(weights, dtype=int)
        w = np.asarray(weights, float)
        s = float(np.sum(w))
        if s <= 0:
            return np.zeros_like(weights, dtype=int)
        w = w / s
        return rng.multinomial(total, w)

    def _inject_counts_from_template(template: np.ndarray, strength: float, rng: np.random.Generator, mode: str = "multinomial"):
        w = np.asarray(template, float)
        f = float(np.sum(w))
        if (not np.isfinite(f)) or (f <= 0.0) or (not np.isfinite(strength)) or (strength <= 0.0):
            return np.zeros_like(w, dtype=int), 0, f
        mode = str(mode).lower().strip()
        if mode == "poisson":
            sig = rng.poisson(float(strength) * w)
            return sig.astype(int), int(np.sum(sig)), f
        Ntot = int(np.round(float(strength)))
        if Ntot <= 0:
            return np.zeros_like(w, dtype=int), 0, f
        if f < 1.0:
            Nwin = int(rng.binomial(Ntot, f))
            if Nwin <= 0:
                return np.zeros_like(w, dtype=int), 0, f
            sig = _inject_counts_weighted(w / f, Nwin, rng)
            return sig.astype(int), int(np.sum(sig)), f
        sig = _inject_counts_weighted(w / f, Ntot, rng)
        return sig.astype(int), int(np.sum(sig)), f


def _grid_local(ax):
    ax.grid(True, which="major", alpha=0.25)
    try:
        ax.minorticks_on()
        ax.grid(True, which="minor", alpha=0.12)
    except Exception:
        pass




# --- v14.5: plotting helpers (fallbacks, in case earlier plot-helper cells were skipped) ---
if "_set_title_above" not in globals():
    def _set_title_above(ax, title: str, *, pad: float = 12.0):
        try:
            ax.set_title(str(title), pad=float(pad))
        except Exception:
            ax.set_title(str(title))

if "_shade_blind_window" not in globals():
    def _shade_blind_window(
        ax,
        blind: Tuple[float, float],
        *,
        alpha: float = 0.25,
        color: str = "0.90",
        label: str = "blind window",
        zorder: int = 0,
    ):
        try:
            ax.axvspan(float(blind[0]), float(blind[1]), color=str(color), alpha=float(alpha), lw=0, zorder=int(zorder), label=label)
        except Exception:
            ax.axvspan(float(blind[0]), float(blind[1]), alpha=float(alpha), lw=0, zorder=int(zorder), label=label)
        try:
            ax.axvline(float(blind[0]), color="0.5", lw=1.0, alpha=0.6, zorder=int(zorder)+1)
            ax.axvline(float(blind[1]), color="0.5", lw=1.0, alpha=0.6, zorder=int(zorder)+1)
        except Exception:
            pass

if "_add_info_box" not in globals():
    def _add_info_box(
        ax,
        text: str,
        *,
        loc: str = "upper right",
        fontsize: int = 9,
        alpha: float = 0.90,
    ):
        try:
            from matplotlib.offsetbox import AnchoredText
            at = AnchoredText(str(text), loc=str(loc), prop=dict(size=int(fontsize)), frameon=True, borderpad=0.45)
            at.patch.set_alpha(float(alpha))
            try:
                at.patch.set_boxstyle("round,pad=0.35")
            except Exception:
                pass
            ax.add_artist(at)
            return at
        except Exception:
            ax.text(
                0.98, 0.98, str(text),
                transform=ax.transAxes,
                va="top", ha="right",
                fontsize=int(fontsize),
                bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="0.5", alpha=float(alpha)),
            )
            return None

def _blind_slice_from_full(
    x_full: np.ndarray,
    edges_full: np.ndarray,
    blind: Tuple[float, float],
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return (mask_blind, centers_blind, edges_blind_slice)."""
    Xc = np.asarray(x_full, float)
    edges = np.asarray(edges_full, float)
    msk = (Xc >= blind[0]) & (Xc <= blind[1])
    idx = np.where(msk)[0]
    if idx.size == 0:
        raise RuntimeError("Blind window has no bins")
    e_slice = edges[idx[0] : idx[-1] + 2]
    return msk, Xc[msk], e_slice



def make_extraction_display_v15(
    ds: DatasetConfig,
    m0: float,
    *,
    Ninj_total: float,
    seed: int = 123,
    inj_mode: str = "multinomial",
    refit_gp_on_toy: bool = True,
    outpath: str,
):
    """Create an extraction-display plot for one dataset, one mass point, one injection strength.

    v14.5 updates (publication / reviewer friendliness):
      - Panel 2 now shows *sideband data* (±0.5σ beyond the blind bounds), not only the blind bins.
      - Consistent blind-window shading + boundary lines (all panels that show the window).
      - Fit summary (Σw, Â, Ainj, pull, …) moved into a framed info box next to the legend.
      - `refit_gp_on_toy` is wired correctly (fixes a v14 keyword bug calling `_fit_gpr`).
    """
    rng = np.random.default_rng(int(seed))
    m0 = float(m0)

    pred = estimate_background_for_dataset(ds, m0)

    # Full histogram arrays
    x_full = np.asarray(pred.x_full, float).reshape(-1)
    edges_full = np.asarray(pred.edges_full, float)
    mu_full = np.asarray(pred.mu_full, float).reshape(-1)
    var_full_pred = getattr(pred, "var_full", None)

    # Full-range template (PDF-mode) for injecting *total* yield across the histogram range
    w_full = build_template(edges_full, m0, pred.sigma_val, norm="pdf")
    f_full = float(np.sum(w_full))

    # Blind-window slice (bin centers + edges that correspond to the blind bins on this histogram)
    msk_blind, x_win, edges_win = _blind_slice_from_full(x_full, edges_full, pred.blind)

    # Blind-window template weights (PDF-mode)
    w_win = build_template(edges_win, m0, pred.sigma_val, norm="pdf")
    f_win = float(np.sum(w_win))

    # --- Build a toy pseudo-dataset (full range) ---
    # Background draw: Poisson around the GP mean for full-range context (includes sideband stat fluctuations).
    bkg_full = rng.poisson(np.clip(mu_full, 0.0, None)).astype(int)

    # Inject signal across the full range
    sig_full, Nsig_full, _ = _inject_counts_from_template(w_full, float(Ninj_total), rng, mode=str(inj_mode))
    y_full_toy = (bkg_full + sig_full).astype(int)

    # Extract blind-window observed counts
    y_win_toy = y_full_toy[msk_blind].astype(int)

    # --- Optionally refit the GP on the toy sidebands ---
    # Default: use the original pred.mu/cov from the *real-data* sideband fit (conditional toys).
    mu_win = np.asarray(pred.mu, float)
    cov_win = np.asarray(pred.cov, float)
    mu_full_plot = mu_full
    var_full_plot = np.asarray(var_full_pred, float).reshape(-1) if (var_full_pred is not None) else None
    refit_ok = False

    if bool(refit_gp_on_toy):
        try:
            blind = pred.blind
            train_nsig = float(globals().get(
                "EXTRACTION_DISPLAY_TRAIN_EXCLUDE_NSIGMA",
                getattr(pred, "train_exclude_nsigma", float(globals().get("GP_TRAIN_EXCLUDE_NSIGMA", BLIND_NSIGMA)))
            ))
            blind_train = (float(mass) - train_nsig * float(pred.sigma_val), float(mass) + train_nsig * float(pred.sigma_val))
            msk_train = (x_full < blind_train[0]) | (x_full > blind_train[1])
            X_train = x_full[msk_train]
            y_train = y_full_toy[msk_train].astype(float)

            kernel = make_kernel_for_dataset(ds, mass=m0, optimized_by_year=_OPTIMIZED_LS_BY_YEAR)

            # v14.5: allow a lower-restart setting for displays (default: global N_RESTARTS)
            refit_restarts = int(globals().get("EXTRACTION_DISPLAY_GP_RESTARTS", N_RESTARTS))

            # IMPORTANT: `_fit_gpr` takes `restarts=...` (v14 bug used `n_restarts=...`)
            gpr = _fit_gpr(X_train, y_train, restarts=int(refit_restarts), kernel=kernel, optimize=bool(globals().get("INJ_REFIT_GP_OPTIMIZE", True)))

            mu_full_plot, var_full_plot = _predict_counts_mean_var_from_log_gpr(gpr, x_full)
            mu_win, cov_win = _predict_counts_from_log_gpr(gpr, x_win)

            mu_full_plot = np.asarray(mu_full_plot, float).reshape(-1)
            var_full_plot = np.asarray(var_full_plot, float).reshape(-1) if (var_full_plot is not None) else None
            refit_ok = True
        except Exception as e:
            print("[extract_display][WARN] refit_gp_on_toy failed, using original GP pred. Error:", e)
            mu_full_plot = mu_full
            var_full_plot = np.asarray(var_full_pred, float).reshape(-1) if (var_full_pred is not None) else None
            refit_ok = False

    # --- Fit A in the blind window (profiled likelihood) ---
    fitd = fit_A_profiled_gaussian_details(
        y_win_toy,
        mu_win,
        cov_win,
        w_win,
        allow_negative=bool(globals().get("INJ_EXTRACT_ALLOW_NEGATIVE", True)),
    )
    A_hat = float(fitd.get("A_hat", np.nan))
    sigma_A = float(fitd.get("sigma_A", np.nan))
    b_fit = np.asarray(fitd.get("b_fit", mu_win), float).reshape(-1)
    lam_fit = np.asarray(fitd.get("lambda_hat", mu_win), float).reshape(-1)

    pull = (A_hat - float(Ninj_total)) / sigma_A if np.isfinite(sigma_A) and sigma_A > 0 else np.nan

    # --- Plot (v12.5-style, updated) ---
    fig = plt.figure(figsize=(11.2, 10.2))
    gs = fig.add_gridspec(3, 1, height_ratios=[1.15, 1.15, 0.85], hspace=0.18)

    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[1, 0])
    ax3 = fig.add_subplot(gs[2, 0])

    # -----------------
    # Panel 1: full range context
    # -----------------
    ax1.step(x_full, y_full_toy, where="mid", label="toy data")
    ax1.plot(x_full, mu_full_plot, label="GP mean")

    # Optional 1σ band (diag) for context (off by default in the notebook config)
    if bool(globals().get("PLOT_FULL_SHOW_BKG_1SIGMA", False)) and (var_full_plot is not None):
        v = np.asarray(var_full_plot, float).reshape(-1)
        if v.size == mu_full_plot.size and np.isfinite(v).any():
            sig = np.sqrt(np.clip(v, 0.0, None))
            lo = np.clip(mu_full_plot - sig, 0.0, None)
            hi = mu_full_plot + sig
            ax1.fill_between(x_full, lo, hi, alpha=0.20, label=r"GP $\pm1\sigma$ (diag)")

    _shade_blind_window(ax1, pred.blind, alpha=0.15, color="0.90", label="blind window", zorder=0)

    # Overlay injected expectation (total-yield parameter)
    ax1.plot(
        x_full,
        mu_full_plot + float(Ninj_total) * w_full,
        linestyle=":",
        label=rf"Injected mean ($A_{{inj}}$={float(Ninj_total):.1f})",
    )

    ax1.set_ylabel("Counts / bin")
    _set_title_above(ax1, f"{ds.label} — extraction display @ {m0:.3f} GeV", pad=float(globals().get("TITLE_PAD", 12.0)))

    if bool(globals().get("FULL_PLOT_LOGY", False)):
        ax1.set_yscale("log")
    _grid_local(ax1)

    # Place legend and a separate info box (publication readability)
    ax1.legend(loc="upper left", frameon=True, fontsize=9)

    info = rf"$\sum w_{{full}}$={f_full:.3f},  $\sum w_{{win}}$={f_win:.3f}"
    info += rf"\n$\hat{{A}}$={A_hat:.2g}$\pm${sigma_A:.2g}"
    info += rf"\n$A_{{inj}}$={float(Ninj_total):.2g},  pull={pull:.2f}"
    if bool(refit_gp_on_toy):
        info += "\n" + ("GP refit on toy sidebands: yes" if refit_ok else "GP refit on toy sidebands: failed → used original")

    _add_info_box(ax1, info, loc="upper right", fontsize=9, alpha=0.92)

    # -----------------
    # Panel 2: blind window fit (zoomed, includes outside-blind bins)
    # -----------------
    zoom_half_sigma = float(globals().get("EXTRACTION_DISPLAY_ZOOM_HALF_SIGMA", 0.5))
    zlo = float(pred.blind[0] - zoom_half_sigma * pred.sigma_val)
    zhi = float(pred.blind[1] + zoom_half_sigma * pred.sigma_val)
    m_zoom = (x_full >= zlo) & (x_full <= zhi)

    xz = x_full[m_zoom]
    yz = y_full_toy[m_zoom]
    muz = mu_full_plot[m_zoom]

    yerr_z = np.sqrt(np.clip(yz, 1.0, None))
    ax2.errorbar(xz, yz, yerr=yerr_z, fmt="o", label="toy data")
    ax2.plot(xz, muz, label="GP mean")

    # Optional 1σ band (diag) in zoom region
    if bool(globals().get("PLOT_BLIND_SHOW_BKG_1SIGMA", True)) and (var_full_plot is not None):
        v = np.asarray(var_full_plot, float).reshape(-1)
        if v.size == mu_full_plot.size:
            vz = v[m_zoom]
            if vz.size == muz.size and np.isfinite(vz).any():
                sig = np.sqrt(np.clip(vz, 0.0, None))
                lo = np.clip(muz - sig, 0.0, None)
                hi = muz + sig
                ax2.fill_between(xz, lo, hi, alpha=0.20, label=r"GP $\pm1\sigma$ (diag)")

    # Profiled bkg and full fitted expectation (defined on the blind bins)
    ax2.plot(x_win, b_fit, linestyle=":", label="profiled bkg (A=Â)")
    ax2.plot(x_win, lam_fit, label=rf"profiled fit (Â={A_hat:.2g})")

    _shade_blind_window(ax2, pred.blind, alpha=0.18, color="0.90", label="blind window", zorder=0)
    ax2.set_xlim(zlo, zhi)
    ax2.set_ylabel("Counts / bin")
    _set_title_above(ax2, f"Blind window fit (±{zoom_half_sigma:.1f}σ beyond bounds)", pad=float(globals().get("TITLE_PAD", 10.0)))
    _grid_local(ax2)
    ax2.legend(loc="upper left", frameon=True, fontsize=9)

    # -----------------
    # Panel 3: extracted signal component in blind window
    # -----------------
    yerr_win = np.sqrt(np.clip(y_win_toy, 1.0, None))
    sig_data = y_win_toy - b_fit
    ax3.errorbar(x_win, sig_data, yerr=yerr_win, fmt="o", label="toy data − profiled bkg")

    ax3.plot(x_win, float(Ninj_total) * w_win, linestyle=":", label=rf"Injected mean ($A_{{inj}} w$)")
    ax3.plot(x_win, float(A_hat) * w_win, label=rf"Extracted ($\hat{{A}} w$)")
    ax3.axhline(0.0, color="k", linewidth=0.8, alpha=0.5)

    ax3.set_xlabel("m (GeV)")
    ax3.set_ylabel("Signal counts / bin")
    _set_title_above(ax3, "Extracted signal component (blind bins)", pad=float(globals().get("TITLE_PAD", 10.0)))
    _grid_local(ax3)
    ax3.legend(loc="best", frameon=True, fontsize=9)

    fig.tight_layout()
    ensure_dir(os.path.dirname(outpath) or ".")
    fig.savefig(outpath, dpi=int(globals().get("SAVEFIG_DPI", 300)))
    plt.close(fig)

    print(f"[extract_display] wrote {outpath}")


# Backwards-compatible alias (v14 name)
make_extraction_display_v14 = make_extraction_display_v15


### RUN: Extraction display plots

This section produces a small set of reviewer-facing plots (one per mass hypothesis / injection strength) using `make_extraction_display_v15`.


In [ ]:
# Configure which dataset(s) to run:
EXTRACTION_DISPLAY_DATASETS = getattr(globals(), "EXTRACTION_DISPLAY_DATASETS", None)  # None -> all enabled

outdir = os.path.join(OUTPUT_DIR, "extraction_display")
ensure_dir(outdir)

def _stable_seed(tag: str, base: int = 0) -> int:
    import hashlib
    h = hashlib.md5(tag.encode("utf-8")).hexdigest()[:8]
    return int((base + int(h, 16)) % (2**31 - 1))

for ds in DATASETS.values():
    if not ds.enabled:
        continue
    if (EXTRACTION_DISPLAY_DATASETS is not None) and (ds.key not in EXTRACTION_DISPLAY_DATASETS):
        continue

    ds_out = os.path.join(outdir, ds.key)
    ensure_dir(ds_out)

    for m0 in EXTRACTION_DISPLAY_MASS_POINTS:
        m0 = float(m0)

        # Determine which injection strengths to use
        strengths = list(EXTRACTION_DISPLAY_STRENGTHS)
        if str(EXTRACTION_DISPLAY_STRENGTH_MODE).lower().strip() == "sigmaa":
            # Compute σA under background-only (Asimov) for this point
            pred0 = estimate_background_for_dataset(ds, m0)
            tmpl0 = build_template(pred0.edges, m0, pred0.sigma_val, norm=None)
            d0 = fit_A_profiled_gaussian_details(pred0.mu, pred0.mu, pred0.cov, tmpl0, allow_negative=True)
            sigmaA0 = float(d0.get("sigma_A", np.nan))
            strengths = [float(k) * sigmaA0 for k in EXTRACTION_DISPLAY_SIGMA_MULTIPLIERS]

        for Ninj in strengths:
            for refit in EXTRACTION_DISPLAY_REFIT_GP:
                seed = _stable_seed(f"exdisp:{ds.key}:{m0:.6f}:{float(Ninj):.6f}:{bool(refit)}", base=int(EXTRACTION_DISPLAY_SEED))

                tag = f"m{m0:.3f}_Ainj{float(Ninj):.2f}" + ("_refit" if refit else "")
                outpath = os.path.join(ds_out, f"extract_display_{tag}.png")

                make_extraction_display_v15(
                    ds, m0,
                    Ninj_total=float(Ninj),
                    seed=seed,
                    inj_mode=str(INJ_MODE),
                    refit_gp_on_toy=bool(refit),
                    outpath=outpath,
                )


In [ ]:

# =============================================================================
# Cell 20.5 — Injected-signal ensembles (v10)
# =============================================================================
#
# Purpose:
#   Optionally inject a Gaussian signal of a known yield into *each* functional-form toy histogram
#   before running the LS upper-bound optimization and/or the full ensemble pull scans.
#
# Controls:
#   - injected_signal_ensembles (bool): master switch
#   - ensemble_injection (float): injected yield (events)
#   - injected_signal_center (float): injected Gaussian center (GeV)
#
# Notes:
#   - The injected Gaussian width defaults to σ(m0) for the dataset year (mass resolution).
#   - By default we inject *integer bin counts* with a fixed total yield using a multinomial draw.
#   - v10 adds explicit diagnostics (no silent failure): the injection helper returns metadata
#     including the realized injected total and the Gaussian "in-range fraction" for your histogram edges.

injected_signal_ensembles = False
ensemble_injection = 10000.0
injected_signal_center = 0.120

# Optional knobs (advanced)
INJECT_SIGNAL_FLUCTUATE = True       # if True, use multinomial draw (exact total N, integer per bin)
INJECT_SIGNAL_SEED = 12345           # base RNG seed (used only if fluctuate=True)
INJECT_SIGNAL_SIGMA_SCALE = 1.0      # sigma_inj = scale * σ(m0)
INJECT_SIGNAL_VERBOSE = False        # if True, print injection info for the first few toys per job

def inject_gaussian_signal_into_hist(
    edges: np.ndarray,
    counts: np.ndarray,
    *,
    center: float,
    sigma: float,
    n_events: float,
    fluctuate: bool = False,
    seed: int = 12345,
    return_add: bool = False,
) -> Tuple[np.ndarray, Dict[str, object]]:
    """Return (new_counts, info) where new_counts = counts + injected signal.

    The signal shape is a binned Gaussian computed by integrating the PDF over each bin,
    then normalizing the weights to sum to 1 over the histogram edges.

    info keys (v10):
      ok: bool
      injected: bool
      mode: "multinomial"|"deterministic"
      n_events: requested injected yield (rounded for multinomial)
      n_added: realized sum of injected counts added to histogram
      gauss_fraction_in_hist: fraction of the Gaussian within [edges[0], edges[-1]]
      weights_sum/min/max: diagnostics on the normalized weight vector
    """
    edges = np.asarray(edges, float).reshape(-1)
    counts = np.asarray(counts, float).reshape(-1)
    if edges.size != counts.size + 1:
        raise ValueError("edges must have length counts+1")

    center = float(center)
    sigma = float(max(0.0, sigma))
    n_events = float(n_events)

    if not (np.isfinite(center) and np.isfinite(sigma) and np.isfinite(n_events)):
        return counts, dict(ok=False, injected=False, reason="non-finite center/sigma/n_events")

    if n_events == 0.0:
        return counts, dict(ok=True, injected=False, n_events=0.0, n_added=0.0)

    if center < float(edges[0]) or center > float(edges[-1]) or sigma <= 0.0:
        return counts, dict(ok=False, injected=False,
                            reason="center outside hist range or sigma<=0",
                            center=center, sigma=sigma, edges_lo=float(edges[0]), edges_hi=float(edges[-1]))

    w_raw = gaussian_bin_integrals(edges, center, sigma)
    frac_in_hist = float(np.sum(w_raw))   # fraction of the Gaussian inside the histogram edges
    if (not np.isfinite(frac_in_hist)) or (frac_in_hist <= 0.0):
        return counts, dict(ok=False, injected=False, reason="template integral is zero",
                            center=center, sigma=sigma)

    w = (w_raw / frac_in_hist).astype(float)

    info_common = dict(
        ok=True,
        injected=True,
        center=float(center),
        sigma=float(sigma),
        gauss_fraction_in_hist=float(frac_in_hist),
        weights_sum=float(np.sum(w)),
        weights_min=float(np.min(w)),
        weights_max=float(np.max(w)),
    )

    if fluctuate:
        rng = np.random.default_rng(int(seed) % (2**32 - 1))
        n_int = int(round(n_events))
        n_int = max(n_int, 0)
        add = rng.multinomial(n_int, w).astype(float)
        info = dict(info_common, mode="multinomial", n_events=float(n_int), n_added=float(np.sum(add)))
        if return_add:
            info["add"] = add
        return counts + add, info
    else:
        add = (n_events * w).astype(float)
        info = dict(info_common, mode="deterministic", n_events=float(n_events), n_added=float(np.sum(add)))
        if return_add:
            info["add"] = add
        return counts + add, info


## v10: quick injected-signal sensitivity scan vs GP length-scale (functional-form toys)

The LS-optimization in the main functional-form section primarily targets **background-only calibration** (e.g. low coherent bias in the median pull vs mass).  
To decide which kernel length-scale bounds are most appropriate for **signal sensitivity**, it's useful to run a *targeted* scan:

- Pick one injected mass (e.g. 60 MeV) and a grid of injected yields (e.g. 0, 5k, 10k).
- For each candidate **LS upper-factor**, inject signal into a subset of toys and evaluate \(Z\approx \hat A/\sigma_A\) at that mass hypothesis.
- Compare how quickly \(Z\) grows with injected yield *and* whether the background-only point stays well-calibrated.

This cell is designed to be lighter-weight than a full mass-scan over all hypotheses.


In [ ]:

# =============================================================================
# Cell 20.6 — v10: Functional-form toy injected-signal sensitivity scan
# =============================================================================
#
# This is a *targeted* sensitivity study:
#   For each LS upper-factor, inject signals at one mass m_inj into a subset of functional-form toys,
#   then extract at m0=m_inj and summarize Z = A_hat / sigma_A.
#
# Why this exists:
#   The standard functional-form LS optimization is tuned on background-only pull-calibration.
#   This scan lets you directly compare *sensitivity* to injected signals across LS choices.

FUNC_INJ_SENSITIVITY = True   # set True to run
FUNC_INJ_SENS_YEAR = "2021"    # choose the toy job year to test
FUNC_INJ_SENS_TAG  = "fSigPowExpQ"  # choose which FUNCFORM_JOBS tag to use (must be enabled or present) #"fSigPow"

FUNC_INJ_MASS_GEV = 0.120
FUNC_INJ_STRENGTHS = [0, 5000, 10000]   # total injected events into the toy histogram
FUNC_INJ_TOY_INDICES = list(range(0, 10))  # keep small for quick scans; increase for stability
FUNC_INJ_LS_UPPER_FACTORS = [2.0, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0]

# Extraction settings (reuse your main toy settings by default)
FUNC_INJ_REBIN = FUNCFORM_NEIGHBORHOOD_REBIN
FUNC_INJ_RESTARTS = FUNCFORM_N_RESTARTS
FUNC_INJ_ALLOW_NEG = FUNCFORM_EXTRACT_ALLOW_NEGATIVE

# Optional: evaluate at multiple mass hypotheses to check leakage
FUNC_INJ_EVAL_MASSES = [FUNC_INJ_MASS_GEV]   # e.g. [0.058, 0.060, 0.062]

# Output (under OUTPUT_DIR)
FUNC_INJ_OUTDIR = os.path.join(OUTPUT_DIR, "funcform_injection_sensitivity_v11")

# -----------------------------
# Helpers (standalone; does not require running the full Cell 21 scan)
# -----------------------------
def _load_funcform_toy_hist(root_path: str, container: str, toy_name: str) -> Tuple[np.ndarray, np.ndarray]:
    import uproot
    with uproot.open(root_path) as f:
        obj = f
        if container:
            obj = obj[container]
        h = obj[toy_name]
        vals, edges = h.to_numpy()
    return np.asarray(edges, float), np.asarray(vals, float)

def _rebin_edges_counts(edges: np.ndarray, counts: np.ndarray, factor: int) -> Tuple[np.ndarray, np.ndarray]:
    edges = np.asarray(edges, float).reshape(-1)
    counts = np.asarray(counts, float).reshape(-1)
    factor = int(max(1, factor))
    if factor == 1:
        return edges, counts
    n = counts.size
    rem = n % factor
    if rem == 0:
        c2 = counts.reshape(-1, factor).sum(axis=1)
        e2 = edges[::factor]
        if e2.size != c2.size + 1:
            e2 = np.concatenate([e2, [edges[-1]]])
        return e2, c2
    # keep a final partial bin
    main = n - rem
    c_main = counts[:main].reshape(-1, factor).sum(axis=1)
    c_last = np.array([counts[main:].sum()], float)
    c2 = np.concatenate([c_main, c_last])
    e2 = np.concatenate([edges[:main+1:factor], [edges[-1]]])
    return e2, c2

def _extract_pull_one_mass(edges_in: np.ndarray, counts_in: np.ndarray, *,
                           ds_year: DatasetConfig,
                           mass: float,
                           rebin: int,
                           restarts: int,
                           allow_negative: bool,
                           kernel) -> Dict[str, float]:
    # Rebin
    edges, counts = _rebin_edges_counts(edges_in, counts_in, rebin)

    centers = 0.5 * (edges[:-1] + edges[1:])
    y = counts.astype(float)

    # Restrict to a reasonable training domain
    lo_req = float(getattr(ds_year, "data_low", float(ds_year.m_low))) if getattr(ds_year, "data_low", None) is not None else float(ds_year.m_low)
    hi_req = float(getattr(ds_year, "data_high", float(ds_year.m_high))) if getattr(ds_year, "data_high", None) is not None else float(ds_year.m_high)

    lo = max(float(edges[0]), lo_req)
    hi = min(float(edges[-1]), hi_req)
    msk_dom = (centers >= lo) & (centers <= hi)
    if not np.any(msk_dom):
        return dict(ok=False, pull=np.nan, A_hat=np.nan, sigma_A=np.nan, blind_lo=np.nan, blind_hi=np.nan, n_blind=0)

    X_dom = centers[msk_dom]
    y_dom = y[msk_dom]
    edges_dom = edges[np.where(msk_dom)[0][0] : np.where(msk_dom)[0][-1] + 2]

    sigma_m = float(ds_year.sigma(float(mass)))
    blind_lo = float(mass) - float(BLIND_NSIGMA) * sigma_m
    blind_hi = float(mass) + float(BLIND_NSIGMA) * sigma_m

    msk_blind = (X_dom >= blind_lo) & (X_dom <= blind_hi)
    if np.count_nonzero(msk_blind) < 3:
        return dict(ok=False, pull=np.nan, A_hat=np.nan, sigma_A=np.nan, blind_lo=blind_lo, blind_hi=blind_hi, n_blind=int(np.count_nonzero(msk_blind)))

    X_train = X_dom[~msk_blind]
    y_train = y_dom[~msk_blind]
    if X_train.size < 8:
        return dict(ok=False, pull=np.nan, A_hat=np.nan, sigma_A=np.nan, blind_lo=blind_lo, blind_hi=blind_hi, n_blind=int(np.count_nonzero(msk_blind)))

    gpr = _fit_gpr(X_train, y_train, restarts=int(restarts), kernel=kernel)
    mu, cov = _predict_counts_from_log_gpr(gpr, X_dom[msk_blind])

    obs = y_dom[msk_blind]
    # Build blind edges
    idx0 = int(np.where(msk_dom)[0][0])
    idx_bl = np.where(msk_dom)[0][0] + np.where(msk_blind)[0]
    e_blind = edges[idx_bl[0]: idx_bl[-1] + 2]

    tmpl = build_template(e_blind, float(mass), sigma_m)
    fit = fit_A_profiled_gaussian(obs, mu, cov, tmpl, allow_negative=bool(allow_negative))
    Ahat = float(fit["A_hat"])
    sA = float(fit["sigma_A"])
    pull = float(Ahat / sA) if (np.isfinite(Ahat) and np.isfinite(sA) and sA > 0) else np.nan
    return dict(ok=True, pull=pull, A_hat=Ahat, sigma_A=sA, blind_lo=blind_lo, blind_hi=blind_hi, n_blind=int(np.count_nonzero(msk_blind)))

# -----------------------------
# Main driver
# -----------------------------
if FUNC_INJ_SENSITIVITY:
    ensure_dir(FUNC_INJ_OUTDIR)

    # Find the requested job
    jobs = list(globals().get("FUNCFORM_JOBS", []))
    job = None
    for j in jobs:
        if str(j.get("year")) == str(FUNC_INJ_SENS_YEAR) and str(j.get("tag")) == str(FUNC_INJ_SENS_TAG):
            job = j
            break
    if job is None:
        raise RuntimeError(f"Could not find FUNCFORM_JOBS entry for year={FUNC_INJ_SENS_YEAR} tag={FUNC_INJ_SENS_TAG}")

    ds_year = DATASETS[str(FUNC_INJ_SENS_YEAR)]
    root_path = str(job["root_path"])
    container = str(job.get("container", ""))
    toy_fmt = str(job.get("toy_name_fmt", "toy_{i}"))

    m_inj = float(FUNC_INJ_MASS_GEV)
    sigma_inj = float(globals().get("INJECT_SIGNAL_SIGMA_SCALE", 1.0)) * float(ds_year.sigma(m_inj))
    fluctuate = bool(globals().get("INJECT_SIGNAL_FLUCTUATE", True))
    base_seed = int(globals().get("INJECT_SIGNAL_SEED", 12345))

    records = []
    for uf in FUNC_INJ_LS_UPPER_FACTORS:
        kernel = make_kernel_for_dataset(
            ds_year,
            policy="resolution_scaled",
            upper_factor=float(uf),
            lower_factor=float(globals().get("FUNCFORM_LS_LOWER_FACTOR", globals().get("KERNEL_LS_RES_LOWER_FACTOR", 0.05))),
            stat=str(globals().get("FUNCFORM_LS_STAT", globals().get("KERNEL_LS_RES_STAT", "median"))),
            npts=int(globals().get("FUNCFORM_LS_NPTS", globals().get("KERNEL_LS_RES_NPTS", 200))),
        )

        for N in FUNC_INJ_STRENGTHS:
            for toy_idx in FUNC_INJ_TOY_INDICES:
                toy_name = toy_fmt.format(i=int(toy_idx))
                edges, counts = _load_funcform_toy_hist(root_path, container, toy_name)

                # deterministic seed (so different LS factors reuse identical injected toys)
                seed = (base_seed
                        + int(toy_idx) * 1000003
                        + int(round(m_inj * 1e6)) * 9176
                        + int(round(float(N) * 10.0)) * 131) % (2**32 - 1)

                counts_inj, inj_info = inject_gaussian_signal_into_hist(
                    edges, counts,
                    center=m_inj,
                    sigma=sigma_inj,
                    n_events=float(N),
                    fluctuate=fluctuate,
                    seed=int(seed),
                    return_add=False,
                )

                for m_eval in FUNC_INJ_EVAL_MASSES:
                    out = _extract_pull_one_mass(
                        edges, counts_inj,
                        ds_year=ds_year,
                        mass=float(m_eval),
                        rebin=int(FUNC_INJ_REBIN),
                        restarts=int(FUNC_INJ_RESTARTS),
                        allow_negative=bool(FUNC_INJ_ALLOW_NEG),
                        kernel=kernel,
                    )
                    records.append(dict(
                        year=str(FUNC_INJ_SENS_YEAR),
                        tag=str(FUNC_INJ_SENS_TAG),
                        ls_upper_factor=float(uf),
                        toy=int(toy_idx),
                        injected_N=float(N),
                        injected_center_GeV=float(m_inj),
                        injected_sigma_GeV=float(sigma_inj),
                        inj_ok=bool(inj_info.get("ok", False)),
                        inj_n_added=float(inj_info.get("n_added", np.nan)),
                        eval_mass_GeV=float(m_eval),
                        pull=float(out.get("pull", np.nan)),
                        A_hat=float(out.get("A_hat", np.nan)),
                        sigma_A=float(out.get("sigma_A", np.nan)),
                        extract_ok=bool(out.get("ok", False)),
                    ))

    df = pd.DataFrame(records)
    out_csv = os.path.join(FUNC_INJ_OUTDIR, "funcform_injection_sensitivity_raw.csv")
    df.to_csv(out_csv, index=False)
    print(f"[funcform][inj-sens] wrote {out_csv}")

    # Summary: median pull (Z) vs Ninj for each LS upper-factor (at each eval mass)
    g = df.groupby(["ls_upper_factor", "eval_mass_GeV", "injected_N"])["pull"]
    summ = pd.DataFrame({
        "ls_upper_factor": g.mean().index.get_level_values(0),
        "eval_mass_GeV": g.mean().index.get_level_values(1),
        "injected_N": g.mean().index.get_level_values(2),
        "pull_median": g.median().values,
        "pull_p16": g.quantile(0.16).values,
        "pull_p84": g.quantile(0.84).values,
        "n": g.count().values,
    }).sort_values(["eval_mass_GeV", "ls_upper_factor", "injected_N"]).reset_index(drop=True)

    out_sum = os.path.join(FUNC_INJ_OUTDIR, "funcform_injection_sensitivity_summary.csv")
    summ.to_csv(out_sum, index=False)
    print(f"[funcform][inj-sens] wrote {out_sum}")

    # Plot: for each eval_mass, overlay LS factors
    try:
        for m_eval, g0 in summ.groupby("eval_mass_GeV", sort=True):
            plt.figure(figsize=(9,5))
            for uf, g1 in g0.groupby("ls_upper_factor", sort=True):
                g1 = g1.sort_values("injected_N")
                x = g1["injected_N"].to_numpy(float)
                y = g1["pull_median"].to_numpy(float)
                yerr = _asym_yerr(y, g1["pull_p16"].to_numpy(float), g1["pull_p84"].to_numpy(float))
                plt.errorbar(x, y, yerr=yerr, capsize=3, marker="o", linestyle="-", label=f"UF={uf:g}")
            plt.axhline(0.0, linestyle="--")
            plt.xlabel("Injected N (events)")
            plt.ylabel("Median pull Z ≈ A_hat / sigma_A")
            plt.title(f"Functional-form toy sensitivity at m0={m_eval:.3f} GeV")
            plt.grid(True, alpha=0.3)
            plt.legend()
            out_plot = os.path.join(FUNC_INJ_OUTDIR, f"inj_sensitivity_m{m_eval:.3f}.png".replace(".","p"))
            plt.savefig(out_plot, dpi=int(globals().get("SAVEFIG_DPI", 300)), bbox_inches="tight")
            plt.show()
            print(f"[funcform][inj-sens] wrote {out_plot}")
    except Exception as e:
        print(f"[funcform][inj-sens][WARN] plot failed: {e}")



In [ ]:
# =============================================================================
# Cell 21 — Functional-form toy analyses (v8 bounded length-scale + optional optimization)
# =============================================================================
#
# v8 additions:
#   - Bound the GP RBF length scale in log(m) using σ(m) (mass resolution) to suppress long-wavelength biases.
#   - Optional scan over multiple "upper-factor" values (e.g. 2×,3×,5× σ_log) to optimize bias/coverage on pseudo-data.
#   - Save the chosen length-scale bounds to KERNEL_LS_OPT_FILE for later reuse in the real-data analysis.
#
# This cell is gated by `functional_form_toy_analyses = True` (set in Inputs).

if bool(globals().get("functional_form_toy_analyses", False)):

    import os
    import json
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    from typing import Dict, List, Optional, Tuple
    from datetime import datetime, timezone

    # -----------------------------
    # Pull-analysis configuration
    # -----------------------------
    _jobs = [j for j in globals().get("FUNCFORM_JOBS", []) if bool(j.get("enabled", True))]
    if len(_jobs) == 0:
        raise RuntimeError("functional_form_toy_analyses=True, but FUNCFORM_JOBS has no enabled entries.")

    _n_toys_expected = int(globals().get("FUNCFORM_N_TOYS", 100))
    _toy_i0, _toy_i1 = globals().get("FUNCFORM_TOY_INDEX_RANGE", (0, _n_toys_expected - 1))
    _toy_i0 = int(_toy_i0); _toy_i1 = int(_toy_i1)
    if _toy_i1 < _toy_i0:
        raise ValueError("FUNCFORM_TOY_INDEX_RANGE must be (start,end) with end>=start")

    _mass_step = float(globals().get("FUNCFORM_MASS_STEP_GEV", MASS_STEP_GEV))
    _rebin = int(globals().get("FUNCFORM_NEIGHBORHOOD_REBIN", NEIGHBORHOOD_REBIN))
    _restarts = int(globals().get("FUNCFORM_N_RESTARTS", N_RESTARTS))
    _allow_neg = bool(globals().get("FUNCFORM_EXTRACT_ALLOW_NEGATIVE", True))

    # v8: length-scale bounds / optimization settings for toys
    _ls_policy = str(globals().get("FUNCFORM_LS_POLICY", globals().get("KERNEL_LS_POLICY", "manual")))
    _ls_upper_factor_default = float(globals().get("FUNCFORM_LS_UPPER_FACTOR", globals().get("KERNEL_LS_RES_UPPER_FACTOR", 5.0)))
    _ls_lower_factor = float(globals().get("FUNCFORM_LS_LOWER_FACTOR", globals().get("KERNEL_LS_RES_LOWER_FACTOR", 0.05)))
    _ls_stat = str(globals().get("FUNCFORM_LS_STAT", globals().get("KERNEL_LS_RES_STAT", "median")))
    _ls_npts = int(globals().get("FUNCFORM_LS_NPTS", globals().get("KERNEL_LS_RES_NPTS", 200)))

    _ls_optimize = bool(globals().get("FUNCFORM_LS_OPTIMIZE", False))
    _ls_grid = list(globals().get("FUNCFORM_LS_UPPER_FACTOR_GRID", [_ls_upper_factor_default]))
    _ls_grid = [float(x) for x in _ls_grid]
    _ls_opt_n_toys = int(globals().get("FUNCFORM_LS_OPTIMIZE_N_TOYS", 30))
    _ls_run_full_after = bool(globals().get("FUNCFORM_LS_RUN_FULL_AFTER_OPTIMIZE", True))
    _ls_metric = str(globals().get("FUNCFORM_LS_OPTIMIZE_METRIC", "rms_median")).lower().strip()
    _ls_save_opt = bool(globals().get("FUNCFORM_LS_SAVE_OPTIMIZED", True))

    # Parallelization (per job)
    _n_workers = int(globals().get("FUNCFORM_N_WORKERS", 1))
    _joblib_backend = str(globals().get("FUNCFORM_JOBLIB_BACKEND", "loky"))
    if _n_workers < 1:
        _n_workers = 1

    # We prefer joblib in notebooks (robust pickling via cloudpickle / loky).
    try:
        from joblib import Parallel, delayed
        _have_joblib = True
    except Exception:
        Parallel = None
        delayed = None
        _have_joblib = False

    if _n_workers > 1 and not _have_joblib:
        print("[funcform][INFO] FUNCFORM_N_WORKERS>1 but joblib is not available; falling back to sequential mode.")
        _n_workers = 1

    # Where to put outputs
    _out_root = OUTPUT_DIR  # keep everything under the existing OUTPUT_DIR

    # -----------------------------
    # v8: load optimized length-scale bounds for reuse (optional)
    # -----------------------------
    _OPTIMIZED_LS_BY_YEAR: Dict[str, object] = {}
    _opt_file_path = globals().get("KERNEL_LS_OPT_FILE", None)
    if bool(globals().get("LOAD_OPTIMIZED_LS_IF_AVAILABLE", False)) and _opt_file_path:
        try:
            _p = str(_opt_file_path)
            if os.path.exists(_p):
                with open(_p, "r") as _f:
                    _cfg = json.load(_f)
                _OPTIMIZED_LS_BY_YEAR = dict(_cfg.get("by_year", _cfg.get("results", {})))
                if bool(globals().get("DEBUG_PRINT", False)):
                    print(f"[funcform][kernel] loaded optimized LS config from {_p}: keys={list(_OPTIMIZED_LS_BY_YEAR.keys())}")
        except Exception:
            _OPTIMIZED_LS_BY_YEAR = {}

    # We'll update/write the optimized file at the end if requested.
    _opt_updates: Dict[str, object] = {}

    # -----------------------------
    # Helper: build a DatasetConfig carrying the right σ(m), f_rad(m), and scan ranges
    # -----------------------------
    def _funcform_ds_for_year(year: str) -> DatasetConfig:
        year = str(year)
        if year == "2015":
            return DatasetConfig(
                key="2015", label="HPS 2015 (funcform toys)", root_path="__TOY__", hist_name="__TOY__",
                m_low=float(RANGE_2015[0]), m_high=float(RANGE_2015[1]),
                sigma_coeffs=list(SIGMA_COEFFS_2015), frad_coeffs=list(FRAD_COEFFS_2015),
                data_low=DATA_RANGE_2015[0], data_high=DATA_RANGE_2015[1],
                enabled=True,
            )
        if year == "2016":
            return DatasetConfig(
                key="2016", label="HPS 2016 (funcform toys)", root_path="__TOY__", hist_name="__TOY__",
                m_low=float(RANGE_2016[0]), m_high=float(RANGE_2016[1]),
                sigma_coeffs=list(SIGMA_COEFFS_2016), frad_coeffs=list(FRAD_COEFFS_2016),
                data_low=DATA_RANGE_2016[0], data_high=DATA_RANGE_2016[1],
                enabled=True,
                sigma_tail_m0=0.18,
                sigma_tail_slope_floor=0.0,
                sigma_tail_slope_override=0.0239,
            )
        if year == "2021":
            return DatasetConfig(
                key="2021", label="HPS 2021 (funcform toys)", root_path="__TOY__", hist_name="__TOY__",
                m_low=float(RANGE_2021[0]), m_high=float(RANGE_2021[1]),
                sigma_coeffs=list(SIGMA_COEFFS_2021), frad_coeffs=list(FRAD_COEFFS_2021),
                data_low=DATA_RANGE_2021[0], data_high=DATA_RANGE_2021[1],
                enabled=True,
            )
        raise ValueError(f"Unknown year '{year}' for funcform toys. Add it to _funcform_ds_for_year().")

    # -----------------------------
    # Helper: limit a histogram to [low,high] and optionally rebin by an integer factor
    # -----------------------------
    def _limit_and_rebin(edges: np.ndarray, counts: np.ndarray,
                         low: Optional[float], high: Optional[float],
                         rebin: int) -> Tuple[np.ndarray, np.ndarray]:
        edges = np.asarray(edges, float).reshape(-1)
        counts = np.asarray(counts, float).reshape(-1)

        if edges.size != counts.size + 1:
            raise ValueError(f"edges.size={edges.size} must equal counts.size+1={counts.size+1}")

        lo = float(edges[0]) if low is None else float(low)
        hi = float(edges[-1]) if high is None else float(high)

        # Clip to histogram bounds
        lo = max(lo, float(edges[0]))
        hi = min(hi, float(edges[-1]))
        if hi <= lo:
            raise ValueError(f"Invalid limit range: [{lo},{hi}] with histogram [{edges[0]},{edges[-1]}]")

        # Select bins whose centers are within [lo,hi]
        centers = 0.5 * (edges[:-1] + edges[1:])
        m = (centers >= lo) & (centers <= hi)
        idx = np.where(m)[0]
        if idx.size == 0:
            raise RuntimeError("No bins left after applying limit range.")

        i0 = int(idx[0]); i1 = int(idx[-1]) + 1
        edges2 = edges[i0 : i1 + 1]
        counts2 = counts[i0 : i1]

        # Rebin: sum adjacent bins in groups of `rebin`
        r = int(max(1, rebin))
        if r > 1:
            n = counts2.size
            n2 = (n // r) * r
            if n2 < r:
                return edges2, counts2
            counts2 = counts2[:n2].reshape(-1, r).sum(axis=1)
            edges2 = edges2[: (n2 + 1) : r]
            # Ensure edges length is consistent
            if edges2.size != counts2.size + 1:
                edges2 = np.concatenate([edges2, [edges2[-1] + (edges2[-1] - edges2[-2])]])
        return edges2, counts2

    # -----------------------------
    # Helper: load a single toy histogram (edges, counts) from a ROOT file
    # -----------------------------
    def _load_one_toy_hist(root_path: str, container: Optional[str], toy_name: str) -> Tuple[np.ndarray, np.ndarray]:
        """Return (edges, counts) for a toy histogram.

        Preferred layout (supported):
          - TDirectory "container" containing TH1 named "toy_name"
          - Top-level TH1 named "toy_name"

        Heuristic layout (supported if metadata branches exist):
          - TTree "container" where "toy_name" is a branch of bin counts and
            either:
              * an "edges"/"bin_edges" branch exists (array-like), or
              * a "mass"/"x"/"centers" branch exists (bin centers), from which edges are inferred.
        """
        root_path = str(root_path)
        toy_name = str(toy_name)

        # --- Try uproot first (fast, no ROOT runtime needed) ---
        try:
            import uproot
            f = uproot.open(root_path)

            def _try_get(key: str):
                try:
                    return f[key]
                except Exception:
                    return None

            # Direct histogram tries
            cand = []
            if container is not None:
                cand += [f"{container}/{toy_name}", f"{container}:{toy_name}"]
            cand += [toy_name]

            for k in cand:
                obj = _try_get(k)
                if obj is None:
                    continue
                if hasattr(obj, "to_numpy"):
                    vals, eds = obj.to_numpy()
                    return np.asarray(eds, float), np.asarray(vals, float)

            # Container object (directory or tree)
            if container is not None:
                cont = _try_get(container)
                if cont is not None:
                    # Directory-like?
                    try:
                        obj = cont[toy_name]
                        if hasattr(obj, "to_numpy"):
                            vals, eds = obj.to_numpy()
                            return np.asarray(eds, float), np.asarray(vals, float)
                    except Exception:
                        pass

                    # Tree-like?
                    try:
                        keys = set(cont.keys())
                        if toy_name in keys:
                            arr = cont[toy_name].array(library="np")
                            edges = None
                            for ekey in ["edges", "bin_edges", "xedges", "mass_edges"]:
                                if ekey in keys:
                                    edges = cont[ekey].array(library="np")
                                    break
                            centers = None
                            if edges is None:
                                for ckey in ["mass", "x", "centers", "bin_centers"]:
                                    if ckey in keys:
                                        centers = cont[ckey].array(library="np")
                                        break

                            counts = np.asarray(arr, float).reshape(-1)
                            if edges is not None:
                                edges = np.asarray(edges, float).reshape(-1)
                                return edges, counts

                            if centers is not None:
                                centers = np.asarray(centers, float).reshape(-1)
                                # infer edges by midpoints (assumes approximately uniform binning)
                                if centers.size < 2:
                                    raise RuntimeError("Cannot infer edges from <2 centers")
                                d = np.diff(centers)
                                # use median spacing
                                dx = float(np.median(d))
                                edges = np.concatenate([[centers[0] - 0.5 * dx], centers + 0.5 * dx])
                                return np.asarray(edges, float), counts
                    except Exception:
                        pass
        except Exception:
            pass

        # --- Fallback: PyROOT (slower) ---
        try:
            import ROOT  # type: ignore
            tf = ROOT.TFile.Open(root_path, "READ")
            if tf is None or tf.IsZombie():
                raise RuntimeError(f"Could not open ROOT file: {root_path}")

            obj = None
            if container is not None:
                d = tf.Get(container)
                if d:
                    obj = d.Get(toy_name) if hasattr(d, "Get") else None
            if obj is None:
                obj = tf.Get(toy_name)

            if obj and hasattr(obj, "GetNbinsX"):
                nb = int(obj.GetNbinsX())
                edges = np.array([obj.GetXaxis().GetBinLowEdge(1 + i) for i in range(nb + 1)], float)
                vals = np.array([obj.GetBinContent(1 + i) for i in range(nb)], float)
                tf.Close()
                return edges, vals

            tf.Close()
        except Exception:
            pass

        raise RuntimeError(f"Could not load toy histogram '{toy_name}' (container={container}) from {root_path}")

    # -----------------------------
    # Single-mass pull on a toy dataset (same extraction as main analysis)
    # -----------------------------
    def _toy_pull_at_mass(ds_year: DatasetConfig,
                          edges_in: np.ndarray,
                          counts_in: np.ndarray,
                          mass: float,
                          rebin: int,
                          restarts: int,
                          allow_negative: bool,
                          kernel) -> Dict[str, float]:

        sigma_val = float(ds_year.sigma(float(mass)))
        blind = (float(mass - BLIND_NSIGMA * sigma_val), float(mass + BLIND_NSIGMA * sigma_val))

        # Determine histogram range presented to the GP (mirror the logic in _build_model)
        first_edge = float(edges_in[0]); last_edge = float(edges_in[-1])
        req_lo = first_edge if ds_year.data_low is None else float(ds_year.data_low)
        req_hi = last_edge  if ds_year.data_high is None else float(ds_year.data_high)

        lower = max(first_edge, req_lo)
        upper = min(last_edge,  req_hi)

        # Auto-expand to include scan window if configured too narrow
        lower = max(first_edge, min(lower, float(ds_year.m_low)))
        upper = min(last_edge,  max(upper, float(ds_year.m_high)))

        edges, counts = _limit_and_rebin(edges_in, counts_in, lower, upper, rebin=rebin)

        centers = 0.5 * (edges[:-1] + edges[1:])
        y = counts.astype(float)

        # Blind-mask for training
        msk_blind = (centers >= blind[0]) & (centers <= blind[1])
        idx = np.where(msk_blind)[0]
        if idx.size == 0:
            return dict(
                ok=False, A_hat=np.nan, sigma_A=np.nan, pull=np.nan,
                blind_lo=blind[0], blind_hi=blind[1], n_blind=0
            )

        X_train = centers[~msk_blind]
        y_train = y[~msk_blind]

        # Fit GP in the same way as the main analysis (but with bounded length-scale)
        gpr = _fit_gpr(X_train, y_train, restarts=int(restarts), kernel=kernel)

        # Predict in blind
        X_blind = centers[msk_blind]
        mu, cov = _predict_counts_from_log_gpr(gpr, X_blind)
        obs = y[msk_blind]

        edges_blind = edges[int(idx[0]) : int(idx[-1]) + 2]

        tmpl = build_template(edges_blind, float(mass), float(sigma_val))

        fit = fit_A_profiled_gaussian(
            n_obs=np.asarray(obs, float),
            b_mean=np.asarray(mu, float),
            b_cov=np.asarray(cov, float),
            template=np.asarray(tmpl, float),
            allow_negative=bool(allow_negative),
        )

        A_hat = float(fit.get("A_hat", np.nan))
        sA = float(fit.get("sigma_A", np.nan))
        ok = bool(fit.get("success", False)) and np.isfinite(A_hat) and np.isfinite(sA) and (sA > 0)
        pull = float(A_hat / sA) if (np.isfinite(A_hat) and np.isfinite(sA) and sA > 0) else np.nan

        return dict(
            ok=bool(ok),
            A_hat=float(A_hat),
            sigma_A=float(sA),
            pull=float(pull),
            blind_lo=float(blind[0]),
            blind_hi=float(blind[1]),
            n_blind=int(idx.size),
        )

    # -----------------------------
    # Aggregation + plots
    # -----------------------------
    def _summarize_pulls(df_all: pd.DataFrame, outdir: str, *, title_prefix: str) -> Tuple[pd.DataFrame, np.ndarray]:
        """Write ensemble summary CSV + plots; return (summary_df, all_pulls_1d)."""
        os.makedirs(outdir, exist_ok=True)

        # Aggregate per mass
        g = df_all.groupby("mass_GeV")["pull"]
        summ = pd.DataFrame({
            "mass_GeV": g.mean().index.values,
            "pull_mean": g.mean().values,
            "pull_median": g.median().values,
            "pull_p16": g.quantile(0.16).values,
            "pull_p84": g.quantile(0.84).values,
            "n": g.count().values,
        })
        summ = summ.sort_values("mass_GeV").reset_index(drop=True)

        # Save summary
        out_csv = os.path.join(outdir, "pull_summary.csv")
        summ.to_csv(out_csv, index=False)

        # Pull vs mass plot
        try:
            plt.figure(figsize=(10, 4.8))
            x = summ["mass_GeV"].values
            med = summ["pull_median"].values
            mu = summ["pull_mean"].values
            p16 = summ["pull_p16"].values
            p84 = summ["pull_p84"].values

            plt.plot(x, mu, linewidth=2, label="pull_mean")
            plt.plot(x, med, linewidth=2, label="pull_median")
            plt.fill_between(x, p16, p84, alpha=0.2, label="16–84% band")

            # v10: mark injected mass hypothesis on the scan (if enabled)
            try:
                if bool(globals().get("injected_signal_ensembles", False)):
                    _m_inj = float(globals().get("injected_signal_center", np.nan))
                    if np.isfinite(_m_inj):
                        plt.axvline(_m_inj, linestyle=":", linewidth=1.5, label=f"injected m={_m_inj:.3f} GeV")
            except Exception:
                pass

            plt.axhline(0.0, linestyle="--")
            plt.xlabel("Mass hypothesis m0 [GeV]")
            plt.ylabel("Pull = A_hat / sigma_A")
            plt.title(f"{title_prefix} — Pull vs mass")
            plt.legend(fontsize=9)
            plt.tight_layout()
            plt.savefig(os.path.join(outdir, "pull_vs_mass.png"), dpi=int(globals().get("SAVEFIG_DPI", 300)))
            plt.close()
        except Exception as e:
            print(f"[funcform][WARN] Failed to write pull_vs_mass.png in {outdir}: {e}")

        # Pull distribution plot
        pulls = df_all["pull"].to_numpy(dtype=float)
        pulls = pulls[np.isfinite(pulls)]
        try:
            plt.figure(figsize=(6.2, 4.6))
            if pulls.size > 0:
                plt.hist(pulls, bins=60, density=True, histtype="step", linewidth=2)
            plt.axvline(0.0, linestyle="--")
            plt.xlabel("Pull = A_hat / sigma_A")
            plt.ylabel("Density")
            plt.title(f"{title_prefix} — Pull distribution")
            plt.tight_layout()
            plt.savefig(os.path.join(outdir, "pull_distribution.png"), dpi=int(globals().get("SAVEFIG_DPI", 300)))
            plt.close()
        except Exception as e:
            print(f"[funcform][WARN] Failed to write pull_distribution.png in {outdir}: {e}")


        # v10: If signal injection is enabled, also write the pull distribution at the injected mass.
        try:
            if bool(globals().get("injected_signal_ensembles", False)):
                _m_inj = float(globals().get("injected_signal_center", np.nan))
                if np.isfinite(_m_inj):
                    mvals = summ["mass_GeV"].to_numpy(float)
                    if mvals.size > 0:
                        j0 = int(np.argmin(np.abs(mvals - _m_inj)))
                        m_closest = float(mvals[j0])
                        pulls_inj = df_all.loc[np.isclose(df_all["mass_GeV"].to_numpy(float), m_closest, atol=1e-12), "pull"].to_numpy(float)
                        pulls_inj = pulls_inj[np.isfinite(pulls_inj)]

                        # Text diagnostic for quick checks in the notebook log
                        try:
                            row0 = summ.iloc[j0]
                            print(f"[funcform][inj] {title_prefix}: nearest m0={m_closest:.6f} GeV to injected m={_m_inj:.6f} GeV; "
                                  f"pull_median={float(row0['pull_median']):.3f}, pull_mean={float(row0['pull_mean']):.3f}")
                        except Exception:
                            pass

                        if pulls_inj.size > 0:
                            plt.figure(figsize=(6.2, 4.6))
                            plt.hist(pulls_inj, bins=50, density=True, histtype="step", linewidth=2)
                            plt.axvline(0.0, linestyle="--")
                            plt.xlabel("Pull = A_hat / sigma_A")
                            plt.ylabel("Density")
                            plt.title(f"{title_prefix} — Pull distribution at injected mass m0={m_closest:.3f} GeV")
                            plt.tight_layout()
                            plt.savefig(os.path.join(outdir, "pull_distribution_at_injected_mass.png"), dpi=int(globals().get("SAVEFIG_DPI", 300)))
                            plt.close()
        except Exception as e:
            print(f"[funcform][WARN] Failed injection-mass pull diagnostic in {outdir}: {e}")
        return summ, pulls

    def _compute_metrics(summary: pd.DataFrame, pulls: np.ndarray, *, metric: str) -> Dict[str, float]:
        """Compute bias/width metrics for LS optimization."""
        pulls = np.asarray(pulls, float)
        pulls = pulls[np.isfinite(pulls)]

        med = summary["pull_median"].to_numpy(float)
        med = med[np.isfinite(med)]
        bias_rms = float(np.sqrt(np.mean(med**2))) if med.size else float("nan")
        bias_max = float(np.max(np.abs(med))) if med.size else float("nan")

        p_mean = float(np.mean(pulls)) if pulls.size else float("nan")
        p_std  = float(np.std(pulls, ddof=1)) if pulls.size > 1 else float("nan")

        # Score for optimization (lower is better)
        # We focus primarily on coherent bias (median vs mass).
        if metric == "max_abs_median":
            score = bias_max
        else:
            # default: rms_median
            score = bias_rms

        return dict(
            bias_rms=float(bias_rms),
            bias_max=float(bias_max),
            pull_mean=float(p_mean),
            pull_std=float(p_std),
            score=float(score),
        )

    # -----------------------------
    # Run a job at a given LS upper-factor
    # -----------------------------
    def _run_job_for_factor(*,
                            job: Dict[str, object],
                            ds_year: DatasetConfig,
                            masses: np.ndarray,
                            toy_indices: List[int],
                            outdir: str,
                            ls_upper_factor: float) -> Tuple[pd.DataFrame, np.ndarray]:
        """Process toys for one job and one LS setting. Returns (summary_df, pulls_1d)."""

        year = str(job.get("year"))
        tag = str(job.get("tag", f"job_{year}"))
        root_path = str(job.get("root_path"))
        container = job.get("container", None)
        toy_name_fmt = str(job.get("toy_name_fmt"))

        os.makedirs(outdir, exist_ok=True)

        # Build a fresh kernel with bounded length-scale (v8)
        kernel = make_kernel_for_dataset(
            ds_year,
            policy=_ls_policy,
            upper_factor=float(ls_upper_factor),
            lower_factor=float(_ls_lower_factor),
            stat=str(_ls_stat),
            npts=int(_ls_npts),
            optimized_by_year=_OPTIMIZED_LS_BY_YEAR,
        )

        # Print the implied bounds (useful debugging)
        try:
            if str(_ls_policy).lower().strip() == "resolution_scaled":
                ls_lo, ls_hi, base = _kernel_bounds_from_resolution(
                    ds_year, upper_factor=float(ls_upper_factor),
                    lower_factor=float(_ls_lower_factor),
                    stat=str(_ls_stat), npts=int(_ls_npts)
                )
                print(f"[funcform][kernel] {year}:{tag}  ls_policy=resolution_scaled  "
                      f"base_sigma_log={base:.4g}  bounds=({ls_lo:.4g},{ls_hi:.4g})  "
                      f"upper_factor={ls_upper_factor:g}")
            else:
                print(f"[funcform][kernel] {year}:{tag}  ls_policy=manual  bounds={tuple(KERNEL_LS_BOUNDS)}  init={KERNEL_LS_INIT}")
        except Exception:
            pass

        def _process_one_toy(toy_idx: int) -> Dict[str, object]:
            """Run the full pull scan for a single toy and write pulls_toy_###.csv."""
            toy_name = toy_name_fmt.format(i=int(toy_idx))
            toy_csv = os.path.join(outdir, f"pulls_toy_{int(toy_idx):03d}.csv")

            try:
                edges_i, counts_i = _load_one_toy_hist(root_path, container, toy_name)
                # v10: cache injection metadata per-toy (propagated into per-mass rows/CSVs)
                _inj_meta = dict(
                    inj_enabled=False,
                    inj_ok=False,
                    inj_reason="",
                    inj_center_GeV=float("nan"),
                    inj_sigma_GeV=float("nan"),
                    inj_n_requested=float(0.0),
                    inj_n_added=float(0.0),
                    inj_mode="",
                    inj_gauss_fraction_in_hist=float("nan"),
                )

                # v10: optional injected-signal ensembles (inject a Gaussian signal into each toy histogram)
                if bool(globals().get("injected_signal_ensembles", False)):
                    _n_inj = float(globals().get("ensemble_injection", 0.0))
                    _m_inj = float(globals().get("injected_signal_center", np.nan))
                    if np.isfinite(_m_inj) and np.isfinite(_n_inj) and (_n_inj != 0.0):
                        _scale = float(globals().get("INJECT_SIGNAL_SIGMA_SCALE", 1.0))
                        _sigma_inj = float(_scale) * float(ds_year.sigma(float(_m_inj)))
                        _fluct = bool(globals().get("INJECT_SIGNAL_FLUCTUATE", False))
                        _base_seed = int(globals().get("INJECT_SIGNAL_SEED", 12345))

                        # Deterministic per-toy seed so that LS scans see identical injected toys
                        _seed = (_base_seed
                                 + int(toy_idx) * 1000003
                                 + int(round(float(_m_inj) * 1e6)) * 9176
                                 + int(round(float(_n_inj) * 10.0)) * 131) % (2**32 - 1)

                        counts_i, _inj_info = inject_gaussian_signal_into_hist(
                            edges_i, counts_i,
                            center=float(_m_inj),
                            sigma=float(_sigma_inj),
                            n_events=float(_n_inj),
                            fluctuate=bool(_fluct),
                            seed=int(_seed),
                        )

                        # Cache injection metadata for downstream CSVs/plots
                        try:
                            _inj_meta.update(dict(
                                inj_enabled=True,
                                inj_ok=bool(_inj_info.get("ok", False)),
                                inj_reason=str(_inj_info.get("reason", "")),
                                inj_center_GeV=float(_inj_info.get("center", _m_inj)),
                                inj_sigma_GeV=float(_inj_info.get("sigma", _sigma_inj)),
                                inj_n_requested=float(_inj_info.get("n_events", _n_inj)),
                                inj_n_added=float(_inj_info.get("n_added", float("nan"))),
                                inj_mode=str(_inj_info.get("mode", "")),
                                inj_gauss_fraction_in_hist=float(_inj_info.get("gauss_fraction_in_hist", float("nan"))),
                            ))
                        except Exception:
                            pass

                        if not bool(_inj_info.get("ok", False)):
                            print(f"[funcform][inj][WARN] {year}:{tag} toy {int(toy_idx)} injection failed: {_inj_info}")
                        else:
                            if bool(globals().get("INJECT_SIGNAL_VERBOSE", False)) and int(toy_idx) in set(toy_indices[:3]):
                                print(f"[funcform][inj] {year}:{tag} toy {int(toy_idx)} injected "
                                      f"{_inj_info.get('n_added', '???')} events at m={_m_inj:.6f} GeV "
                                      f"(sigma={_sigma_inj:.6g} GeV, mode={_inj_info.get('mode','')})")

                rows = []
                for m0 in masses:
                    try:
                        out = _toy_pull_at_mass(
                            ds_year=ds_year,
                            edges_in=edges_i,
                            counts_in=counts_i,
                            mass=float(m0),
                            rebin=int(_rebin),
                            restarts=int(_restarts),
                            allow_negative=bool(_allow_neg),
                            kernel=kernel,
                        )
                        rows.append(dict(
                            year=year,
                            tag=tag,
                            **_inj_meta,
                            ls_upper_factor=float(ls_upper_factor),
                            toy=int(toy_idx),
                            mass_GeV=float(m0),
                            pull=float(out["pull"]),
                            A_hat=float(out["A_hat"]),
                            sigma_A=float(out["sigma_A"]),
                            extract_ok=bool(out.get("ok", False)),
                            blind_lo_GeV=float(out["blind_lo"]),
                            blind_hi_GeV=float(out["blind_hi"]),
                            n_blind_bins=int(out["n_blind"]),
                        ))
                    except Exception as e_mass:
                        rows.append(dict(
                            year=year, tag=tag, ls_upper_factor=float(ls_upper_factor), toy=int(toy_idx),
                            **_inj_meta,
                            mass_GeV=float(m0),
                            pull=np.nan, A_hat=np.nan, sigma_A=np.nan,
                            extract_ok=False,
                            blind_lo_GeV=np.nan, blind_hi_GeV=np.nan, n_blind_bins=0,
                            error=str(e_mass),
                        ))

                df = pd.DataFrame(rows)
                df.to_csv(toy_csv, index=False)
                return dict(ok=True, toy=int(toy_idx), csv=toy_csv)
            except Exception as e:
                # still write something for bookkeeping
                df = pd.DataFrame([dict(year=year, tag=tag, ls_upper_factor=float(ls_upper_factor),
                                        toy=int(toy_idx), error=str(e))])
                df.to_csv(toy_csv, index=False)
                return dict(ok=False, toy=int(toy_idx), csv=toy_csv, error=str(e))

        # Run toys (optionally in parallel)
        if _n_workers > 1:
            results = Parallel(n_jobs=int(_n_workers), backend=str(_joblib_backend))(
                delayed(_process_one_toy)(ti) for ti in toy_indices
            )
        else:
            results = [_process_one_toy(ti) for ti in toy_indices]

        # Aggregate all toy CSVs
        dfs = []
        for r in results:
            try:
                dfs.append(pd.read_csv(r["csv"]))
            except Exception:
                continue

        df_all = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame(columns=["mass_GeV", "pull"])

        # Filter to finite pulls
        df_all = df_all[np.isfinite(df_all["pull"].to_numpy(float))].copy()

        title_prefix = f"{year}:{tag} (lsUF={ls_upper_factor:g})"
        summary, pulls = _summarize_pulls(df_all, outdir, title_prefix=title_prefix)
        return summary, pulls

    # -----------------------------
    # Run all jobs (+ optional LS optimization)
    # -----------------------------
    all_job_pull_arrays: List[Tuple[str, np.ndarray]] = []

    print("\n[funcform] Starting functional-form toy pull analyses (v8)")
    print("[funcform] Jobs enabled:", [f"{j.get('year')}:{j.get('tag')}" for j in _jobs])
    print(f"[funcform] Toy indices: {_toy_i0}..{_toy_i1}  (expected {_n_toys_expected} toys/job)")
    print(f"[funcform] GP restarts={_restarts}, rebin={_rebin}, step={_mass_step:.6f} GeV, allow_negative={_allow_neg}")
    print(f"[funcform] LS policy={_ls_policy}  optimize={_ls_optimize}  workers={_n_workers} backend={_joblib_backend}")
    print("")

    for job in _jobs:
        year = str(job.get("year"))
        tag = str(job.get("tag", f"job_{year}"))
        ds_year = _funcform_ds_for_year(year)

        # Mass grid (use the same scan region as configured in Inputs)
        masses = np.round(np.arange(ds_year.m_low, ds_year.m_high + 0.5 * _mass_step, _mass_step), 6)

        outdir_year = os.path.join(_out_root, f"funcform_{year}")
        # v9: write injected-signal ensembles into a distinct subfolder to avoid overwriting no-injection results
        _inj_on = bool(globals().get("injected_signal_ensembles", False))
        if _inj_on:
            _injN = float(globals().get("ensemble_injection", 0.0))
            _injM = float(globals().get("injected_signal_center", 0.0))
            _inj_label = f"inj_{_injN:g}evt_m{_injM:.6f}"
            outdir_job = os.path.join(outdir_year, tag, _inj_label)
        else:
            outdir_job = os.path.join(outdir_year, tag, "noinj")
        os.makedirs(outdir_job, exist_ok=True)

        print(f"[funcform] --- {year}:{tag} ---")
        print(f"[funcform] ROOT file: {job.get('root_path')}")
        print(f"[funcform] container={job.get('container', None)}  toy_name_fmt='{job.get('toy_name_fmt')}'")
        print(f"[funcform] scan range: [{ds_year.m_low:.6f},{ds_year.m_high:.6f}] GeV  ({len(masses)} mass points)")
        print(f"[funcform] output: {outdir_job}")
        print("")

        toy_indices_all = list(range(_toy_i0, _toy_i1 + 1))

        # -----------------------------
        # Optimization scan (optional)
        # -----------------------------
        best_factor = float(_ls_upper_factor_default)
        best_summary = None
        best_pulls = None

        if _ls_optimize and str(_ls_policy).lower().strip() == "resolution_scaled":
            nopt = int(min(len(toy_indices_all), max(1, _ls_opt_n_toys)))
            toy_indices_opt = toy_indices_all[:nopt]

            metrics_rows = []
            for uf in _ls_grid:
                outdir_factor = os.path.join(outdir_job, f"lsUF_{uf:g}", f"opt_{nopt:03d}toys")
                summary, pulls = _run_job_for_factor(
                    job=job, ds_year=ds_year, masses=masses, toy_indices=toy_indices_opt, outdir=outdir_factor,
                    ls_upper_factor=float(uf),
                )
                m = _compute_metrics(summary, pulls, metric=_ls_metric)
                metrics_rows.append(dict(
                    year=year, tag=tag, ls_upper_factor=float(uf),
                    n_toys=int(nopt),
                    **m,
                    outdir=outdir_factor,
                ))

            dfm = pd.DataFrame(metrics_rows)
            dfm = dfm.sort_values(["score", "bias_rms", "bias_max"]).reset_index(drop=True)
            dfm.to_csv(os.path.join(outdir_job, "ls_optimization_metrics.csv"), index=False)

            best_factor = float(dfm.loc[0, "ls_upper_factor"])
            print(f"[funcform][OPT] Best upper-factor (metric={_ls_metric}) for {year}:{tag}: {best_factor:g}")
            print(f"[funcform][OPT] Wrote metrics: {os.path.join(outdir_job, 'ls_optimization_metrics.csv')}")

            # Plot: overlay median pull vs mass for all scanned factors
            try:
                plt.figure(figsize=(10, 4.8))
                for _, row in dfm.iterrows():
                    p = os.path.join(str(row["outdir"]), "pull_summary.csv")
                    if not os.path.exists(p):
                        continue
                    s = pd.read_csv(p)
                    plt.plot(s["mass_GeV"].values, s["pull_median"].values, linewidth=2, label=f"UF={float(row['ls_upper_factor']):g}")
                plt.axhline(0.0, linestyle="--")
                plt.xlabel("Mass hypothesis m0 [GeV]")
                plt.ylabel("Median pull")
                plt.title(f"{year}:{tag} — median pull vs mass (LS upper-factor scan)")
                plt.legend(fontsize=8, ncol=2)
                plt.tight_layout()
                plt.savefig(os.path.join(outdir_job, "ls_optimization_median_overlay.png"), dpi=int(globals().get("SAVEFIG_DPI", 300)))
                plt.close()
            except Exception as e:
                print(f"[funcform][WARN] Could not write ls_optimization_median_overlay.png: {e}")

            # After optimization: run full ensemble for best factor (recommended)
            if bool(_ls_run_full_after):
                outdir_best = os.path.join(outdir_job, f"lsUF_{best_factor:g}", "full")
                best_summary, best_pulls = _run_job_for_factor(
                    job=job, ds_year=ds_year, masses=masses, toy_indices=toy_indices_all, outdir=outdir_best,
                    ls_upper_factor=float(best_factor),
                )
            else:
                # Reuse best factor's optimization run
                outdir_best = os.path.join(outdir_job, f"lsUF_{best_factor:g}", f"opt_{nopt:03d}toys")
                best_summary = pd.read_csv(os.path.join(outdir_best, "pull_summary.csv"))
                # Rebuild pull array from per-toy CSVs in outdir_best
                pulls = []
                for ti in toy_indices_opt:
                    p = os.path.join(outdir_best, f"pulls_toy_{ti:03d}.csv")
                    if os.path.exists(p):
                        d = pd.read_csv(p)
                        if "pull" in d.columns:
                            pulls.append(d["pull"].to_numpy(float))
                best_pulls = np.concatenate(pulls) if pulls else np.array([], float)

        else:
            # Single run (no optimization)
            outdir_best = os.path.join(outdir_job, f"lsUF_{best_factor:g}", "full")
            best_summary, best_pulls = _run_job_for_factor(
                job=job, ds_year=ds_year, masses=masses, toy_indices=toy_indices_all, outdir=outdir_best,
                ls_upper_factor=float(best_factor),
            )

        # Record for combined distribution plot
        if best_pulls is not None:
            label = f"{year}:{tag} (UF={best_factor:g})"
            all_job_pull_arrays.append((label, np.asarray(best_pulls, float)))

        # Save optimized bounds for this year (optional)
        if _ls_save_opt and str(_ls_policy).lower().strip() == "resolution_scaled":
            try:
                ls_lo, ls_hi, base = _kernel_bounds_from_resolution(
                    ds_year,
                    upper_factor=float(best_factor),
                    lower_factor=float(_ls_lower_factor),
                    stat=str(_ls_stat),
                    npts=int(_ls_npts),
                )
                ent = dict(
                    year=str(year),
                    tag=str(tag),
                    length_scale_policy="resolution_scaled",
                    upper_factor=float(best_factor),
                    lower_factor=float(_ls_lower_factor),
                    stat=str(_ls_stat),
                    npts=int(_ls_npts),
                    base_sigma_log=float(base),
                    length_scale_bounds=[float(ls_lo), float(ls_hi)],
                    length_scale_init=float(np.sqrt(ls_lo * ls_hi)),
                    created_utc=datetime.now(timezone.utc).isoformat(),
                )
                _opt_updates[str(year)] = ent
            except Exception as e:
                print(f"[funcform][WARN] Could not compute/save optimized LS bounds for {year}:{tag}: {e}")

        print(f"[funcform] --- {year}:{tag} done (best UF={best_factor:g}) ---\n")

    # -----------------------------
    # Final: combined pull distribution overlay across all jobs
    # -----------------------------
    try:
        plt.figure(figsize=(6.6, 4.8))
        for label, arr in all_job_pull_arrays:
            arr = np.asarray(arr, float)
            arr = arr[np.isfinite(arr)]
            if arr.size == 0:
                continue
            plt.hist(arr, bins=60, density=True, histtype="step", linewidth=2, label=label)
        plt.axvline(0.0, linestyle="--")
        plt.xlabel("Pull = A_hat / sigma_A")
        plt.ylabel("Density")
        plt.title("Functional-form toy pull distributions (all jobs)")
        plt.legend(fontsize=8)
        plt.tight_layout()
        out_plot = os.path.join(_out_root, "funcform_pull_distributions_all.png")
        plt.savefig(out_plot, dpi=int(globals().get("SAVEFIG_DPI", 300)))
        plt.close()
        print(f"[funcform] Wrote combined plot: {out_plot}")
    except Exception as e:
        print(f"[funcform][WARN] Failed to write combined pull-distribution plot: {e}")

    # -----------------------------
    # Write optimized LS bounds file (optional)
    # -----------------------------
    if bool(globals().get("SAVE_OPTIMIZED_LS", True)) and _opt_file_path and _opt_updates:
        try:
            _p = str(_opt_file_path)
            base_cfg = {}
            if os.path.exists(_p):
                try:
                    with open(_p, "r") as _f:
                        base_cfg = json.load(_f)
                except Exception:
                    base_cfg = {}

            base_cfg.setdefault("by_year", {})
            # v11: write metadata so bounds are not misinterpreted when PRE_LOG/x-transform changes
            base_cfg.setdefault("meta", {})
            base_cfg["meta"].update({
                "version": "v11",
                "pre_log": bool(PRE_LOG),
                "x_transform": ("ln" if bool(PRE_LOG) else "identity"),
                "kernel_ls_policy": str(globals().get("KERNEL_LS_POLICY", "")),
                "note": "length_scale_bounds are expressed in the GP x-units (x=ln(m) if PRE_LOG else x=m)",
            })
            # Update per-year entries
            for yr, ent in _opt_updates.items():
                base_cfg["by_year"][yr] = ent

            base_cfg["updated_utc"] = datetime.now(timezone.utc).isoformat()

            with open(_p, "w") as _f:
                json.dump(base_cfg, _f, indent=2, sort_keys=True)
            print(f"[funcform][kernel] Wrote optimized LS config: {_p}")
        except Exception as e:
            print(f"[funcform][WARN] Failed to write optimized LS config file: {e}")

    print("[funcform] All functional-form toy analyses complete.")

# 


# Checklist and plot menu for a publication-ready review (v15)

This is a living checklist of analysis validation steps and *reviewer-facing* figures that are typically expected for a GP-based bump hunt.

## A. Core closure tests

1. **Asimov sanity checks**
   - Verify `A_hat ≈ 0` and `pull ≈ N(0,1)` at **0σ injection** (both conditional toys and refit-on-toy toys).
   - Confirm that the profiled extraction returns the expected \(\sigma_A\) and that it scales smoothly with mass.

2. **Linearity / bias vs injected signal**
   - Plot `A_hat_mean` vs `A_inj` (and/or vs injected \(n_\sigma\)).
   - Plot bias `A_hat_mean − A_inj` vs `A_inj`, and include uncertainties (SEM from toys).
   - Repeat for both:
     - **conditional toys** (fast diagnostic)
     - **full procedural toys** (GP refit on each toy)

3. **Pull width and coverage**
   - Plot `pull_std` vs `A_inj` and vs mass.
   - Plot empirical coverage `P(|pull|<1)` and `P(|pull|<2)`.

## B. Signal-leakage / GP-refit bias studies (v15 focus)

When `INJ_REFIT_GP_ON_TOY=True`, a real injected signal has **tails** that can contaminate the sidebands used to train the GP. This can bias `A_hat` low because the GP absorbs part of the signal.

Recommended diagnostic axes:
- `BLIND_NSIGMA` (extraction window)
- `INJ_TRAIN_EXCLUDE_NSIGMA` (GP training exclusion used in refit toys)
- `INJ_SHAPE_MODE`:
  - `"full"` = realistic (tails leak)
  - `"window"` = diagnostic control (no leakage)

Suggested plots (directly supported by v15 toy outputs):
- `A_hat_mean − A_inj` vs `f_train_frac` and/or vs `Nsig_train_mean`
- `f_train_frac` vs mass for the chosen `(BLIND_NSIGMA, INJ_TRAIN_EXCLUDE_NSIGMA)` pair
- Compare extracted bias for `INJ_SHAPE_MODE="full"` vs `"window"` at fixed injected \(n_\sigma\)

## C. GP fit-quality diagnostics (publication staples)

For each dataset (and ideally shown at representative masses):
- **Full-range fit + zoom panel** with consistent blind-window shading.
- **Sideband residual pulls** (data − GP mean) / sqrt(data) or / posterior σ:
  - histogram of pulls (should be ~N(0,1) if the model is good)
  - pull vs mass/bin-center “strip plot” to reveal structure
- **Kernel hyperparameters vs mass** (e.g. best-fit length-scale `ls_opt`):
  - `ls_opt`, `ls_lo`, `ls_hi` vs mass
  - flag masses where optimizer hits bounds frequently
- **Fit stability**
  - fraction of masses where the GP fit fails / restarts are needed
  - any strong dependence on `NEIGHBORHOOD_REBIN`, `N_RESTARTS`, or LS bounds

## D. Expected upper-limit bands and “procedural variance” (v15)

Two reasonable choices exist for expected bands:

1. **Conditional (fast) expected bands** (default behavior):
   - Toys are drawn only in the blind window from the conditional posterior `(μ, Σ)` and then Poisson-sampled.
   - This captures uncertainty encoded in `(μ, Σ)` but *does not* refit the GP per toy.

2. **Full procedural expected bands** (recommended for final/reviewer plots when feasible):
   - Enable `UL_BANDS_REFIT_GP_ON_TOY=True`.
   - Each toy generates a full pseudo-dataset and refits the GP on sidebands before computing the UL.
   - This captures the procedural variance of the background fit.

Both single-dataset and combined-dataset expected bands now support this toggle.

## E. Reviewer-facing “final result” figures (typical set)

- Observed \(\varepsilon^2\) UL vs mass with expected ±1σ/±2σ bands.
- Local **p0** vs mass (and corresponding local significance Z).
- If combined results are shown:
  - individual-dataset ULs (same axes) + combined UL, clearly labeled
  - a note on overlap definition and independence assumption (block-diagonal covariance)

